# MOLM Unified Experiment Suite

This notebook provides a single Kaggle workflow for the extended MOLM evaluation suite and adds **Standard-MOLM** as a controlled shared-learning reference alongside Routed-MOLM, NN, and deterministic LDA.

It is designed to reuse the definitive result bundle and ESM-2 feature caches when they are attached under `/kaggle/input`; dataset folder names are discovered recursively rather than hard-coded.

## What this notebook does

1. Finds and verifies the previous definitive result bundle (`Routed-MOLM`, `NN`, deterministic `LDA`).
2. Reuses the exact ESM-2 caches and reconstructs the same five representations.
3. Adds **Standard-MOLM** with a shared encoder, task-specific towers, and the focal + ranking + gap objective.
4. Runs Standard-MOLM on the same 16 mutation holdouts, five representations, and five optimization seeds.
5. Recomputes mutation statistics with Standard-MOLM included: McNemar tests, paired 10,000-draw sequence bootstrap, Friedman/Wilcoxon/exact sign-flip tests, Holm correction, and Site-vs-Mean representation contrasts.
6. Runs Standard-MOLM on ISO, IgG-primary42, and IgG-all96 and recomputes external Spearman inference.
7. Recomputes fixed-screening-budget Pareto comparisons at `K={5,10,15,20,25}` with recall, precision, enrichment, hypervolume, IGD, and Recall-AUC.
8. Evaluates Standard-MOLM latent-PCA robustness using tower 64D, tower 32D, latent 16D, and logits. PCA is fit on EMI training representations only.
9. Runs the A--H Routed-MOLM component/capacity ablation.
10. Runs the ranking/gap loss ablation: focal only, focal+ranking, focal+gap, and focal+ranking+gap.
11. Runs Hamming-distance generalization (`d=1`, `d=2`, `d>=3`) from the frozen mutation-holdout rows.
12. Writes a unified reproducibility/result bundle.

### Interpretation note
The ISO/IgG external benchmarks were observed during earlier model development. Analyses added later in the project should therefore be interpreted as retrospective robustness/controlled analyses rather than newly blind external validation.


> **Kaggle runtime note:** this version accepts the Kaggle `.csv` form of the prior result bundle and audits the pinned `phase0_config` biological mapping through `phase0_config.config`. Scientific settings, Standard-MOLM architecture, seeds, split definitions, and statistical procedures are unchanged.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, shutil, subprocess, zipfile, itertools, time, platform
import numpy as np
import pandas as pd

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/molm_unified_experiments')
BASE_EXTRACT = WORK_ROOT / 'base_result_input'
REPO = WORK_ROOT / 'repo'
CODE_DIR = WORK_ROOT / 'code'
CACHE_DIR = WORK_ROOT / 'esm_cache'
FEATURE_DIR = WORK_ROOT / 'features'
STANDARD_MUT_DIR = WORK_ROOT / 'standard_mutation'
STANDARD_EXT_DIR = WORK_ROOT / 'standard_external'
COMPONENT_DIR = WORK_ROOT / 'component_ablation'
LOSS_DIR = WORK_ROOT / 'loss_ablation'
ANALYSIS_DIR = WORK_ROOT / 'analysis'
LOG_DIR = WORK_ROOT / 'logs'
for p in [WORK_ROOT,BASE_EXTRACT,CODE_DIR,CACHE_DIR,FEATURE_DIR,STANDARD_MUT_DIR,STANDARD_EXT_DIR,COMPONENT_DIR,LOSS_DIR,ANALYSIS_DIR,LOG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ---- Locked scientific settings from the definitive experiment ----
REPO_URL = 'https://github.com/DigantaX/molm-pipeline.git'
PINNED_COMMIT = 'c5923984f0d5176977edb4a4ffd8fc5f98536043'
SEEDS = [42, 123, 456, 789, 2024]
FEATURE_TYPES = ['onehot','mean_esm2','mean_fusion','site_esm2','site_fusion']
FEATURE_DIMS = {'onehot':2300,'mean_esm2':320,'mean_fusion':2620,'site_esm2':2560,'site_fusion':4860}
REPRESENTATION_CONTRASTS = [
    ('site_esm2','mean_esm2','Site-ESM2 - Mean-ESM2'),
    ('site_fusion','mean_fusion','Site-Fusion - Mean-Fusion'),
]
K_VALUES = [5,10,15,20,25]
FULL_CURVE_MAX_K = 25
BOOTSTRAP_DRAWS = 10_000
BOOTSTRAP_SEED = 20260819
EPOCHS = 25
BATCH_SIZE = 64
LEARNING_RATE = 5e-5

# ---- Unified-run switches ----
RUN_STANDARD_MAIN = True
RUN_COMPONENT_ABLATION = True
RUN_COMPONENT_EXTERNAL = True
RUN_LOSS_ABLATION = True
RUN_HAMMING = True
RUN_STANDARD_LATENT_ROBUSTNESS = True

# Controlled ablations default to OneHot so they isolate method/loss effects
# without multiplying the already-large five-representation experiment.
COMPONENT_FEATURES = ['onehot']
LOSS_ABLATION_FEATURES = ['onehot']
COMPONENT_ARM_SET = 'core'

REQUIRE_TWO_GPUS = True
RESUME = True

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Input root exists:', INPUT_ROOT.exists())

## 1. Auto-discover Kaggle inputs

The resolver accepts either extracted result files or ZIP archives. It searches recursively, so names such as `esm_reuse` and `MOLM_Definitive_..._Results` can change without editing this notebook.

In [ ]:
def sha256_file(path):
    path=Path(path); h=hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()

def find_direct(name, root=INPUT_ROOT):
    return sorted(root.rglob(name)) if root.exists() else []

# Kaggle may expose files that were originally *.csv.gz as plain *.csv after
# creating a dataset from notebook output. Treat these as equivalent inputs.
BASE_FILE_ALIASES = {
    'mutation_raw_predictions_all.csv.gz': [
        'mutation_raw_predictions_all.csv.gz',
        'mutation_raw_predictions_all.csv',
    ],
    'external_raw_predictions_all.csv.gz': [
        'external_raw_predictions_all.csv.gz',
        'external_raw_predictions_all.csv',
    ],
    'mutation_seed_consensus_scores.csv.gz': [
        'mutation_seed_consensus_scores.csv.gz',
        'mutation_seed_consensus_scores.csv',
    ],
    'external_seed_consensus_scores.csv.gz': [
        'external_seed_consensus_scores.csv.gz',
        'external_seed_consensus_scores.csv',
    ],
    'mutation_training_diagnostics_all.csv.gz': [
        'mutation_training_diagnostics_all.csv.gz',
        'mutation_training_diagnostics_all.csv',
    ],
    'external_training_diagnostics_all.csv.gz': [
        'external_training_diagnostics_all.csv.gz',
        'external_training_diagnostics_all.csv',
    ],
}

def aliases_for(name):
    if name in BASE_FILE_ALIASES:
        return BASE_FILE_ALIASES[name]
    if name.endswith('.csv.gz'):
        return [name, name[:-3]]  # *.csv.gz -> *.csv
    if name.endswith('.gz'):
        return [name, name[:-3]]
    return [name]

# Inventory first so a saved Kaggle failure always shows what is mounted.
_inventory_paths = [p for p in sorted(INPUT_ROOT.rglob('*')) if p.is_file()]
print(f"Kaggle mounted {len(_inventory_paths)} input files.")
for p in _inventory_paths[:200]:
    try:
        size_mb = p.stat().st_size / 1024**2
    except OSError:
        size_mb = float('nan')
    print(f"  {p}  ({size_mb:.3f} MB)")
if len(_inventory_paths) > 200:
    print(f"  ... {len(_inventory_paths)-200} more files")

def first_existing_under(root, logical_name):
    root = Path(root)
    if not root.exists():
        return None
    for candidate_name in aliases_for(logical_name):
        hits = sorted(root.rglob(candidate_name))
        if hits:
            return hits[0]
    return None

def is_definitive_result_dir(folder):
    folder = Path(folder)
    lock = first_existing_under(folder, 'LOCKED_CONFIG.json')
    mut = first_existing_under(folder, 'mutation_raw_predictions_all.csv.gz')
    ext = first_existing_under(folder, 'external_raw_predictions_all.csv.gz')
    return lock is not None and mut is not None and ext is not None

def discover_direct_result_dir():
    # Anchor on the large mutation raw file, because LOCKED_CONFIG.json also
    # exists in the ESM-cache dataset and must not be used as the base result lock.
    mutation_anchors = []
    for alias in aliases_for('mutation_raw_predictions_all.csv.gz'):
        mutation_anchors.extend(find_direct(alias, INPUT_ROOT))
    for anchor in sorted(set(mutation_anchors)):
        # The definitive Kaggle dataset is normally flat. Check the anchor's
        # parent first, then a few ancestors in case Kaggle added a wrapper.
        candidates = [anchor.parent] + list(anchor.parents)[:4]
        for folder in candidates:
            if folder == INPUT_ROOT.parent:
                continue
            lock = first_existing_under(folder, 'LOCKED_CONFIG.json')
            ext = first_existing_under(folder, 'external_raw_predictions_all.csv.gz')
            mut = first_existing_under(folder, 'mutation_raw_predictions_all.csv.gz')
            if lock is not None and ext is not None and mut is not None:
                # Prefer the smallest coherent folder: if all three share the
                # same immediate directory, return that exact directory.
                parents = {lock.parent, ext.parent, mut.parent}
                if len(parents) == 1:
                    return parents.pop()
                return folder
    return None

def zip_has_logical_files(zip_path):
    try:
        with zipfile.ZipFile(zip_path, 'r') as z:
            basenames = {Path(n).name for n in z.namelist()}
        needed = ['LOCKED_CONFIG.json',
                  'mutation_raw_predictions_all.csv.gz',
                  'external_raw_predictions_all.csv.gz']
        return all(any(alias in basenames for alias in aliases_for(name)) for name in needed)
    except (zipfile.BadZipFile, OSError, PermissionError):
        return False

def extract_result_zip(zip_path, destination):
    if not zip_has_logical_files(zip_path):
        return None
    if destination.exists():
        shutil.rmtree(destination)
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extractall(destination)
    # Find a coherent extracted directory using the same anchor logic locally.
    mut_hits = []
    for alias in aliases_for('mutation_raw_predictions_all.csv.gz'):
        mut_hits.extend(destination.rglob(alias))
    for mut in sorted(mut_hits):
        lock = first_existing_under(mut.parent, 'LOCKED_CONFIG.json')
        ext = first_existing_under(mut.parent, 'external_raw_predictions_all.csv.gz')
        if lock is not None and ext is not None:
            return mut.parent
    # fallback if archive has a wrapper directory
    for folder in [destination] + [p for p in destination.rglob('*') if p.is_dir()]:
        if is_definitive_result_dir(folder):
            return folder
    return None

def discover_base_results():
    # 1) Normal Kaggle dataset mount (this is the user's current layout).
    direct_root = discover_direct_result_dir()
    if direct_root is not None:
        print("Found coherent definitive results directory:", direct_root)
        print("  mutation:", first_existing_under(direct_root, 'mutation_raw_predictions_all.csv.gz'))
        print("  external:", first_existing_under(direct_root, 'external_raw_predictions_all.csv.gz'))
        print("  lock:", first_existing_under(direct_root, 'LOCKED_CONFIG.json'))
        return direct_root, 'direct', None

    # 2) Actual ZIP archives. Do not call NPZ files ZIP candidates; NPZ is a ZIP
    # container internally but is not a result archive.
    archive_candidates = [
        p for p in _inventory_paths
        if p.suffix.lower() == '.zip'
    ]
    print(f"Result ZIP candidates detected: {len(archive_candidates)}")
    for z in archive_candidates:
        root = extract_result_zip(z, BASE_EXTRACT)
        if root is not None:
            print("Found definitive result bundle in archive:", z)
            return root, 'zip', z

    # Helpful diagnostics.
    mut_any = []
    ext_any = []
    for alias in aliases_for('mutation_raw_predictions_all.csv.gz'):
        mut_any.extend(find_direct(alias, INPUT_ROOT))
    for alias in aliases_for('external_raw_predictions_all.csv.gz'):
        ext_any.extend(find_direct(alias, INPUT_ROOT))
    raise FileNotFoundError(
        "Could not locate a coherent definitive result directory.\n"
        f"Mutation raw candidates: {[str(p) for p in mut_any[:20]]}\n"
        f"External raw candidates: {[str(p) for p in ext_any[:20]]}\n"
        "Expected the mutation raw, external raw, and LOCKED_CONFIG.json to "
        "belong to the same definitive-results Kaggle input."
    )

BASE_RESULT_ROOT, BASE_MODE, BASE_ZIP = discover_base_results()
print('Base results:', BASE_RESULT_ROOT)
print('Discovery mode:', BASE_MODE, 'zip=', BASE_ZIP)

rows=[]
for p in _inventory_paths:
    rows.append({'path':str(p),'size_mb':p.stat().st_size/1024**2})
input_inventory=pd.DataFrame(rows)
display(input_inventory.head(200))


In [ ]:
def base_file(name, required=True):
    # Always prefer the coherent definitive result directory selected above.
    for candidate_name in aliases_for(name):
        candidates = sorted(BASE_RESULT_ROOT.rglob(candidate_name)) if BASE_RESULT_ROOT.exists() else []
        if candidates:
            return candidates[0]

    # Extraction fallback only; intentionally do NOT search all of /kaggle/input
    # here because esm_reuse contains a second LOCKED_CONFIG.json.
    if BASE_EXTRACT.exists():
        for candidate_name in aliases_for(name):
            candidates = sorted(BASE_EXTRACT.rglob(candidate_name))
            if candidates:
                return candidates[0]

    if required:
        raise FileNotFoundError(
            f"{name} (aliases={aliases_for(name)}) not found inside "
            f"definitive result root {BASE_RESULT_ROOT}"
        )
    return None


LOCKED_CONFIG_PATH = base_file('LOCKED_CONFIG.json')
LOCK_SHA_PATH = base_file('LOCKED_CONFIG.sha256')
BASE_LOCK_SHA = sha256_file(LOCKED_CONFIG_PATH)
recorded = LOCK_SHA_PATH.read_text(encoding='utf-8').strip()
if recorded != BASE_LOCK_SHA:
    raise RuntimeError(f'Base lock hash mismatch: recorded={recorded} observed={BASE_LOCK_SHA}')
locked_config=json.loads(LOCKED_CONFIG_PATH.read_text(encoding='utf-8'))
if locked_config['repository']['commit'] != PINNED_COMMIT:
    raise RuntimeError('Pinned repository commit differs from definitive lock.')
print('Verified base LOCK:', BASE_LOCK_SHA)
print('Models in base lock:', list(locked_config.get('models',{})))
print('Pareto budgets:', locked_config.get('pareto_budgets'))


## 2. Repository + exact helper code

The notebook first looks for a repository snapshot under `/kaggle/input`. If none exists, it clones the pinned GitHub commit. Internet should therefore be **On** unless you also attach the repository as a Kaggle input.

In [ ]:
import importlib.metadata
import torch

if REQUIRE_TWO_GPUS and torch.cuda.device_count() != 2:
    raise RuntimeError(f'Select Kaggle T4 x2. Found {torch.cuda.device_count()} GPU(s).')
print('GPUs:', [torch.cuda.get_device_properties(i).name for i in range(torch.cuda.device_count())])

try:
    esm_version=importlib.metadata.version('fair-esm')
except importlib.metadata.PackageNotFoundError:
    esm_version=None
if esm_version != '2.0.0':
    subprocess.run([sys.executable,'-m','pip','install','-q','fair-esm==2.0.0'],check=True)
print('fair-esm:', importlib.metadata.version('fair-esm'))

repo_candidates=[]
for p in INPUT_ROOT.rglob('phase0_config.py'):
    parent=p.parent
    if (parent/'data'/'emi_binding.csv').exists(): repo_candidates.append(parent)
if repo_candidates:
    if REPO.exists(): shutil.rmtree(REPO)
    shutil.copytree(repo_candidates[0], REPO)
    print('Using repository snapshot from input:', repo_candidates[0])
else:
    if REPO.exists(): shutil.rmtree(REPO)
    subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True)
    subprocess.run(['git','-C',str(REPO),'checkout',PINNED_COMMIT],check=True)

if (REPO/'.git').exists():
    commit=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
    if commit != PINNED_COMMIT: raise RuntimeError(f'Repo commit mismatch: {commit}')
print('Repository ready:', REPO)

In [ ]:
# Exact ESM helper from the definitive notebook (embedded fallback).
ESM_HELPER_FALLBACK = 'from __future__ import annotations\n\nimport hashlib\nimport json\nimport shutil\nimport zipfile\nfrom pathlib import Path\nfrom typing import Dict, Iterable, List, Tuple\n\nimport numpy as np\nimport pandas as pd\n\nESM_PACKAGE = "fair-esm"\nESM_PACKAGE_VERSION = "2.0.0"\nESM_MODEL_NAME = "esm2_t6_8M_UR50D"\nESM_REPR_LAYER = 6\nESM_DIM = 320\nSEQ_LENGTH = 115\n\n# Corrected aligned-sequence mapping used throughout the manuscript pipeline.\nSITE_SPEC = [\n    {"order": 0, "kabat_site": 33, "python_index": 32, "expected_wt": "Y"},\n    {"order": 1, "kabat_site": 50, "python_index": 49, "expected_wt": "R"},\n    {"order": 2, "kabat_site": 54, "python_index": 54, "expected_wt": "R"},\n    {"order": 3, "kabat_site": 55, "python_index": 55, "expected_wt": "R"},\n    {"order": 4, "kabat_site": 56, "python_index": 56, "expected_wt": "G"},\n    {"order": 5, "kabat_site": 95, "python_index": 98, "expected_wt": "A"},\n    {"order": 6, "kabat_site": 97, "python_index": 100, "expected_wt": "W"},\n    {"order": 7, "kabat_site": 102, "python_index": 103, "expected_wt": "Y"},\n]\n\nAA_ORDER = np.array(sorted("ACDEFGHIKLMNPQRSTVWY"))\nAA_TO_INDEX = {aa: idx for idx, aa in enumerate(AA_ORDER.tolist())}\n\n\ndef sha256_bytes(data: bytes) -> str:\n    return hashlib.sha256(data).hexdigest()\n\n\ndef sha256_file(path: str | Path) -> str:\n    path = Path(path)\n    h = hashlib.sha256()\n    with path.open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef sequence_hash(sequences: Iterable[str]) -> str:\n    payload = "\\n".join(map(str, sequences)).encode("utf-8")\n    return sha256_bytes(payload)\n\n\ndef mapping_signature() -> str:\n    payload = {\n        "model": ESM_MODEL_NAME,\n        "layer": ESM_REPR_LAYER,\n        "esm_dim": ESM_DIM,\n        "sequence_length": SEQ_LENGTH,\n        "special_token_rule": "ESM BOS at token 0; residue i maps to token i+1; EOS excluded",\n        "mean_pooling": "arithmetic mean over residue tokens 1..L inclusive",\n        "site_pooling": "concatenate contextual vectors in fixed SITE_SPEC order; no averaging",\n        "site_spec": SITE_SPEC,\n    }\n    return sha256_bytes(json.dumps(payload, sort_keys=True).encode("utf-8"))\n\n\ndef validate_sequences(sequences: Iterable[str]) -> np.ndarray:\n    sequences = np.asarray(list(map(str, sequences))).astype(str)\n    if len(sequences) == 0:\n        raise ValueError("No sequences supplied.")\n    lengths = np.asarray([len(seq) for seq in sequences], dtype=int)\n    if not np.all(lengths == SEQ_LENGTH):\n        raise ValueError(\n            f"Expected every aligned VH sequence to have length {SEQ_LENGTH}; "\n            f"observed lengths={sorted(set(lengths.tolist()))}."\n        )\n    unsupported = sorted({aa for seq in sequences for aa in seq if aa not in AA_TO_INDEX})\n    if unsupported:\n        raise ValueError(f"Unsupported residues: {unsupported}")\n    return sequences\n\n\ndef audit_site_mapping(sequences: Iterable[str], output_csv: str | Path) -> pd.DataFrame:\n    sequences = validate_sequences(sequences)\n    rows = []\n    for spec in SITE_SPEC:\n        pos = int(spec["python_index"])\n        observed = np.asarray([seq[pos] for seq in sequences]).astype(str)\n        counts = pd.Series(observed).value_counts()\n        rows.append(\n            {\n                "site_order": int(spec["order"]),\n                "kabat_site": int(spec["kabat_site"]),\n                "python_sequence_index_0based": pos,\n                "esm_token_index_0based": pos + 1,\n                "expected_wt_residue": str(spec["expected_wt"]),\n                "expected_wt_present": bool(spec["expected_wt"] in set(observed.tolist())),\n                "n_unique_residues": int(len(counts)),\n                "observed_residues": ",".join(sorted(counts.index.tolist())),\n                "most_common_residue": str(counts.index[0]),\n                "most_common_count": int(counts.iloc[0]),\n            }\n        )\n    audit = pd.DataFrame(rows)\n    # External subsets may not contain every WT residue, so presence is recorded rather than required.\n    output_csv = Path(output_csv)\n    output_csv.parent.mkdir(parents=True, exist_ok=True)\n    audit.to_csv(output_csv, index=False)\n    return audit\n\n\ndef expected_metadata(dataset_name: str, sequences: Iterable[str]) -> Dict:\n    sequences = validate_sequences(sequences)\n    return {\n        "format_version": 2,\n        "dataset_name": str(dataset_name),\n        "sequence_count": int(len(sequences)),\n        "aligned_sequence_length": SEQ_LENGTH,\n        "sequence_sha256": sequence_hash(sequences),\n        "mapping_signature_sha256": mapping_signature(),\n        "esm_package": ESM_PACKAGE,\n        "esm_package_version": ESM_PACKAGE_VERSION,\n        "esm_model": ESM_MODEL_NAME,\n        "esm_repr_layer": ESM_REPR_LAYER,\n        "esm_dim": ESM_DIM,\n        "mean_esm2_dim": ESM_DIM,\n        "site_esm2_dim": len(SITE_SPEC) * ESM_DIM,\n        "site_spec": SITE_SPEC,\n        "mean_pooling": "residue-token arithmetic mean; BOS/EOS excluded",\n        "site_pooling": "ordered concatenation of 8 contextual residue vectors",\n        "unirep_used": False,\n    }\n\n\ndef validate_cache(\n    cache_path: str | Path,\n    metadata_path: str | Path,\n    dataset_name: str,\n    sequences: Iterable[str],\n) -> Tuple[bool, str]:\n    cache_path = Path(cache_path)\n    metadata_path = Path(metadata_path)\n    sequences = validate_sequences(sequences)\n    if not cache_path.exists():\n        return False, "cache file absent"\n    if not metadata_path.exists():\n        return False, "metadata file absent"\n    try:\n        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))\n    except Exception as exc:\n        return False, f"metadata unreadable: {exc!r}"\n    expected = expected_metadata(dataset_name, sequences)\n    for key in [\n        "format_version",\n        "dataset_name",\n        "sequence_count",\n        "aligned_sequence_length",\n        "sequence_sha256",\n        "mapping_signature_sha256",\n        "esm_package_version",\n        "esm_model",\n        "esm_repr_layer",\n        "esm_dim",\n        "mean_esm2_dim",\n        "site_esm2_dim",\n        "unirep_used",\n    ]:\n        if metadata.get(key) != expected.get(key):\n            return False, f"metadata mismatch at {key}: {metadata.get(key)!r} != {expected.get(key)!r}"\n    try:\n        bundle = np.load(cache_path, allow_pickle=False)\n        if bundle["mean_esm2"].shape != (len(sequences), ESM_DIM):\n            return False, f"bad mean_esm2 shape {bundle[\'mean_esm2\'].shape}"\n        if bundle["site_esm2"].shape != (len(sequences), len(SITE_SPEC) * ESM_DIM):\n            return False, f"bad site_esm2 shape {bundle[\'site_esm2\'].shape}"\n        if not np.array_equal(bundle["sequences"].astype(str), sequences):\n            return False, "sequence content/order mismatch"\n    except Exception as exc:\n        return False, f"cache unreadable: {exc!r}"\n    if metadata.get("cache_sha256") and metadata["cache_sha256"] != sha256_file(cache_path):\n        return False, "cache SHA-256 mismatch"\n    return True, "valid"\n\n\ndef find_compatible_cache(\n    search_root: str | Path,\n    cache_filename: str,\n    metadata_filename: str,\n    dataset_name: str,\n    sequences: Iterable[str],\n) -> Tuple[Path | None, Path | None]:\n    search_root = Path(search_root)\n    if not search_root.exists():\n        return None, None\n    for cache_path in search_root.rglob(cache_filename):\n        # Prefer metadata beside the cache, but also accept the requested name.\n        candidates = [cache_path.parent / metadata_filename, cache_path.with_suffix(".meta.json")]\n        for metadata_path in candidates:\n            ok, _ = validate_cache(cache_path, metadata_path, dataset_name, sequences)\n            if ok:\n                return cache_path, metadata_path\n    return None, None\n\n\ndef import_compatible_cache_from_zip(\n    search_root: str | Path,\n    cache_filename: str,\n    metadata_filename: str,\n    dataset_name: str,\n    sequences: Iterable[str],\n    working_cache_path: str | Path,\n    working_metadata_path: str | Path,\n) -> bool:\n    """\n    Import a validated cache directly from a ZIP mounted under /kaggle/input.\n\n    This supports reusing MOLM_EMI_ESM2_Cache_For_Reuse.zip from a previous\n    Kaggle run without manually extracting it.\n    """\n    search_root = Path(search_root)\n    working_cache_path = Path(working_cache_path)\n    working_metadata_path = Path(working_metadata_path)\n\n    if not search_root.exists():\n        return False\n\n    for zip_path in search_root.rglob("*.zip"):\n        try:\n            with zipfile.ZipFile(zip_path, "r") as archive:\n                names = archive.namelist()\n                cache_members = [\n                    name for name in names\n                    if Path(name).name == cache_filename\n                ]\n                metadata_members = [\n                    name for name in names\n                    if Path(name).name == metadata_filename\n                ]\n                if not cache_members or not metadata_members:\n                    continue\n\n                working_cache_path.parent.mkdir(parents=True, exist_ok=True)\n                working_cache_path.write_bytes(\n                    archive.read(cache_members[0])\n                )\n                working_metadata_path.write_bytes(\n                    archive.read(metadata_members[0])\n                )\n\n            ok, reason = validate_cache(\n                working_cache_path,\n                working_metadata_path,\n                dataset_name,\n                sequences,\n            )\n            if ok:\n                print(\n                    f"[{dataset_name}] Imported validated cache from ZIP: "\n                    f"{zip_path}"\n                )\n                return True\n\n            # Never leave an incompatible extracted cache behind.\n            working_cache_path.unlink(missing_ok=True)\n            working_metadata_path.unlink(missing_ok=True)\n            print(\n                f"[{dataset_name}] Ignored incompatible cache ZIP "\n                f"{zip_path.name}: {reason}"\n            )\n        except (zipfile.BadZipFile, KeyError, OSError):\n            continue\n\n    return False\n\n\ndef build_cache(\n    dataset_name: str,\n    sequences: Iterable[str],\n    cache_path: str | Path,\n    metadata_path: str | Path,\n    audit_csv: str | Path,\n    batch_size: int = 64,\n    device: str = "auto",\n) -> Dict:\n    sequences = validate_sequences(sequences)\n    audit_site_mapping(sequences, audit_csv)\n\n    try:\n        import torch\n        import esm\n    except ImportError as exc:\n        raise RuntimeError("Install fair-esm==2.0.0 before building ESM features.") from exc\n\n    if device == "auto":\n        torch_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    else:\n        torch_device = torch.device(device)\n\n    model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()\n    converter = alphabet.get_batch_converter()\n    model = model.eval().to(torch_device)\n\n    mean_features = np.empty((len(sequences), ESM_DIM), dtype=np.float32)\n    site_features = np.empty((len(sequences), len(SITE_SPEC) * ESM_DIM), dtype=np.float32)\n\n    with torch.no_grad():\n        for start in range(0, len(sequences), int(batch_size)):\n            stop = min(start + int(batch_size), len(sequences))\n            batch = [(str(i), seq) for i, seq in enumerate(sequences[start:stop], start=start)]\n            _, batch_strings, tokens = converter(batch)\n            tokens = tokens.to(torch_device)\n            result = model(tokens, repr_layers=[ESM_REPR_LAYER], return_contacts=False)\n            reps = result["representations"][ESM_REPR_LAYER]\n\n            for local_idx, sequence in enumerate(batch_strings):\n                residue_reps = reps[local_idx, 1 : len(sequence) + 1]\n                if tuple(residue_reps.shape) != (SEQ_LENGTH, ESM_DIM):\n                    raise RuntimeError(f"Unexpected residue representation shape {tuple(residue_reps.shape)}")\n                mean_features[start + local_idx] = residue_reps.mean(dim=0).cpu().numpy()\n\n                selected = []\n                for spec in SITE_SPEC:\n                    seq_index = int(spec["python_index"])\n                    token_index = seq_index + 1\n                    expected_aa = sequence[seq_index]\n                    token_id = int(tokens[local_idx, token_index].item())\n                    token_text = alphabet.get_tok(token_id)\n                    if token_text != expected_aa:\n                        raise RuntimeError(\n                            "BOS/token-offset audit failed: "\n                            f"dataset={dataset_name}, row={start+local_idx}, Kabat={spec[\'kabat_site\']}, "\n                            f"sequence_index={seq_index}, token_index={token_index}, "\n                            f"sequence residue={expected_aa!r}, ESM token={token_text!r}."\n                        )\n                    selected.append(reps[local_idx, token_index])\n                site_features[start + local_idx] = torch.cat(selected, dim=0).cpu().numpy()\n            print(f"[{dataset_name}] ESM-2 {stop}/{len(sequences)}", flush=True)\n\n    cache_path = Path(cache_path)\n    metadata_path = Path(metadata_path)\n    cache_path.parent.mkdir(parents=True, exist_ok=True)\n    np.savez_compressed(\n        cache_path,\n        sequences=sequences.astype("U"),\n        mean_esm2=mean_features,\n        site_esm2=site_features,\n    )\n    metadata = expected_metadata(dataset_name, sequences)\n    metadata["cache_sha256"] = sha256_file(cache_path)\n    metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")\n    ok, reason = validate_cache(cache_path, metadata_path, dataset_name, sequences)\n    if not ok:\n        raise RuntimeError(f"New cache did not validate: {reason}")\n    return metadata\n\n\ndef reuse_or_build_cache(\n    dataset_name: str,\n    sequences: Iterable[str],\n    working_cache_path: str | Path,\n    working_metadata_path: str | Path,\n    working_audit_csv: str | Path,\n    search_root: str | Path | None = None,\n    batch_size: int = 64,\n    device: str = "auto",\n) -> Dict:\n    sequences = validate_sequences(sequences)\n    working_cache_path = Path(working_cache_path)\n    working_metadata_path = Path(working_metadata_path)\n    working_audit_csv = Path(working_audit_csv)\n\n    ok, reason = validate_cache(\n        working_cache_path,\n        working_metadata_path,\n        dataset_name,\n        sequences,\n    )\n    if ok:\n        audit_site_mapping(sequences, working_audit_csv)\n        print(f"[{dataset_name}] Reusing validated working cache: {working_cache_path}")\n        return json.loads(working_metadata_path.read_text(encoding="utf-8"))\n\n    if search_root is not None:\n        found_cache, found_meta = find_compatible_cache(\n            search_root,\n            working_cache_path.name,\n            working_metadata_path.name,\n            dataset_name,\n            sequences,\n        )\n        if found_cache is not None:\n            working_cache_path.parent.mkdir(parents=True, exist_ok=True)\n            shutil.copy2(found_cache, working_cache_path)\n            shutil.copy2(found_meta, working_metadata_path)\n            audit_site_mapping(sequences, working_audit_csv)\n            print(f"[{dataset_name}] Imported validated cache from: {found_cache}")\n            return json.loads(working_metadata_path.read_text(encoding="utf-8"))\n\n        imported_from_zip = import_compatible_cache_from_zip(\n            search_root=search_root,\n            cache_filename=working_cache_path.name,\n            metadata_filename=working_metadata_path.name,\n            dataset_name=dataset_name,\n            sequences=sequences,\n            working_cache_path=working_cache_path,\n            working_metadata_path=working_metadata_path,\n        )\n        if imported_from_zip:\n            audit_site_mapping(sequences, working_audit_csv)\n            return json.loads(working_metadata_path.read_text(encoding="utf-8"))\n\n    print(f"[{dataset_name}] Computing cache because {reason}.")\n    return build_cache(\n        dataset_name=dataset_name,\n        sequences=sequences,\n        cache_path=working_cache_path,\n        metadata_path=working_metadata_path,\n        audit_csv=working_audit_csv,\n        batch_size=batch_size,\n        device=device,\n    )\n\n\ndef generate_onehot(sequences: Iterable[str]) -> np.ndarray:\n    sequences = validate_sequences(sequences)\n    output = np.zeros((len(sequences), SEQ_LENGTH, len(AA_ORDER)), dtype=np.float32)\n    for row_idx, sequence in enumerate(sequences):\n        for pos, aa in enumerate(sequence):\n            output[row_idx, pos, AA_TO_INDEX[aa]] = 1.0\n    return output.reshape(len(sequences), -1)\n\n\ndef load_cache(cache_path: str | Path, sequences: Iterable[str]) -> Tuple[np.ndarray, np.ndarray]:\n    sequences = validate_sequences(sequences)\n    bundle = np.load(cache_path, allow_pickle=False)\n    if not np.array_equal(bundle["sequences"].astype(str), sequences):\n        raise RuntimeError("Refusing ESM cache: sequence content/order differs.")\n    return (\n        np.asarray(bundle["mean_esm2"], dtype=np.float32),\n        np.asarray(bundle["site_esm2"], dtype=np.float32),\n    )\n\n\ndef build_five_representations(sequences: Iterable[str], cache_path: str | Path) -> Dict[str, np.ndarray]:\n    sequences = validate_sequences(sequences)\n    onehot = generate_onehot(sequences)\n    mean_esm2, site_esm2 = load_cache(cache_path, sequences)\n    mean_fusion = np.concatenate([onehot, mean_esm2], axis=1).astype(np.float32)\n    site_fusion = np.concatenate([onehot, site_esm2], axis=1).astype(np.float32)\n    expected = {\n        "onehot": (len(sequences), 2300),\n        "mean_esm2": (len(sequences), 320),\n        "mean_fusion": (len(sequences), 2620),\n        "site_esm2": (len(sequences), 2560),\n        "site_fusion": (len(sequences), 4860),\n    }\n    reps = {\n        "onehot": onehot,\n        "mean_esm2": mean_esm2,\n        "mean_fusion": mean_fusion,\n        "site_esm2": site_esm2,\n        "site_fusion": site_fusion,\n    }\n    for key, shape in expected.items():\n        if reps[key].shape != shape:\n            raise RuntimeError(f"{key}: expected {shape}, observed {reps[key].shape}")\n    return reps\n'
COMMON_HELPER_FALLBACK = 'from __future__ import annotations\n\nimport gc\nimport os\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis\nfrom sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, matthews_corrcoef, roc_auc_score)\n\nREPO = Path(os.environ["MOLM_REPO"]).resolve()\nSEED = int(os.environ.get("MOLM_SEED", "42"))\n# Each worker process is launched with MOLM_FEATURE_TYPE.  This value is\n# diagnostic metadata only; it does not affect model computation.\nFEATURE_TYPE = os.environ["MOLM_FEATURE_TYPE"]\nDOMINANCE_WEIGHT = float(os.environ.get("MOLM_DOMINANCE_WEIGHT", "0.03"))\nDOMINANCE_MARGIN = float(os.environ.get("MOLM_DOMINANCE_MARGIN", "0.20"))\nDOMINANCE_WARMUP = int(os.environ.get("MOLM_DOMINANCE_WARMUP", "5"))\nDOMINANCE_MAX_PAIRS = int(os.environ.get("MOLM_DOMINANCE_MAX_PAIRS", "4096"))\nPRIVATE_ADAPTER_DIM = int(os.environ.get("MOLM_PRIVATE_ADAPTER_DIM", "32"))\nPRIVATE_GATE_INIT = float(os.environ.get("MOLM_PRIVATE_GATE_INIT", "-2.0"))\nROUTING_EPS = float(os.environ.get("MOLM_ROUTING_EPS", "1e-12"))\n\nsys.path.insert(0, str(REPO))\nfrom phase0_config import (  # noqa: E402\n    DiagnosticMOLM,\n    config,\n    focal_bce_with_logits,\n    gap_hinge_loss,\n    hard_reset_rng,\n    ranking_loss,\n)\n\n# Locked final protocol.\nconfig.PARETO_LOSS = False\nconfig.EPOCHS = int(os.environ.get("MOLM_EPOCHS", "25"))\nconfig.BATCH_SIZE = int(os.environ.get("MOLM_BATCH_SIZE", "64"))\nconfig.LEARNING_RATE = float(os.environ.get("MOLM_LEARNING_RATE", "5e-5"))\n\ndef task_losses(\n    outputs,\n    y_aff,\n    y_ova,\n    aff_pos_weight,\n    ova_pos_weight,\n):\n    aff = (\n        focal_bce_with_logits(\n            y_aff,\n            outputs["aff_score"],\n            config.FOCAL_GAMMA,\n            aff_pos_weight,\n        )\n        + config.RANKING_WEIGHT_AFF\n        * ranking_loss(\n            outputs["aff_score"],\n            y_aff,\n            config.RANKING_MARGIN,\n        )\n        + config.GAP_WEIGHT_AFF\n        * gap_hinge_loss(\n            outputs["aff_score"],\n            y_aff,\n            config.GAP_MARGIN,\n        )\n    )\n    ova = (\n        focal_bce_with_logits(\n            y_ova,\n            outputs["spec_score"],\n            config.FOCAL_GAMMA,\n            ova_pos_weight,\n        )\n        + config.RANKING_WEIGHT_SPEC\n        * ranking_loss(\n            outputs["spec_score"],\n            y_ova,\n            config.RANKING_MARGIN,\n        )\n        + config.GAP_WEIGHT_SPEC\n        * gap_hinge_loss(\n            outputs["spec_score"],\n            y_ova,\n            config.GAP_MARGIN,\n        )\n    )\n    return aff, ova\n\n\ndef dominance_pair_loss(\n    aff_scores,\n    ova_scores,\n    y_aff,\n    y_ova,\n    margin=0.20,\n    max_pairs=4096,\n):\n    """\n    Coordinate-wise binary dominance:\n      target positive > target negative;\n      OVA negative > OVA positive.\n\n    Only coordinates with a strict label improvement are constrained.\n    Equal coordinates and incomparable phenotype pairs are unconstrained.\n    """\n    aff_scores = aff_scores.reshape(-1).float()\n    ova_scores = ova_scores.reshape(-1).float()\n    y_aff = y_aff.reshape(-1).float().to(aff_scores.device)\n    y_ova = y_ova.reshape(-1).float().to(aff_scores.device)\n\n    desirability = torch.stack([y_aff, 1.0 - y_ova], dim=1)\n    weakly_better = (\n        desirability[:, None, :] >= desirability[None, :, :]\n    ).all(dim=-1)\n    strictly_better = (\n        desirability[:, None, :] > desirability[None, :, :]\n    )\n    pairs = (\n        weakly_better & strictly_better.any(dim=-1)\n    ).nonzero(as_tuple=False)\n\n    if pairs.numel() == 0:\n        return aff_scores.sum() * 0.0\n\n    if len(pairs) > int(max_pairs):\n        choice = torch.randperm(\n            len(pairs),\n            device=pairs.device,\n        )[: int(max_pairs)]\n        pairs = pairs[choice]\n\n    better = pairs[:, 0]\n    worse = pairs[:, 1]\n\n    strict_aff = y_aff[better] > y_aff[worse]\n    strict_ova = (\n        (1.0 - y_ova[better])\n        > (1.0 - y_ova[worse])\n    )\n\n    terms = []\n    if strict_aff.any():\n        terms.append(\n            F.softplus(\n                float(margin)\n                - (\n                    aff_scores[better[strict_aff]]\n                    - aff_scores[worse[strict_aff]]\n                )\n            )\n        )\n    if strict_ova.any():\n        terms.append(\n            F.softplus(\n                float(margin)\n                - (\n                    ova_scores[worse[strict_ova]]\n                    - ova_scores[better[strict_ova]]\n                )\n            )\n        )\n\n    if not terms:\n        return aff_scores.sum() * 0.0\n    return torch.cat(terms).mean()\n\n\nclass LowRankPrivateAdapter(nn.Module):\n    def __init__(\n        self,\n        input_dim,\n        output_dim,\n        bottleneck_dim=32,\n        gate_init=-2.0,\n    ):\n        super().__init__()\n        self.down = nn.Linear(input_dim, bottleneck_dim)\n        self.norm = nn.LayerNorm(bottleneck_dim)\n        self.up = nn.Linear(bottleneck_dim, output_dim)\n        self.gate_logit = nn.Parameter(\n            torch.tensor(float(gate_init))\n        )\n        nn.init.normal_(\n            self.up.weight,\n            mean=0.0,\n            std=1e-3,\n        )\n        nn.init.zeros_(self.up.bias)\n\n    def forward(self, inputs):\n        correction = self.up(\n            F.gelu(\n                self.norm(\n                    self.down(inputs)\n                )\n            )\n        )\n        return (\n            torch.sigmoid(self.gate_logit)\n            * correction\n        )\n\n    def gate_value(self):\n        return float(\n            torch.sigmoid(\n                self.gate_logit.detach()\n            ).cpu()\n        )\n\n\nclass ParetoRoutedMOLM(DiagnosticMOLM):\n    def __init__(\n        self,\n        *args,\n        private_adapter_dim=32,\n        private_gate_init=-2.0,\n        **kwargs,\n    ):\n        super().__init__(*args, **kwargs)\n        shared_out_dim = (\n            self._shared_dims[-1]\n            if self._shared_dims\n            else self.input_dim\n        )\n        self.aff_private_adapter = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n        self.spec_private_adapter = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n        self.to(\n            next(self.parameters()).device\n        )\n\n    def forward(self, inputs, training=None):\n        inputs = self._coerce_input(inputs)\n        shared = self.run_shared(\n            inputs,\n            training,\n        )\n        aff_routed = (\n            shared\n            + self.aff_private_adapter(inputs)\n        )\n        spec_routed = (\n            shared\n            + self.spec_private_adapter(inputs)\n        )\n        aff_latent = self.run_tower(\n            aff_routed,\n            self.aff_layers,\n            self.aff_proj,\n            self.aff_proj_norm,\n            training,\n        )\n        spec_latent = self.run_tower(\n            spec_routed,\n            self.spec_layers,\n            self.spec_proj,\n            self.spec_proj_norm,\n            training,\n        )\n        aff_logit = self.aff_head(\n            aff_latent\n        ).squeeze(-1)\n        spec_logit = self.spec_head(\n            spec_latent\n        ).squeeze(-1)\n\n        return {\n            "aff_score": aff_logit,\n            "spec_score": spec_logit,\n            "aff_latent": aff_latent,\n            "spec_latent": spec_latent,\n            "shared": shared,\n        }\n\n    def private_gate_values(self):\n        return {\n            "aff_private_gate": (\n                self.aff_private_adapter.gate_value()\n            ),\n            "ova_private_gate": (\n                self.spec_private_adapter.gate_value()\n            ),\n        }\n\n\nclass IndependentTaskNet(nn.Module):\n    """Independent neural baseline for one binary task."""\n\n    def __init__(self, input_dim):\n        super().__init__()\n        dims = (\n            list(config.SHARED_DIMS)\n            + list(config.TOWER_DIMS)\n        )\n        self.blocks = DiagnosticMOLM._make_blocks(\n            input_dim,\n            dims,\n            config.DROPOUT_RATE,\n        )\n        output_dim = (\n            dims[-1]\n            if dims\n            else input_dim\n        )\n        self.proj = nn.Linear(\n            output_dim,\n            config.LATENT_DIM,\n        )\n        self.proj_norm = nn.LayerNorm(\n            config.LATENT_DIM\n        )\n        self.head = nn.Linear(\n            config.LATENT_DIM,\n            1,\n        )\n\n    @staticmethod\n    def _dropout(layer, inputs, training):\n        if training is None:\n            return layer(inputs)\n        return F.dropout(\n            inputs,\n            p=layer.p,\n            training=training,\n        )\n\n    def forward(self, inputs, training=None):\n        for index in range(\n            0,\n            len(self.blocks),\n            3,\n        ):\n            inputs = self.blocks[index](inputs)\n            inputs = self.blocks[index + 1](inputs)\n            inputs = F.gelu(inputs)\n            inputs = self._dropout(\n                self.blocks[index + 2],\n                inputs,\n                training,\n            )\n        latent = self.proj_norm(\n            self.proj(inputs)\n        )\n        score = self.head(\n            latent\n        ).squeeze(-1)\n        return score\n\n\nclass IndependentNNPair(nn.Module):\n    """Two independently parameterized task networks."""\n\n    def __init__(self, input_dim):\n        super().__init__()\n        self.aff_net = IndependentTaskNet(input_dim)\n        self.ova_net = IndependentTaskNet(input_dim)\n\n    def forward(self, inputs, training=None):\n        return {\n            "aff_score": self.aff_net(\n                inputs,\n                training=training,\n            ),\n            "spec_score": self.ova_net(\n                inputs,\n                training=training,\n            ),\n        }\n\n\nclass RoutedTrainer:\n    def __init__(\n        self,\n        model,\n        aff_pos_weight,\n        ova_pos_weight,\n    ):\n        self.device = torch.device(\n            "cuda"\n            if torch.cuda.is_available()\n            else "cpu"\n        )\n        self.model = model.to(self.device)\n        self.aff_pos_weight = float(\n            aff_pos_weight\n        )\n        self.ova_pos_weight = float(\n            ova_pos_weight\n        )\n        self.optimizer = torch.optim.AdamW(\n            self.model.parameters(),\n            lr=config.LEARNING_RATE,\n            weight_decay=1e-4,\n        )\n        self.rows = []\n\n    @staticmethod\n    def _replace_none(\n        gradients,\n        parameters,\n    ):\n        return [\n            (\n                torch.zeros_like(parameter)\n                if gradient is None\n                else gradient\n            )\n            for gradient, parameter\n            in zip(\n                gradients,\n                parameters,\n            )\n        ]\n\n    @staticmethod\n    def _dot(first, second):\n        return sum(\n            (left * right).sum()\n            for left, right\n            in zip(first, second)\n        )\n\n    @staticmethod\n    def _norm_squared(gradients):\n        return sum(\n            (gradient * gradient).sum()\n            for gradient in gradients\n        )\n\n    def fit(\n        self,\n        X,\n        y_aff,\n        y_ova,\n        label,\n    ):\n        dataset = (\n            torch.utils.data.TensorDataset(\n                torch.as_tensor(\n                    X,\n                    dtype=torch.float32,\n                ),\n                torch.as_tensor(\n                    y_aff,\n                    dtype=torch.float32,\n                ),\n                torch.as_tensor(\n                    y_ova,\n                    dtype=torch.float32,\n                ),\n            )\n        )\n        generator = (\n            torch.Generator().manual_seed(SEED)\n        )\n        loader = torch.utils.data.DataLoader(\n            dataset,\n            batch_size=config.BATCH_SIZE,\n            shuffle=True,\n            generator=generator,\n        )\n\n        shared_parameters = [\n            parameter\n            for parameter\n            in self.model.shared_layers.parameters()\n            if parameter.requires_grad\n        ]\n\n        for epoch in range(config.EPOCHS):\n            dominance_active = (\n                DOMINANCE_WEIGHT > 0\n                and epoch >= DOMINANCE_WARMUP\n            )\n            totals = []\n            dominance_values = []\n            conflicts = []\n            pre_cosines = []\n            post_cosines = []\n\n            for (\n                X_batch,\n                y_aff_batch,\n                y_ova_batch,\n            ) in loader:\n                X_batch = X_batch.to(self.device)\n                y_aff_batch = y_aff_batch.to(\n                    self.device\n                )\n                y_ova_batch = y_ova_batch.to(\n                    self.device\n                )\n\n                self.model.train()\n                self.optimizer.zero_grad(\n                    set_to_none=True\n                )\n\n                outputs = self.model(\n                    X_batch,\n                    training=True,\n                )\n                aff_loss, ova_loss = task_losses(\n                    outputs,\n                    y_aff_batch,\n                    y_ova_batch,\n                    self.aff_pos_weight,\n                    self.ova_pos_weight,\n                )\n                dominance = dominance_pair_loss(\n                    outputs["aff_score"],\n                    outputs["spec_score"],\n                    y_aff_batch,\n                    y_ova_batch,\n                    margin=DOMINANCE_MARGIN,\n                    max_pairs=DOMINANCE_MAX_PAIRS,\n                )\n\n                aff_gradients = torch.autograd.grad(\n                    aff_loss,\n                    shared_parameters,\n                    retain_graph=True,\n                    allow_unused=True,\n                )\n                ova_gradients = torch.autograd.grad(\n                    ova_loss,\n                    shared_parameters,\n                    retain_graph=True,\n                    allow_unused=True,\n                )\n                aff_gradients = self._replace_none(\n                    aff_gradients,\n                    shared_parameters,\n                )\n                ova_gradients = self._replace_none(\n                    ova_gradients,\n                    shared_parameters,\n                )\n\n                if dominance_active:\n                    dominance_gradients = (\n                        torch.autograd.grad(\n                            dominance,\n                            shared_parameters,\n                            retain_graph=True,\n                            allow_unused=True,\n                        )\n                    )\n                    dominance_gradients = (\n                        self._replace_none(\n                            dominance_gradients,\n                            shared_parameters,\n                        )\n                    )\n                else:\n                    dominance_gradients = [\n                        torch.zeros_like(parameter)\n                        for parameter\n                        in shared_parameters\n                    ]\n\n                dot_product = self._dot(\n                    aff_gradients,\n                    ova_gradients,\n                )\n                aff_norm = self._norm_squared(\n                    aff_gradients\n                )\n                ova_norm = self._norm_squared(\n                    ova_gradients\n                )\n                pre_cosine = dot_product / (\n                    torch.sqrt(\n                        aff_norm * ova_norm\n                    )\n                    + ROUTING_EPS\n                )\n                conflict = bool(\n                    dot_product.detach().cpu() < 0\n                )\n\n                if conflict:\n                    aff_coefficient = (\n                        dot_product\n                        / (\n                            ova_norm\n                            + ROUTING_EPS\n                        )\n                    )\n                    ova_coefficient = (\n                        dot_product\n                        / (\n                            aff_norm\n                            + ROUTING_EPS\n                        )\n                    )\n                    routed_aff = [\n                        gradient_aff\n                        - aff_coefficient\n                        * gradient_ova\n                        for gradient_aff, gradient_ova\n                        in zip(\n                            aff_gradients,\n                            ova_gradients,\n                        )\n                    ]\n                    routed_ova = [\n                        gradient_ova\n                        - ova_coefficient\n                        * gradient_aff\n                        for gradient_aff, gradient_ova\n                        in zip(\n                            aff_gradients,\n                            ova_gradients,\n                        )\n                    ]\n                else:\n                    routed_aff = aff_gradients\n                    routed_ova = ova_gradients\n\n                post_dot = self._dot(\n                    routed_aff,\n                    routed_ova,\n                )\n                post_aff_norm = self._norm_squared(\n                    routed_aff\n                )\n                post_ova_norm = self._norm_squared(\n                    routed_ova\n                )\n                post_cosine = post_dot / (\n                    torch.sqrt(\n                        post_aff_norm\n                        * post_ova_norm\n                    )\n                    + ROUTING_EPS\n                )\n\n                total = aff_loss + ova_loss\n                if dominance_active:\n                    total = (\n                        total\n                        + DOMINANCE_WEIGHT\n                        * dominance\n                    )\n\n                total.backward()\n\n                for (\n                    parameter,\n                    gradient_aff,\n                    gradient_ova,\n                    gradient_dominance,\n                ) in zip(\n                    shared_parameters,\n                    routed_aff,\n                    routed_ova,\n                    dominance_gradients,\n                ):\n                    routed_gradient = (\n                        gradient_aff\n                        + gradient_ova\n                    )\n                    if dominance_active:\n                        routed_gradient = (\n                            routed_gradient\n                            + DOMINANCE_WEIGHT\n                            * gradient_dominance\n                        )\n                    parameter.grad = (\n                        routed_gradient.detach().clone()\n                    )\n\n                torch.nn.utils.clip_grad_norm_(\n                    self.model.parameters(),\n                    1.0,\n                )\n                self.optimizer.step()\n\n                totals.append(\n                    float(total.detach().cpu())\n                )\n                dominance_values.append(\n                    float(\n                        dominance.detach().cpu()\n                    )\n                )\n                conflicts.append(float(conflict))\n                pre_cosines.append(\n                    float(\n                        pre_cosine.detach().cpu()\n                    )\n                )\n                post_cosines.append(\n                    float(\n                        post_cosine.detach().cpu()\n                    )\n                )\n\n            gates = (\n                self.model.private_gate_values()\n            )\n            row = {\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "model": "Routed-MOLM",\n                "epoch": epoch + 1,\n                "dominance_active": (\n                    dominance_active\n                ),\n                "mean_total_loss": float(\n                    np.mean(totals)\n                ),\n                "mean_dominance_loss": float(\n                    np.mean(\n                        dominance_values\n                    )\n                ),\n                "conflict_rate": float(\n                    np.mean(conflicts)\n                ),\n                "mean_pre_route_cosine": float(\n                    np.mean(pre_cosines)\n                ),\n                "mean_post_route_cosine": float(\n                    np.mean(post_cosines)\n                ),\n                **gates,\n            }\n            self.rows.append(row)\n\n            if (\n                epoch == 0\n                or (epoch + 1) % 5 == 0\n            ):\n                print(\n                    f"{label} epoch={epoch+1:02d}/"\n                    f"{config.EPOCHS} "\n                    f"loss={row[\'mean_total_loss\']:.4f} "\n                    f"conflict={row[\'conflict_rate\']:.3f}",\n                    flush=True,\n                )\n\n        return self\n\n\nclass NNTrainer:\n    def __init__(\n        self,\n        model,\n        aff_pos_weight,\n        ova_pos_weight,\n    ):\n        self.device = torch.device(\n            "cuda"\n            if torch.cuda.is_available()\n            else "cpu"\n        )\n        self.model = model.to(self.device)\n        self.aff_pos_weight = float(\n            aff_pos_weight\n        )\n        self.ova_pos_weight = float(\n            ova_pos_weight\n        )\n        self.optimizer = torch.optim.AdamW(\n            self.model.parameters(),\n            lr=config.LEARNING_RATE,\n            weight_decay=1e-4,\n        )\n        self.rows = []\n\n    def fit(\n        self,\n        X,\n        y_aff,\n        y_ova,\n        label,\n    ):\n        dataset = (\n            torch.utils.data.TensorDataset(\n                torch.as_tensor(\n                    X,\n                    dtype=torch.float32,\n                ),\n                torch.as_tensor(\n                    y_aff,\n                    dtype=torch.float32,\n                ),\n                torch.as_tensor(\n                    y_ova,\n                    dtype=torch.float32,\n                ),\n            )\n        )\n        generator = (\n            torch.Generator().manual_seed(\n                SEED + 100000\n            )\n        )\n        loader = torch.utils.data.DataLoader(\n            dataset,\n            batch_size=config.BATCH_SIZE,\n            shuffle=True,\n            generator=generator,\n        )\n\n        for epoch in range(config.EPOCHS):\n            totals = []\n            for (\n                X_batch,\n                y_aff_batch,\n                y_ova_batch,\n            ) in loader:\n                X_batch = X_batch.to(self.device)\n                y_aff_batch = y_aff_batch.to(\n                    self.device\n                )\n                y_ova_batch = y_ova_batch.to(\n                    self.device\n                )\n\n                self.model.train()\n                self.optimizer.zero_grad(\n                    set_to_none=True\n                )\n                outputs = self.model(\n                    X_batch,\n                    training=True,\n                )\n                aff_loss = focal_bce_with_logits(\n                    y_aff_batch,\n                    outputs["aff_score"],\n                    config.FOCAL_GAMMA,\n                    self.aff_pos_weight,\n                )\n                ova_loss = focal_bce_with_logits(\n                    y_ova_batch,\n                    outputs["spec_score"],\n                    config.FOCAL_GAMMA,\n                    self.ova_pos_weight,\n                )\n                total = aff_loss + ova_loss\n                total.backward()\n                torch.nn.utils.clip_grad_norm_(\n                    self.model.parameters(),\n                    1.0,\n                )\n                self.optimizer.step()\n                totals.append(\n                    float(total.detach().cpu())\n                )\n\n            row = {\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "model": "NN",\n                "epoch": epoch + 1,\n                "dominance_active": False,\n                "mean_total_loss": float(\n                    np.mean(totals)\n                ),\n                "mean_dominance_loss": np.nan,\n                "conflict_rate": np.nan,\n                "mean_pre_route_cosine": np.nan,\n                "mean_post_route_cosine": np.nan,\n                "aff_private_gate": np.nan,\n                "ova_private_gate": np.nan,\n            }\n            self.rows.append(row)\n\n            if (\n                epoch == 0\n                or (epoch + 1) % 5 == 0\n            ):\n                print(\n                    f"{label} epoch={epoch+1:02d}/"\n                    f"{config.EPOCHS} "\n                    f"loss={row[\'mean_total_loss\']:.4f}",\n                    flush=True,\n                )\n\n        return self\n\n\ndef make_routed(input_dim):\n    return ParetoRoutedMOLM(\n        input_dim=input_dim,\n        latent_dim=config.LATENT_DIM,\n        shared_dims=config.SHARED_DIMS,\n        tower_dims=config.TOWER_DIMS,\n        dropout_rate=config.DROPOUT_RATE,\n        grl_lambda=config.GRL_LAMBDA,\n        private_adapter_dim=PRIVATE_ADAPTER_DIM,\n        private_gate_init=PRIVATE_GATE_INIT,\n    )\n\n\ndef predict_torch_model(model, X):\n    model.eval()\n    device = next(model.parameters()).device\n    with torch.no_grad():\n        outputs = model(\n            torch.as_tensor(\n                X,\n                dtype=torch.float32,\n                device=device,\n            ),\n            training=False,\n        )\n    return (\n        outputs["aff_score"]\n        .detach()\n        .cpu()\n        .numpy()\n        .reshape(-1),\n        outputs["spec_score"]\n        .detach()\n        .cpu()\n        .numpy()\n        .reshape(-1),\n    )\n\n\ndef binary_metrics(y_true, score):\n    y_true = np.asarray(\n        y_true,\n        dtype=int,\n    ).reshape(-1)\n    score = np.asarray(\n        score,\n        dtype=float,\n    ).reshape(-1)\n    y_pred = (score >= 0.0).astype(int)\n\n    tn, fp, fn, tp = confusion_matrix(\n        y_true,\n        y_pred,\n        labels=[0, 1],\n    ).ravel()\n\n    sensitivity = (\n        tp / (tp + fn)\n        if (tp + fn)\n        else np.nan\n    )\n    specificity = (\n        tn / (tn + fp)\n        if (tn + fp)\n        else np.nan\n    )\n    npv = (\n        tn / (tn + fn)\n        if (tn + fn)\n        else np.nan\n    )\n\n    if len(np.unique(y_true)) == 2:\n        auroc = roc_auc_score(\n            y_true,\n            score,\n        )\n        auprc = average_precision_score(\n            y_true,\n            score,\n        )\n    else:\n        auroc = np.nan\n        auprc = np.nan\n\n    return {\n        "n": int(len(y_true)),\n        "n_negative": int((y_true == 0).sum()),\n        "n_positive": int((y_true == 1).sum()),\n        "accuracy": float(\n            accuracy_score(y_true, y_pred)\n        ),\n        "balanced_accuracy": float(\n            balanced_accuracy_score(\n                y_true,\n                y_pred,\n            )\n        ),\n        "mcc": float(\n            matthews_corrcoef(\n                y_true,\n                y_pred,\n            )\n        ),\n        "f1": float(\n            f1_score(\n                y_true,\n                y_pred,\n                zero_division=0,\n            )\n        ),\n        "sensitivity": float(sensitivity),\n        "specificity": float(specificity),\n        "npv": float(npv),\n        "auroc": float(auroc),\n        "auprc": float(auprc),\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tp": int(tp),\n    }\n\n\n\ndef train_lda_pair(X_train, y_aff_train, y_ova_train):\n    aff = LinearDiscriminantAnalysis(solver="svd")\n    ova = LinearDiscriminantAnalysis(solver="svd")\n    aff.fit(X_train, np.asarray(y_aff_train, dtype=int))\n    ova.fit(X_train, np.asarray(y_ova_train, dtype=int))\n    return aff, ova\n\n\ndef predict_lda_pair(pair, X):\n    aff, ova = pair\n    return (\n        np.asarray(aff.decision_function(X), dtype=float).reshape(-1),\n        np.asarray(ova.decision_function(X), dtype=float).reshape(-1),\n    )\n\n\ndef fit_routed(X_train, y_aff_train, y_ova_train, seed, label="Routed-MOLM"):\n    os.environ["MOLM_SEED"] = str(seed)\n    hard_reset_rng(int(seed), label)\n    aff_pos_weight = float((np.asarray(y_aff_train) == 0).sum() / max(int((np.asarray(y_aff_train) == 1).sum()), 1))\n    ova_pos_weight = float((np.asarray(y_ova_train) == 0).sum() / max(int((np.asarray(y_ova_train) == 1).sum()), 1))\n    model = make_routed(X_train.shape[1])\n    trainer = RoutedTrainer(model, aff_pos_weight, ova_pos_weight)\n    trainer.fit(X_train, y_aff_train, y_ova_train, label)\n    return model, trainer\n\n\ndef fit_nn(X_train, y_aff_train, y_ova_train, seed, label="NN"):\n    os.environ["MOLM_SEED"] = str(seed)\n    hard_reset_rng(int(seed) + 100000, label)\n    aff_pos_weight = float((np.asarray(y_aff_train) == 0).sum() / max(int((np.asarray(y_aff_train) == 1).sum()), 1))\n    ova_pos_weight = float((np.asarray(y_ova_train) == 0).sum() / max(int((np.asarray(y_ova_train) == 1).sum()), 1))\n    model = IndependentNNPair(X_train.shape[1])\n    trainer = NNTrainer(model, aff_pos_weight, ova_pos_weight)\n    trainer.fit(X_train, y_aff_train, y_ova_train, label)\n    return model, trainer\n'
COMPONENT_WORKER_SOURCE = 'from __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport os\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom scipy.stats import spearmanr\n\n# Required env must be set before importing the shared helper.\nREPO = Path(os.environ["MOLM_REPO"]).resolve()\nFEATURE_TYPE = os.environ["MOLM_FEATURE_TYPE"]\nSEED = int(os.environ.get("MOLM_SEED", "42"))\nSTAGE = os.environ.get("MOLM_STAGE", "mutation")\nARM_SET = os.environ.get("MOLM_ARM_SET", "core")\nOUT_DIR = Path(os.environ["MOLM_JOB_OUTPUT"]).resolve()\nEMI_FEATURE_PATH = Path(os.environ["MOLM_EMI_FEATURE_PATH"]).resolve()\nEXTERNAL_FEATURE_PATH = Path(os.environ.get("MOLM_EXTERNAL_FEATURE_PATH", "")).resolve() if os.environ.get("MOLM_EXTERNAL_FEATURE_PATH") else None\nHOLDOUT_PATH = Path(os.environ.get("MOLM_HOLDOUT_PATH", "")).resolve() if os.environ.get("MOLM_HOLDOUT_PATH") else None\nLOCK_HASH = os.environ["MOLM_LOCK_HASH"]\nSMOKE = os.environ.get("MOLM_SMOKE", "0") == "1"\n\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nfrom molm_definitive_common import (  # noqa: E402\n    DOMINANCE_MARGIN,\n    DOMINANCE_MAX_PAIRS,\n    DOMINANCE_WARMUP,\n    DOMINANCE_WEIGHT,\n    PRIVATE_ADAPTER_DIM,\n    PRIVATE_GATE_INIT,\n    ROUTING_EPS,\n    IndependentNNPair,\n    LowRankPrivateAdapter,\n    binary_metrics,\n    config,\n    dominance_pair_loss,\n    hard_reset_rng,\n    predict_torch_model,\n    task_losses,\n)\n\n# Pinned repository model.\nimport sys\nsys.path.insert(0, str(REPO))\nfrom phase0_config import DiagnosticMOLM  # noqa: E402\n\n\n@dataclass(frozen=True)\nclass ArmSpec:\n    name: str\n    architecture: str  # shared | private | capacity_shared | independent\n    dominance: bool\n    pcgrad: bool\n    label: str\n\n\nARM_SPECS = {\n    "shared_base": ArmSpec(\n        "shared_base", "shared", False, False,\n        "A Shared Base MOLM",\n    ),\n    "shared_dom": ArmSpec(\n        "shared_dom", "shared", True, False,\n        "B Shared + Dominance",\n    ),\n    "shared_dom_pcgrad": ArmSpec(\n        "shared_dom_pcgrad", "shared", True, True,\n        "C Shared + Dominance + PCGrad",\n    ),\n    "shared_dom_private": ArmSpec(\n        "shared_dom_private", "private", True, False,\n        "D Shared + Dominance + Private",\n    ),\n    "full_routed": ArmSpec(\n        "full_routed", "private", True, True,\n        "E Full Routed-MOLM",\n    ),\n    "independent_st_dom": ArmSpec(\n        "independent_st_dom", "independent", True, False,\n        "F Independent-parameter + matched joint loss",\n    ),\n    "shared_capacity_dom": ArmSpec(\n        "shared_capacity_dom", "capacity_shared", True, False,\n        "G Capacity-Matched Shared + Dominance",\n    ),\n    "shared_capacity_dom_pcgrad": ArmSpec(\n        "shared_capacity_dom_pcgrad", "capacity_shared", True, True,\n        "H Capacity-Matched Shared + Dominance + PCGrad",\n    ),\n}\n\nCORE_ARMS = [\n    "shared_base",\n    "shared_dom",\n    "shared_dom_pcgrad",\n    "shared_dom_private",\n    "full_routed",\n    "independent_st_dom",\n    "shared_capacity_dom",\n    "shared_capacity_dom_pcgrad",\n]\nSITE_CONTROL_ARMS = [\n    "shared_dom",\n    "full_routed",\n    "independent_st_dom",\n]\n\nif ARM_SET == "core":\n    ARMS = CORE_ARMS\nelif ARM_SET == "site_control":\n    ARMS = SITE_CONTROL_ARMS\nelif ARM_SET == "smoke":\n    ARMS = CORE_ARMS\nelse:\n    raise ValueError(f"Unknown ARM_SET={ARM_SET!r}")\n\n\nclass PrivateAblationMOLM(DiagnosticMOLM):\n    """Task-private low-rank corrections on top of the same shared encoder."""\n\n    def __init__(\n        self,\n        *args,\n        private_adapter_dim=32,\n        private_gate_init=-2.0,\n        **kwargs,\n    ):\n        super().__init__(*args, **kwargs)\n        shared_out_dim = self._shared_dims[-1] if self._shared_dims else self.input_dim\n        self.aff_private_adapter = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n        self.spec_private_adapter = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n\n    def forward(self, inputs, training=None):\n        inputs = self._coerce_input(inputs)\n        shared = self.run_shared(inputs, training)\n        aff_private = self.aff_private_adapter(inputs)\n        ova_private = self.spec_private_adapter(inputs)\n        aff_routed = shared + aff_private\n        ova_routed = shared + ova_private\n\n        aff_latent = self.run_tower(\n            aff_routed,\n            self.aff_layers,\n            self.aff_proj,\n            self.aff_proj_norm,\n            training,\n        )\n        ova_latent = self.run_tower(\n            ova_routed,\n            self.spec_layers,\n            self.spec_proj,\n            self.spec_proj_norm,\n            training,\n        )\n        aff_score = self.aff_head(aff_latent).squeeze(-1)\n        ova_score = self.spec_head(ova_latent).squeeze(-1)\n        return {\n            "aff_score": aff_score,\n            "spec_score": ova_score,\n            "aff_latent": aff_latent,\n            "spec_latent": ova_latent,\n            "shared": shared,\n            "aff_private": aff_private,\n            "ova_private": ova_private,\n        }\n\n    def private_gate_values(self):\n        return {\n            "aff_private_gate": self.aff_private_adapter.gate_value(),\n            "ova_private_gate": self.spec_private_adapter.gate_value(),\n        }\n\n\nclass CapacityMatchedSharedMOLM(DiagnosticMOLM):\n    """\n    Parameter-count control.\n\n    Two adapters with the same shapes as the task-private arm are present, but\n    BOTH tasks receive the same averaged correction. Therefore this adds\n    approximately the same adapter parameter budget without task-private routing.\n    """\n\n    def __init__(\n        self,\n        *args,\n        private_adapter_dim=32,\n        private_gate_init=-2.0,\n        **kwargs,\n    ):\n        super().__init__(*args, **kwargs)\n        shared_out_dim = self._shared_dims[-1] if self._shared_dims else self.input_dim\n        self.capacity_adapter_1 = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n        self.capacity_adapter_2 = LowRankPrivateAdapter(\n            self.input_dim,\n            shared_out_dim,\n            bottleneck_dim=private_adapter_dim,\n            gate_init=private_gate_init,\n        )\n\n    def forward(self, inputs, training=None):\n        inputs = self._coerce_input(inputs)\n        shared = self.run_shared(inputs, training)\n        correction_1 = self.capacity_adapter_1(inputs)\n        correction_2 = self.capacity_adapter_2(inputs)\n        shared_correction = 0.5 * (correction_1 + correction_2)\n        routed = shared + shared_correction\n\n        aff_latent = self.run_tower(\n            routed,\n            self.aff_layers,\n            self.aff_proj,\n            self.aff_proj_norm,\n            training,\n        )\n        ova_latent = self.run_tower(\n            routed,\n            self.spec_layers,\n            self.spec_proj,\n            self.spec_proj_norm,\n            training,\n        )\n        return {\n            "aff_score": self.aff_head(aff_latent).squeeze(-1),\n            "spec_score": self.spec_head(ova_latent).squeeze(-1),\n            "aff_latent": aff_latent,\n            "spec_latent": ova_latent,\n            "shared": shared,\n            "shared_capacity_correction": shared_correction,\n        }\n\n    def capacity_gate_values(self):\n        return {\n            "capacity_gate_1": self.capacity_adapter_1.gate_value(),\n            "capacity_gate_2": self.capacity_adapter_2.gate_value(),\n        }\n\n\ndef make_model(input_dim: int, spec: ArmSpec):\n    kwargs = dict(\n        input_dim=input_dim,\n        latent_dim=config.LATENT_DIM,\n        shared_dims=config.SHARED_DIMS,\n        tower_dims=config.TOWER_DIMS,\n        dropout_rate=config.DROPOUT_RATE,\n        grl_lambda=config.GRL_LAMBDA,\n    )\n    if spec.architecture == "shared":\n        return DiagnosticMOLM(**kwargs)\n    if spec.architecture == "private":\n        return PrivateAblationMOLM(\n            **kwargs,\n            private_adapter_dim=PRIVATE_ADAPTER_DIM,\n            private_gate_init=PRIVATE_GATE_INIT,\n        )\n    if spec.architecture == "capacity_shared":\n        return CapacityMatchedSharedMOLM(\n            **kwargs,\n            private_adapter_dim=PRIVATE_ADAPTER_DIM,\n            private_gate_init=PRIVATE_GATE_INIT,\n        )\n    if spec.architecture == "independent":\n        return IndependentNNPair(input_dim)\n    raise ValueError(spec.architecture)\n\n\ndef trainable_parameter_count(model):\n    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))\n\n\ndef _replace_none(grads, params):\n    return [\n        torch.zeros_like(p) if g is None else g\n        for g, p in zip(grads, params)\n    ]\n\n\ndef _dot(a, b):\n    if not a:\n        return torch.tensor(float("nan"))\n    return sum((x * y).sum() for x, y in zip(a, b))\n\n\ndef _norm_sq(a):\n    if not a:\n        return torch.tensor(float("nan"))\n    return sum((x * x).sum() for x in a)\n\n\ndef _cosine(a, b):\n    dot = _dot(a, b)\n    if not torch.isfinite(dot):\n        return torch.tensor(float("nan"), device=a[0].device if a else "cpu")\n    return dot / (torch.sqrt(_norm_sq(a) * _norm_sq(b)) + ROUTING_EPS)\n\n\ndef _grad_norm(a):\n    if not a:\n        return float("nan")\n    value = torch.sqrt(_norm_sq(a))\n    return float(value.detach().cpu())\n\n\ndef _ratio_norm(numerator, denominator):\n    num = numerator.norm(dim=1)\n    den = denominator.norm(dim=1).clamp_min(1e-12)\n    return float((num / den).mean().detach().cpu())\n\n\nclass ControlledTrainer:\n    def __init__(self, model, spec, aff_pos_weight, ova_pos_weight):\n        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        self.model = model.to(self.device)\n        self.spec = spec\n        self.aff_pos_weight = float(aff_pos_weight)\n        self.ova_pos_weight = float(ova_pos_weight)\n        self.optimizer = torch.optim.AdamW(\n            self.model.parameters(),\n            lr=config.LEARNING_RATE,\n            weight_decay=1e-4,\n        )\n        self.rows = []\n\n    def fit(self, X, y_aff, y_ova, label):\n        dataset = torch.utils.data.TensorDataset(\n            torch.as_tensor(X, dtype=torch.float32),\n            torch.as_tensor(y_aff, dtype=torch.float32),\n            torch.as_tensor(y_ova, dtype=torch.float32),\n        )\n        generator = torch.Generator().manual_seed(SEED)\n        loader = torch.utils.data.DataLoader(\n            dataset,\n            batch_size=config.BATCH_SIZE,\n            shuffle=True,\n            generator=generator,\n        )\n\n        has_shared = self.spec.architecture != "independent"\n        shared_params = (\n            [\n                p for p in self.model.shared_layers.parameters()\n                if p.requires_grad\n            ]\n            if has_shared else []\n        )\n\n        for epoch in range(config.EPOCHS):\n            dom_active = bool(\n                self.spec.dominance\n                and DOMINANCE_WEIGHT > 0\n                and epoch >= DOMINANCE_WARMUP\n            )\n            totals, dom_values = [], []\n            conflicts, pre_cos, post_cos = [], [], []\n            aff_norms, ova_norms, proj_fracs = [], [], []\n            private_aff_ratios, private_ova_ratios = [], []\n            capacity_ratios = []\n\n            epoch_aff_sum = [\n                torch.zeros_like(p, device=self.device) for p in shared_params\n            ]\n            epoch_ova_sum = [\n                torch.zeros_like(p, device=self.device) for p in shared_params\n            ]\n\n            for Xb, yab, yob in loader:\n                Xb = Xb.to(self.device)\n                yab = yab.to(self.device)\n                yob = yob.to(self.device)\n\n                self.model.train()\n                self.optimizer.zero_grad(set_to_none=True)\n\n                outputs = self.model(Xb, training=True)\n                aff_loss, ova_loss = task_losses(\n                    outputs,\n                    yab,\n                    yob,\n                    self.aff_pos_weight,\n                    self.ova_pos_weight,\n                )\n\n                if self.spec.dominance:\n                    dom = dominance_pair_loss(\n                        outputs["aff_score"],\n                        outputs["spec_score"],\n                        yab,\n                        yob,\n                        margin=DOMINANCE_MARGIN,\n                        max_pairs=DOMINANCE_MAX_PAIRS,\n                    )\n                else:\n                    dom = outputs["aff_score"].sum() * 0.0\n\n                routed_aff = routed_ova = None\n                dom_grads = None\n\n                if has_shared:\n                    aff_grads = _replace_none(\n                        torch.autograd.grad(\n                            aff_loss,\n                            shared_params,\n                            retain_graph=True,\n                            allow_unused=True,\n                        ),\n                        shared_params,\n                    )\n                    ova_grads = _replace_none(\n                        torch.autograd.grad(\n                            ova_loss,\n                            shared_params,\n                            retain_graph=True,\n                            allow_unused=True,\n                        ),\n                        shared_params,\n                    )\n\n                    for acc, grad in zip(epoch_aff_sum, aff_grads):\n                        acc.add_(grad.detach())\n                    for acc, grad in zip(epoch_ova_sum, ova_grads):\n                        acc.add_(grad.detach())\n\n                    dot = _dot(aff_grads, ova_grads)\n                    conflict = bool(float(dot.detach().cpu()) < 0.0)\n                    pre = _cosine(aff_grads, ova_grads)\n\n                    if self.spec.pcgrad and conflict:\n                        aff_norm_sq = _norm_sq(aff_grads)\n                        ova_norm_sq = _norm_sq(ova_grads)\n                        aff_coeff = dot / (ova_norm_sq + ROUTING_EPS)\n                        ova_coeff = dot / (aff_norm_sq + ROUTING_EPS)\n                        routed_aff = [\n                            ga - aff_coeff * go\n                            for ga, go in zip(aff_grads, ova_grads)\n                        ]\n                        routed_ova = [\n                            go - ova_coeff * ga\n                            for ga, go in zip(aff_grads, ova_grads)\n                        ]\n                    else:\n                        routed_aff = aff_grads\n                        routed_ova = ova_grads\n\n                    post = _cosine(routed_aff, routed_ova)\n\n                    original_norm = torch.sqrt(\n                        _norm_sq(aff_grads) + _norm_sq(ova_grads)\n                    )\n                    projection_norm = torch.sqrt(\n                        sum(\n                            ((ra - ga) ** 2).sum()\n                            for ra, ga in zip(routed_aff, aff_grads)\n                        )\n                        + sum(\n                            ((ro - go) ** 2).sum()\n                            for ro, go in zip(routed_ova, ova_grads)\n                        )\n                    )\n                    projection_fraction = float(\n                        (projection_norm / (original_norm + ROUTING_EPS))\n                        .detach().cpu()\n                    )\n\n                    if self.spec.pcgrad and dom_active:\n                        dom_grads = _replace_none(\n                            torch.autograd.grad(\n                                dom,\n                                shared_params,\n                                retain_graph=True,\n                                allow_unused=True,\n                            ),\n                            shared_params,\n                        )\n\n                    conflicts.append(float(conflict))\n                    pre_cos.append(float(pre.detach().cpu()))\n                    post_cos.append(float(post.detach().cpu()))\n                    aff_norms.append(_grad_norm(aff_grads))\n                    ova_norms.append(_grad_norm(ova_grads))\n                    proj_fracs.append(projection_fraction)\n\n                total = aff_loss + ova_loss\n                if dom_active:\n                    total = total + DOMINANCE_WEIGHT * dom\n\n                total.backward()\n\n                # For PCGrad arms only, overwrite shared gradients with the\n                # routed task gradients + ordinary dominance gradient.\n                if has_shared and self.spec.pcgrad:\n                    for index, param in enumerate(shared_params):\n                        combined = routed_aff[index] + routed_ova[index]\n                        if dom_active:\n                            combined = (\n                                combined\n                                + DOMINANCE_WEIGHT * dom_grads[index]\n                            )\n                        param.grad = combined.detach().clone()\n\n                torch.nn.utils.clip_grad_norm_(\n                    self.model.parameters(),\n                    max_norm=1.0,\n                )\n                self.optimizer.step()\n\n                totals.append(float(total.detach().cpu()))\n                dom_values.append(float(dom.detach().cpu()))\n\n                if "aff_private" in outputs:\n                    private_aff_ratios.append(\n                        _ratio_norm(outputs["aff_private"], outputs["shared"])\n                    )\n                    private_ova_ratios.append(\n                        _ratio_norm(outputs["ova_private"], outputs["shared"])\n                    )\n                if "shared_capacity_correction" in outputs:\n                    capacity_ratios.append(\n                        _ratio_norm(\n                            outputs["shared_capacity_correction"],\n                            outputs["shared"],\n                        )\n                    )\n\n            epoch_global_cos = (\n                float(_cosine(epoch_aff_sum, epoch_ova_sum).detach().cpu())\n                if has_shared else np.nan\n            )\n\n            gates = {\n                "aff_private_gate": np.nan,\n                "ova_private_gate": np.nan,\n                "capacity_gate_1": np.nan,\n                "capacity_gate_2": np.nan,\n            }\n            if hasattr(self.model, "private_gate_values"):\n                gates.update(self.model.private_gate_values())\n            if hasattr(self.model, "capacity_gate_values"):\n                gates.update(self.model.capacity_gate_values())\n\n            row = {\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": self.spec.name,\n                "arm_label": self.spec.label,\n                "architecture": self.spec.architecture,\n                "dominance_enabled": bool(self.spec.dominance),\n                "pcgrad_enabled": bool(self.spec.pcgrad),\n                "epoch": epoch + 1,\n                "dominance_active": dom_active,\n                "mean_total_loss": float(np.mean(totals)),\n                "mean_dominance_loss": float(np.mean(dom_values)),\n                "batch_conflict_rate": (\n                    float(np.mean(conflicts)) if conflicts else np.nan\n                ),\n                "mean_batch_pre_cosine": (\n                    float(np.mean(pre_cos)) if pre_cos else np.nan\n                ),\n                "mean_batch_post_cosine": (\n                    float(np.mean(post_cos)) if post_cos else np.nan\n                ),\n                "epoch_aggregate_cosine": epoch_global_cos,\n                "mean_aff_shared_grad_norm": (\n                    float(np.mean(aff_norms)) if aff_norms else np.nan\n                ),\n                "mean_ova_shared_grad_norm": (\n                    float(np.mean(ova_norms)) if ova_norms else np.nan\n                ),\n                "mean_projection_fraction": (\n                    float(np.mean(proj_fracs)) if proj_fracs else np.nan\n                ),\n                "mean_aff_private_to_shared_ratio": (\n                    float(np.mean(private_aff_ratios))\n                    if private_aff_ratios else np.nan\n                ),\n                "mean_ova_private_to_shared_ratio": (\n                    float(np.mean(private_ova_ratios))\n                    if private_ova_ratios else np.nan\n                ),\n                "mean_capacity_correction_to_shared_ratio": (\n                    float(np.mean(capacity_ratios))\n                    if capacity_ratios else np.nan\n                ),\n                **gates,\n            }\n            self.rows.append(row)\n\n            if epoch == 0 or (epoch + 1) % 5 == 0:\n                print(\n                    f"[{FEATURE_TYPE}] {self.spec.name} "\n                    f"epoch={epoch+1:02d}/{config.EPOCHS} "\n                    f"loss={row[\'mean_total_loss\']:.4f} "\n                    f"conflict={row[\'batch_conflict_rate\']}",\n                    flush=True,\n                )\n\n        return self\n\n\ndef class_pos_weight(y):\n    y = np.asarray(y, dtype=int)\n    positives = int((y == 1).sum())\n    negatives = int((y == 0).sum())\n    return float(negatives / max(positives, 1))\n\n\ndef fit_arm(X, y_aff, y_ova, arm_name):\n    spec = ARM_SPECS[arm_name]\n\n    # Reset to the SAME seed before each arm. Shared-base initial weights and\n    # minibatch order are therefore matched wherever architectures overlap.\n    hard_reset_rng(SEED, f"{FEATURE_TYPE} {arm_name}")\n\n    model = make_model(X.shape[1], spec)\n    trainer = ControlledTrainer(\n        model,\n        spec,\n        class_pos_weight(y_aff),\n        class_pos_weight(y_ova),\n    )\n    trainer.fit(\n        X,\n        y_aff,\n        y_ova,\n        f"{FEATURE_TYPE} seed={SEED} {arm_name}",\n    )\n    return model, trainer\n\n\ndef sha256_file(path):\n    h = hashlib.sha256()\n    with open(path, "rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef arm_manifest_path(parent, arm):\n    return parent / arm / "manifest.json"\n\n\ndef manifest_matches(path, expected):\n    if not path.exists():\n        return False\n    try:\n        observed = json.loads(path.read_text(encoding="utf-8"))\n    except Exception:\n        return False\n    return all(observed.get(k) == v for k, v in expected.items())\n\n\ndef write_arm_outputs(\n    arm_dir,\n    manifest,\n    raw_df,\n    metrics_df,\n    diagnostics_df,\n    parameter_count,\n):\n    arm_dir.mkdir(parents=True, exist_ok=True)\n    raw_df.to_csv(arm_dir / "raw_predictions.csv.gz", index=False)\n    metrics_df.to_csv(arm_dir / "metrics.csv", index=False)\n    diagnostics_df.to_csv(\n        arm_dir / "training_diagnostics.csv.gz",\n        index=False,\n    )\n    pd.DataFrame([{\n        **manifest,\n        "trainable_parameters": int(parameter_count),\n    }]).to_csv(\n        arm_dir / "parameter_count.csv",\n        index=False,\n    )\n    (arm_dir / "manifest.json").write_text(\n        json.dumps(manifest, indent=2, sort_keys=True),\n        encoding="utf-8",\n    )\n\n\ndef run_smoke():\n    bundle = np.load(EMI_FEATURE_PATH, allow_pickle=False)\n    X = np.asarray(bundle[FEATURE_TYPE], dtype=np.float32)\n    y_aff = np.asarray(bundle["y_aff"], dtype=np.float32)\n    y_ova = np.asarray(bundle["y_ova"], dtype=np.float32)\n\n    selected = []\n    for a in [0, 1]:\n        for o in [0, 1]:\n            idx = np.flatnonzero(\n                (y_aff.astype(int) == a)\n                & (y_ova.astype(int) == o)\n            )\n            selected.extend(idx[:24].tolist())\n    selected = np.asarray(sorted(set(selected)), dtype=int)\n    if len(selected) < 32:\n        raise RuntimeError("Smoke subset is too small.")\n\n    old_epochs = config.EPOCHS\n    config.EPOCHS = 1\n    try:\n        rows = []\n        for arm in CORE_ARMS:\n            model, trainer = fit_arm(\n                X[selected],\n                y_aff[selected],\n                y_ova[selected],\n                arm,\n            )\n            pred_aff, pred_ova = predict_torch_model(\n                model, X[selected[:8]]\n            )\n            if not (\n                np.isfinite(pred_aff).all()\n                and np.isfinite(pred_ova).all()\n            ):\n                raise RuntimeError(f"Nonfinite smoke prediction: {arm}")\n            rows.append({\n                "arm": arm,\n                "parameter_count": trainable_parameter_count(model),\n                "diagnostic_rows": len(trainer.rows),\n            })\n            del model, trainer\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            gc.collect()\n        pd.DataFrame(rows).to_csv(\n            OUT_DIR / "smoke_summary.csv",\n            index=False,\n        )\n        print("COMPONENT ABLATION RUNTIME PREFLIGHT PASS")\n    finally:\n        config.EPOCHS = old_epochs\n\n\ndef run_mutation():\n    if HOLDOUT_PATH is None:\n        raise RuntimeError("Mutation stage requires MOLM_HOLDOUT_PATH.")\n\n    bundle = np.load(EMI_FEATURE_PATH, allow_pickle=False)\n    X = np.asarray(bundle[FEATURE_TYPE], dtype=np.float32)\n    y_aff = np.asarray(bundle["y_aff"], dtype=np.float32)\n    y_ova = np.asarray(bundle["y_ova"], dtype=np.float32)\n    sequences = bundle["sequences"].astype(str)\n    holdouts = json.loads(HOLDOUT_PATH.read_text(encoding="utf-8"))\n\n    for holdout in holdouts:\n        holdout_id = holdout["holdout_id"]\n        train_idx = np.asarray(holdout["train_idx"], dtype=int)\n        test_idx = np.asarray(holdout["test_idx"], dtype=int)\n        holdout_dir = OUT_DIR / holdout_id\n\n        for arm in ARMS:\n            expected = {\n                "lock_sha256": LOCK_HASH,\n                "stage": "mutation",\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": arm,\n                "holdout_id": holdout_id,\n            }\n            manifest_path = arm_manifest_path(holdout_dir, arm)\n            if manifest_matches(manifest_path, expected):\n                print(\n                    f"SKIP VERIFIED seed={SEED} feature={FEATURE_TYPE} "\n                    f"holdout={holdout_id} arm={arm}",\n                    flush=True,\n                )\n                continue\n\n            print(\n                f"TRAIN seed={SEED} feature={FEATURE_TYPE} "\n                f"holdout={holdout_id} arm={arm} "\n                f"train={len(train_idx)} test={len(test_idx)}",\n                flush=True,\n            )\n            model, trainer = fit_arm(\n                X[train_idx],\n                y_aff[train_idx],\n                y_ova[train_idx],\n                arm,\n            )\n            pred_aff, pred_ova = predict_torch_model(\n                model, X[test_idx]\n            )\n\n            raw = pd.DataFrame({\n                "stage": "mutation",\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": arm,\n                "arm_label": ARM_SPECS[arm].label,\n                "holdout_id": holdout_id,\n                "kabat_site": int(holdout["kabat_site"]),\n                "python_index": int(holdout["python_index"]),\n                "heldout_residue": holdout["heldout_residue"],\n                "row_id": test_idx,\n                "sequence_id": sequences[test_idx],\n                "true_aff": y_aff[test_idx],\n                "true_ova": y_ova[test_idx],\n                "pred_aff": pred_aff,\n                "pred_ova": pred_ova,\n            })\n\n            metric_rows = []\n            for task, truth, score in [\n                ("affinity", y_aff[test_idx], pred_aff),\n                ("ova", y_ova[test_idx], pred_ova),\n            ]:\n                metric_rows.append({\n                    "stage": "mutation",\n                    "seed": SEED,\n                    "feature": FEATURE_TYPE,\n                    "arm": arm,\n                    "arm_label": ARM_SPECS[arm].label,\n                    "holdout_id": holdout_id,\n                    "kabat_site": int(holdout["kabat_site"]),\n                    "heldout_residue": holdout["heldout_residue"],\n                    "task": task,\n                    **binary_metrics(truth, score),\n                })\n\n            write_arm_outputs(\n                holdout_dir / arm,\n                expected,\n                raw,\n                pd.DataFrame(metric_rows),\n                pd.DataFrame(trainer.rows),\n                trainable_parameter_count(model),\n            )\n\n            del model, trainer\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            gc.collect()\n\n\ndef run_external():\n    if EXTERNAL_FEATURE_PATH is None:\n        raise RuntimeError("External stage requires MOLM_EXTERNAL_FEATURE_PATH.")\n\n    emi = np.load(EMI_FEATURE_PATH, allow_pickle=False)\n    ext = np.load(EXTERNAL_FEATURE_PATH, allow_pickle=False)\n\n    X_train = np.asarray(emi[FEATURE_TYPE], dtype=np.float32)\n    y_aff_train = np.asarray(emi["y_aff"], dtype=np.float32)\n    y_ova_train = np.asarray(emi["y_ova"], dtype=np.float32)\n\n    dataset_specs = [\n        ("ISO", "iso", np.arange(int(ext["iso_n"]), dtype=int)),\n        (\n            "IgG-primary42",\n            "igg",\n            np.asarray(ext["igg_primary42_indices"], dtype=int),\n        ),\n        (\n            "IgG-all96",\n            "igg",\n            np.arange(int(ext["igg_n"]), dtype=int),\n        ),\n    ]\n\n    for arm in ARMS:\n        arm_dir = OUT_DIR / arm\n        expected = {\n            "lock_sha256": LOCK_HASH,\n            "stage": "external",\n            "seed": SEED,\n            "feature": FEATURE_TYPE,\n            "arm": arm,\n        }\n        manifest_path = arm_manifest_path(OUT_DIR, arm)\n        if manifest_matches(manifest_path, expected):\n            print(\n                f"SKIP VERIFIED external seed={SEED} "\n                f"feature={FEATURE_TYPE} arm={arm}",\n                flush=True,\n            )\n            continue\n\n        model, trainer = fit_arm(\n            X_train,\n            y_aff_train,\n            y_ova_train,\n            arm,\n        )\n\n        raw_frames = []\n        metric_rows = []\n\n        for dataset_name, prefix, indices in dataset_specs:\n            X_eval = np.asarray(\n                ext[f"{prefix}__{FEATURE_TYPE}"][indices],\n                dtype=np.float32,\n            )\n            true_aff = np.asarray(\n                ext[f"{prefix}__y_aff"][indices],\n                dtype=float,\n            )\n            true_ova = np.asarray(\n                ext[f"{prefix}__y_ova"][indices],\n                dtype=float,\n            )\n            seqs = ext[f"{prefix}__sequences"][indices].astype(str)\n            source_rows = np.asarray(indices, dtype=int)\n            sample_ids = (\n                ext["igg__sample_ids"][indices].astype(str)\n                if prefix == "igg"\n                else np.asarray([""] * len(indices), dtype=str)\n            )\n\n            pred_aff, pred_ova = predict_torch_model(\n                model, X_eval\n            )\n\n            raw_frames.append(pd.DataFrame({\n                "stage": "external",\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": arm,\n                "arm_label": ARM_SPECS[arm].label,\n                "dataset": dataset_name,\n                "row_id": np.arange(len(indices), dtype=int),\n                "source_row_id": source_rows,\n                "sequence_id": seqs,\n                "sample_id": sample_ids,\n                "true_aff": true_aff,\n                "true_ova": true_ova,\n                "pred_aff": pred_aff,\n                "pred_ova": pred_ova,\n            }))\n\n            metric_rows.append({\n                "stage": "external",\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "arm": arm,\n                "arm_label": ARM_SPECS[arm].label,\n                "dataset": dataset_name,\n                "aff_spearman": float(\n                    spearmanr(true_aff, pred_aff).statistic\n                ),\n                "ova_spearman": float(\n                    spearmanr(true_ova, pred_ova).statistic\n                ),\n                "n": int(len(indices)),\n            })\n\n        write_arm_outputs(\n            arm_dir,\n            expected,\n            pd.concat(raw_frames, ignore_index=True),\n            pd.DataFrame(metric_rows),\n            pd.DataFrame(trainer.rows),\n            trainable_parameter_count(model),\n        )\n\n        del model, trainer\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n        gc.collect()\n\n\nif __name__ == "__main__":\n    start = time.time()\n    print(\n        f"stage={STAGE} seed={SEED} feature={FEATURE_TYPE} "\n        f"arm_set={ARM_SET} device="\n        f"{torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'cpu\'}",\n        flush=True,\n    )\n\n    if SMOKE or STAGE == "smoke":\n        run_smoke()\n    elif STAGE == "mutation":\n        run_mutation()\n    elif STAGE == "external":\n        run_external()\n    else:\n        raise ValueError(f"Unknown stage={STAGE!r}")\n\n    print(f"DONE in {(time.time() - start)/60:.2f} min", flush=True)\n'
LOSS_WORKER_SOURCE = 'from __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport os\nimport random\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import (\n    accuracy_score,\n    average_precision_score,\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    matthews_corrcoef,\n    roc_auc_score,\n)\n\nREPO = Path(os.environ["MOLM_REPO"]).resolve()\nFEATURE_TYPE = os.environ["MOLM_FEATURE_TYPE"]\nSEED = int(os.environ.get("MOLM_SEED", "42"))\nOUT_DIR = Path(os.environ["MOLM_JOB_OUTPUT"]).resolve()\nEMI_FEATURE_PATH = Path(os.environ["MOLM_EMI_FEATURE_PATH"]).resolve()\nHOLDOUT_PATH = Path(os.environ["MOLM_HOLDOUT_PATH"]).resolve()\nLOCK_HASH = os.environ["MOLM_LOCK_HASH"]\nSMOKE = os.environ.get("MOLM_SMOKE", "0") == "1"\n\nOUT_DIR.mkdir(parents=True, exist_ok=True)\n\nimport sys\nsys.path.insert(0, str(REPO))\nfrom phase0_config import (  # noqa: E402\n    DiagnosticMOLM,\n    config,\n    focal_bce_with_logits,\n    gap_hinge_loss,\n    hard_reset_rng,\n    ranking_loss,\n)\n\n# Exact training protocol inherited from the attached A-H ablation notebook.\nconfig.PARETO_LOSS = False\nconfig.EPOCHS = int(os.environ.get("MOLM_EPOCHS", "25"))\nconfig.BATCH_SIZE = int(os.environ.get("MOLM_BATCH_SIZE", "64"))\nconfig.LEARNING_RATE = float(os.environ.get("MOLM_LEARNING_RATE", "5e-5"))\n\n# Assert the pinned repository loss settings expected by this experiment.\nEXPECTED = {\n    "RANKING_WEIGHT_AFF": 0.3,\n    "RANKING_WEIGHT_SPEC": 0.6,\n    "GAP_WEIGHT_AFF": 0.2,\n    "GAP_WEIGHT_SPEC": 0.6,\n    "RANKING_MARGIN": 0.3,\n    "GAP_MARGIN": 0.2,\n    "FOCAL_GAMMA": 2.0,\n}\nfor key, expected in EXPECTED.items():\n    observed = float(getattr(config, key))\n    if not np.isclose(observed, expected):\n        raise RuntimeError(\n            f"Pinned loss setting changed: {key}={observed}, expected={expected}"\n        )\n\nLOSS_ARMS = {\n    "L0_focal": {\n        "label": "L0 Focal only",\n        "ranking": False,\n        "gap": False,\n    },\n    "LR_focal_ranking": {\n        "label": "LR Focal + Ranking",\n        "ranking": True,\n        "gap": False,\n    },\n    "LG_focal_gap": {\n        "label": "LG Focal + Gap",\n        "ranking": False,\n        "gap": True,\n    },\n    "LRG_full": {\n        "label": "LRG Focal + Ranking + Gap",\n        "ranking": True,\n        "gap": True,\n    },\n}\n\nARM_ORDER = [\n    "L0_focal",\n    "LR_focal_ranking",\n    "LG_focal_gap",\n    "LRG_full",\n]\n\n\ndef class_pos_weight(y):\n    y = np.asarray(y, dtype=int)\n    positives = int((y == 1).sum())\n    negatives = int((y == 0).sum())\n    return float(negatives / max(positives, 1))\n\n\ndef trainable_parameter_count(model):\n    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))\n\n\ndef model_parameter_sha256(model):\n    h = hashlib.sha256()\n    for name, tensor in model.state_dict().items():\n        h.update(name.encode("utf-8"))\n        h.update(np.asarray(tensor.detach().cpu()).tobytes())\n    return h.hexdigest()\n\n\ndef make_shared_model(input_dim):\n    return DiagnosticMOLM(\n        input_dim=input_dim,\n        latent_dim=config.LATENT_DIM,\n        shared_dims=config.SHARED_DIMS,\n        tower_dims=config.TOWER_DIMS,\n        dropout_rate=config.DROPOUT_RATE,\n        grl_lambda=config.GRL_LAMBDA,\n    )\n\n\ndef binary_metrics(y_true, score):\n    y_true = np.asarray(y_true, dtype=int).reshape(-1)\n    score = np.asarray(score, dtype=float).reshape(-1)\n    y_pred = (score >= 0.0).astype(int)\n\n    tn, fp, fn, tp = confusion_matrix(\n        y_true, y_pred, labels=[0, 1]\n    ).ravel()\n\n    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan\n    specificity = tn / (tn + fp) if (tn + fp) else np.nan\n    npv = tn / (tn + fn) if (tn + fn) else np.nan\n\n    if len(np.unique(y_true)) == 2:\n        auroc = roc_auc_score(y_true, score)\n        auprc = average_precision_score(y_true, score)\n    else:\n        auroc = np.nan\n        auprc = np.nan\n\n    return {\n        "n": int(len(y_true)),\n        "n_negative": int((y_true == 0).sum()),\n        "n_positive": int((y_true == 1).sum()),\n        "accuracy": float(accuracy_score(y_true, y_pred)),\n        "balanced_accuracy": float(\n            balanced_accuracy_score(y_true, y_pred)\n        ),\n        "mcc": float(matthews_corrcoef(y_true, y_pred)),\n        "f1": float(f1_score(y_true, y_pred, zero_division=0)),\n        "sensitivity": float(sensitivity),\n        "specificity": float(specificity),\n        "npv": float(npv),\n        "auroc": float(auroc),\n        "auprc": float(auprc),\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tp": int(tp),\n    }\n\n\ndef predict_model(model, X):\n    model.eval()\n    device = next(model.parameters()).device\n    with torch.no_grad():\n        out = model(\n            torch.as_tensor(\n                X,\n                dtype=torch.float32,\n                device=device,\n            ),\n            training=False,\n        )\n    return (\n        out["aff_score"].detach().cpu().numpy().reshape(-1),\n        out["spec_score"].detach().cpu().numpy().reshape(-1),\n    )\n\n\ndef loss_components(\n    outputs,\n    y_aff,\n    y_ova,\n    aff_pos_weight,\n    ova_pos_weight,\n    arm_name,\n):\n    spec = LOSS_ARMS[arm_name]\n\n    aff_focal = focal_bce_with_logits(\n        y_aff,\n        outputs["aff_score"],\n        config.FOCAL_GAMMA,\n        aff_pos_weight,\n    )\n    ova_focal = focal_bce_with_logits(\n        y_ova,\n        outputs["spec_score"],\n        config.FOCAL_GAMMA,\n        ova_pos_weight,\n    )\n\n    aff_rank_raw = ranking_loss(\n        outputs["aff_score"],\n        y_aff,\n        config.RANKING_MARGIN,\n    )\n    ova_rank_raw = ranking_loss(\n        outputs["spec_score"],\n        y_ova,\n        config.RANKING_MARGIN,\n    )\n\n    aff_gap_raw = gap_hinge_loss(\n        outputs["aff_score"],\n        y_aff,\n        config.GAP_MARGIN,\n    )\n    ova_gap_raw = gap_hinge_loss(\n        outputs["spec_score"],\n        y_ova,\n        config.GAP_MARGIN,\n    )\n\n    zero_aff = outputs["aff_score"].sum() * 0.0\n    zero_ova = outputs["spec_score"].sum() * 0.0\n\n    aff_rank_weighted = (\n        config.RANKING_WEIGHT_AFF * aff_rank_raw\n        if spec["ranking"] else zero_aff\n    )\n    ova_rank_weighted = (\n        config.RANKING_WEIGHT_SPEC * ova_rank_raw\n        if spec["ranking"] else zero_ova\n    )\n    aff_gap_weighted = (\n        config.GAP_WEIGHT_AFF * aff_gap_raw\n        if spec["gap"] else zero_aff\n    )\n    ova_gap_weighted = (\n        config.GAP_WEIGHT_SPEC * ova_gap_raw\n        if spec["gap"] else zero_ova\n    )\n\n    aff_total = (\n        aff_focal\n        + aff_rank_weighted\n        + aff_gap_weighted\n    )\n    ova_total = (\n        ova_focal\n        + ova_rank_weighted\n        + ova_gap_weighted\n    )\n\n    return {\n        "aff_total": aff_total,\n        "ova_total": ova_total,\n        "total": aff_total + ova_total,\n        "aff_focal": aff_focal,\n        "ova_focal": ova_focal,\n        "aff_rank_raw": aff_rank_raw,\n        "ova_rank_raw": ova_rank_raw,\n        "aff_gap_raw": aff_gap_raw,\n        "ova_gap_raw": ova_gap_raw,\n        "aff_rank_weighted": aff_rank_weighted,\n        "ova_rank_weighted": ova_rank_weighted,\n        "aff_gap_weighted": aff_gap_weighted,\n        "ova_gap_weighted": ova_gap_weighted,\n    }\n\n\nclass LossAblationTrainer:\n    def __init__(\n        self,\n        model,\n        arm_name,\n        aff_pos_weight,\n        ova_pos_weight,\n        initial_param_sha256,\n    ):\n        self.device = torch.device(\n            "cuda" if torch.cuda.is_available() else "cpu"\n        )\n        self.model = model.to(self.device)\n        self.arm_name = arm_name\n        self.aff_pos_weight = float(aff_pos_weight)\n        self.ova_pos_weight = float(ova_pos_weight)\n        self.initial_param_sha256 = initial_param_sha256\n        self.optimizer = torch.optim.AdamW(\n            self.model.parameters(),\n            lr=config.LEARNING_RATE,\n            weight_decay=1e-4,\n        )\n        self.rows = []\n\n    def fit(self, X, y_aff, y_ova):\n        dataset = torch.utils.data.TensorDataset(\n            torch.as_tensor(X, dtype=torch.float32),\n            torch.as_tensor(y_aff, dtype=torch.float32),\n            torch.as_tensor(y_ova, dtype=torch.float32),\n        )\n\n        # Same generator seed for every loss arm.\n        generator = torch.Generator().manual_seed(SEED)\n        loader = torch.utils.data.DataLoader(\n            dataset,\n            batch_size=config.BATCH_SIZE,\n            shuffle=True,\n            generator=generator,\n        )\n\n        for epoch in range(config.EPOCHS):\n            accum = {\n                "total": [],\n                "aff_total": [],\n                "ova_total": [],\n                "aff_focal": [],\n                "ova_focal": [],\n                "aff_rank_raw": [],\n                "ova_rank_raw": [],\n                "aff_gap_raw": [],\n                "ova_gap_raw": [],\n                "aff_rank_weighted": [],\n                "ova_rank_weighted": [],\n                "aff_gap_weighted": [],\n                "ova_gap_weighted": [],\n            }\n\n            for Xb, yab, yob in loader:\n                Xb = Xb.to(self.device)\n                yab = yab.to(self.device)\n                yob = yob.to(self.device)\n\n                self.model.train()\n                self.optimizer.zero_grad(set_to_none=True)\n\n                outputs = self.model(Xb, training=True)\n                components = loss_components(\n                    outputs,\n                    yab,\n                    yob,\n                    self.aff_pos_weight,\n                    self.ova_pos_weight,\n                    self.arm_name,\n                )\n                components["total"].backward()\n                torch.nn.utils.clip_grad_norm_(\n                    self.model.parameters(),\n                    max_norm=1.0,\n                )\n                self.optimizer.step()\n\n                for key in accum:\n                    accum[key].append(\n                        float(components[key].detach().cpu())\n                    )\n\n            spec = LOSS_ARMS[self.arm_name]\n            row = {\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "loss_arm": self.arm_name,\n                "loss_label": spec["label"],\n                "ranking_enabled": bool(spec["ranking"]),\n                "gap_enabled": bool(spec["gap"]),\n                "epoch": epoch + 1,\n                "initial_param_sha256": self.initial_param_sha256,\n                "ranking_weight_aff": float(config.RANKING_WEIGHT_AFF),\n                "ranking_weight_ova": float(config.RANKING_WEIGHT_SPEC),\n                "gap_weight_aff": float(config.GAP_WEIGHT_AFF),\n                "gap_weight_ova": float(config.GAP_WEIGHT_SPEC),\n                "ranking_margin": float(config.RANKING_MARGIN),\n                "gap_margin": float(config.GAP_MARGIN),\n            }\n            for key, values in accum.items():\n                row[f"mean_{key}"] = float(np.mean(values))\n            self.rows.append(row)\n\n            if epoch == 0 or (epoch + 1) % 5 == 0:\n                print(\n                    f"[{FEATURE_TYPE}] {self.arm_name} "\n                    f"epoch={epoch+1:02d}/{config.EPOCHS} "\n                    f"loss={row[\'mean_total\']:.4f}",\n                    flush=True,\n                )\n\n        return self\n\n\ndef fit_loss_arm(X, y_aff, y_ova, arm_name):\n    # Reset to the SAME seed before each arm:\n    # same model initialization + same minibatch generator seed.\n    hard_reset_rng(SEED, f"{FEATURE_TYPE} {arm_name}")\n\n    model = make_shared_model(X.shape[1])\n    initial_hash = model_parameter_sha256(model)\n\n    trainer = LossAblationTrainer(\n        model=model,\n        arm_name=arm_name,\n        aff_pos_weight=class_pos_weight(y_aff),\n        ova_pos_weight=class_pos_weight(y_ova),\n        initial_param_sha256=initial_hash,\n    )\n    trainer.fit(X, y_aff, y_ova)\n    return model, trainer\n\n\ndef manifest_matches(path, expected):\n    if not path.exists():\n        return False\n    try:\n        observed = json.loads(path.read_text(encoding="utf-8"))\n    except Exception:\n        return False\n    return all(observed.get(k) == v for k, v in expected.items())\n\n\ndef write_outputs(\n    arm_dir,\n    manifest,\n    raw_df,\n    metrics_df,\n    diagnostics_df,\n    model,\n    initial_hash,\n):\n    arm_dir.mkdir(parents=True, exist_ok=True)\n\n    raw_df.to_csv(\n        arm_dir / "raw_predictions.csv.gz",\n        index=False,\n    )\n    metrics_df.to_csv(\n        arm_dir / "metrics.csv",\n        index=False,\n    )\n    diagnostics_df.to_csv(\n        arm_dir / "training_diagnostics.csv.gz",\n        index=False,\n    )\n\n    pd.DataFrame([{\n        **manifest,\n        "trainable_parameters": trainable_parameter_count(model),\n        "initial_param_sha256": initial_hash,\n    }]).to_csv(\n        arm_dir / "model_audit.csv",\n        index=False,\n    )\n\n    (arm_dir / "manifest.json").write_text(\n        json.dumps(manifest, indent=2, sort_keys=True),\n        encoding="utf-8",\n    )\n\n\ndef run_smoke():\n    bundle = np.load(\n        EMI_FEATURE_PATH,\n        allow_pickle=False,\n    )\n    X = np.asarray(\n        bundle[FEATURE_TYPE],\n        dtype=np.float32,\n    )\n    y_aff = np.asarray(\n        bundle["y_aff"],\n        dtype=np.float32,\n    )\n    y_ova = np.asarray(\n        bundle["y_ova"],\n        dtype=np.float32,\n    )\n\n    selected = []\n    for a in [0, 1]:\n        for o in [0, 1]:\n            idx = np.flatnonzero(\n                (y_aff.astype(int) == a)\n                & (y_ova.astype(int) == o)\n            )\n            selected.extend(idx[:24].tolist())\n    selected = np.asarray(sorted(set(selected)), dtype=int)\n\n    old_epochs = config.EPOCHS\n    config.EPOCHS = 1\n    try:\n        rows = []\n        hashes = []\n        params = []\n        for arm in ARM_ORDER:\n            model, trainer = fit_loss_arm(\n                X[selected],\n                y_aff[selected],\n                y_ova[selected],\n                arm,\n            )\n            pred_aff, pred_ova = predict_model(\n                model,\n                X[selected[:8]],\n            )\n\n            if not (\n                np.isfinite(pred_aff).all()\n                and np.isfinite(pred_ova).all()\n            ):\n                raise RuntimeError(\n                    f"Nonfinite smoke prediction for {arm}"\n                )\n\n            diag = trainer.rows[0]\n            spec = LOSS_ARMS[arm]\n\n            if not spec["ranking"]:\n                if (\n                    abs(diag["mean_aff_rank_weighted"]) > 1e-12\n                    or abs(diag["mean_ova_rank_weighted"]) > 1e-12\n                ):\n                    raise RuntimeError(\n                        f"Disabled ranking term nonzero in {arm}"\n                    )\n\n            if not spec["gap"]:\n                if (\n                    abs(diag["mean_aff_gap_weighted"]) > 1e-12\n                    or abs(diag["mean_ova_gap_weighted"]) > 1e-12\n                ):\n                    raise RuntimeError(\n                        f"Disabled gap term nonzero in {arm}"\n                    )\n\n            h = trainer.initial_param_sha256\n            pc = trainable_parameter_count(model)\n            hashes.append(h)\n            params.append(pc)\n\n            rows.append({\n                "loss_arm": arm,\n                "loss_label": LOSS_ARMS[arm]["label"],\n                "initial_param_sha256": h,\n                "trainable_parameters": pc,\n                "ranking_enabled": spec["ranking"],\n                "gap_enabled": spec["gap"],\n                "diagnostic_rows": len(trainer.rows),\n            })\n\n            del model, trainer\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            gc.collect()\n\n        # Fairness invariant: exact same architecture, parameters, initialization.\n        if len(set(hashes)) != 1:\n            raise RuntimeError(\n                "Loss arms did not start from identical model parameters."\n            )\n        if len(set(params)) != 1:\n            raise RuntimeError(\n                "Loss arms do not have identical parameter counts."\n            )\n\n        pd.DataFrame(rows).to_csv(\n            OUT_DIR / "smoke_summary.csv",\n            index=False,\n        )\n        print("RANKING/GAP LOSS ABLATION PREFLIGHT PASS")\n    finally:\n        config.EPOCHS = old_epochs\n\n\ndef run_mutation():\n    bundle = np.load(\n        EMI_FEATURE_PATH,\n        allow_pickle=False,\n    )\n    X = np.asarray(\n        bundle[FEATURE_TYPE],\n        dtype=np.float32,\n    )\n    y_aff = np.asarray(\n        bundle["y_aff"],\n        dtype=np.float32,\n    )\n    y_ova = np.asarray(\n        bundle["y_ova"],\n        dtype=np.float32,\n    )\n    sequences = bundle["sequences"].astype(str)\n\n    holdouts = json.loads(\n        HOLDOUT_PATH.read_text(encoding="utf-8")\n    )\n\n    for holdout in holdouts:\n        hid = holdout["holdout_id"]\n        train_idx = np.asarray(\n            holdout["train_idx"],\n            dtype=int,\n        )\n        test_idx = np.asarray(\n            holdout["test_idx"],\n            dtype=int,\n        )\n\n        # Strict held-out-residue audit.\n        site = int(holdout["python_index"])\n        residue = str(holdout["heldout_residue"])\n        observed = np.asarray(\n            [seq[site] for seq in sequences],\n            dtype=str,\n        )\n        if np.any(observed[train_idx] == residue):\n            raise RuntimeError(\n                f"{hid}: held-out residue leaks into training."\n            )\n        if not np.all(observed[test_idx] == residue):\n            raise RuntimeError(\n                f"{hid}: test set does not uniformly carry held-out residue."\n            )\n\n        holdout_dir = OUT_DIR / hid\n\n        initial_hashes = {}\n        parameter_counts = {}\n\n        for arm in ARM_ORDER:\n            spec = LOSS_ARMS[arm]\n            expected = {\n                "lock_sha256": LOCK_HASH,\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "loss_arm": arm,\n                "holdout_id": hid,\n            }\n            arm_dir = holdout_dir / arm\n            manifest_path = arm_dir / "manifest.json"\n\n            if manifest_matches(manifest_path, expected):\n                print(\n                    f"SKIP VERIFIED seed={SEED} "\n                    f"feature={FEATURE_TYPE} holdout={hid} arm={arm}",\n                    flush=True,\n                )\n                audit = pd.read_csv(arm_dir / "model_audit.csv").iloc[0]\n                initial_hashes[arm] = str(audit["initial_param_sha256"])\n                parameter_counts[arm] = int(audit["trainable_parameters"])\n                continue\n\n            print(\n                f"TRAIN seed={SEED} feature={FEATURE_TYPE} "\n                f"holdout={hid} arm={arm} "\n                f"train={len(train_idx)} test={len(test_idx)}",\n                flush=True,\n            )\n\n            model, trainer = fit_loss_arm(\n                X[train_idx],\n                y_aff[train_idx],\n                y_ova[train_idx],\n                arm,\n            )\n\n            pred_aff, pred_ova = predict_model(\n                model,\n                X[test_idx],\n            )\n\n            raw = pd.DataFrame({\n                "seed": SEED,\n                "feature": FEATURE_TYPE,\n                "loss_arm": arm,\n                "loss_label": spec["label"],\n                "holdout_id": hid,\n                "kabat_site": int(holdout["kabat_site"]),\n                "python_index": int(holdout["python_index"]),\n                "heldout_residue": residue,\n                "global_row_id": test_idx,\n                "sequence_id": sequences[test_idx],\n                "true_aff": y_aff[test_idx],\n                "true_ova": y_ova[test_idx],\n                "pred_aff": pred_aff,\n                "pred_ova": pred_ova,\n            })\n\n            metric_rows = []\n            for task, truth, score in [\n                ("affinity", y_aff[test_idx], pred_aff),\n                ("ova", y_ova[test_idx], pred_ova),\n            ]:\n                metric_rows.append({\n                    "seed": SEED,\n                    "feature": FEATURE_TYPE,\n                    "loss_arm": arm,\n                    "loss_label": spec["label"],\n                    "holdout_id": hid,\n                    "kabat_site": int(holdout["kabat_site"]),\n                    "heldout_residue": residue,\n                    "task": task,\n                    **binary_metrics(truth, score),\n                })\n\n            initial_hash = trainer.initial_param_sha256\n            initial_hashes[arm] = initial_hash\n            parameter_counts[arm] = trainable_parameter_count(model)\n\n            write_outputs(\n                arm_dir=arm_dir,\n                manifest=expected,\n                raw_df=raw,\n                metrics_df=pd.DataFrame(metric_rows),\n                diagnostics_df=pd.DataFrame(trainer.rows),\n                model=model,\n                initial_hash=initial_hash,\n            )\n\n            del model, trainer\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n            gc.collect()\n\n        # Per-holdout fairness invariant across all four loss conditions.\n        if len(set(initial_hashes.values())) != 1:\n            raise RuntimeError(\n                f"{hid}: loss arms did not share identical initialization."\n            )\n        if len(set(parameter_counts.values())) != 1:\n            raise RuntimeError(\n                f"{hid}: loss arms differ in parameter count."\n            )\n\n\nif __name__ == "__main__":\n    start = time.time()\n    print(\n        f"seed={SEED} feature={FEATURE_TYPE} "\n        f"device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'cpu\'}",\n        flush=True,\n    )\n\n    if SMOKE:\n        run_smoke()\n    else:\n        run_mutation()\n\n    print(\n        f"DONE in {(time.time() - start)/60:.2f} min",\n        flush=True,\n    )\n'

# Prefer helper source shipped inside the definitive result bundle.
for name, fallback in [
    ('molm_definitive_esm_cache.py', ESM_HELPER_FALLBACK),
    ('molm_definitive_common.py', COMMON_HELPER_FALLBACK),
]:
    source_path=base_file(name, required=False)
    source=source_path.read_text(encoding='utf-8') if source_path else fallback
    compile(source, name, 'exec')
    (CODE_DIR/name).write_text(source,encoding='utf-8')
    print(name, hashlib.sha256(source.encode()).hexdigest())

(CODE_DIR/'molm_routed_component_ablation_worker_v2.py').write_text(COMPONENT_WORKER_SOURCE,encoding='utf-8')
(CODE_DIR/'molm_ranking_gap_loss_ablation_worker.py').write_text(LOSS_WORKER_SOURCE,encoding='utf-8')
compile(COMPONENT_WORKER_SOURCE,'component_worker','exec')
compile(LOSS_WORKER_SOURCE,'loss_worker','exec')
print('Embedded ablation workers written.')

## 3. Rebuild the exact five representations from reusable caches

No ESM embeddings are recomputed when a compatible cache is found. The helper validates sequence order, ESM model/layer, site mapping, and cache hashes before reuse.

In [ ]:
sys.path.insert(0,str(CODE_DIR))
sys.path.insert(0,str(REPO))
from molm_definitive_esm_cache import (
    SITE_SPEC, build_five_representations, reuse_or_build_cache, sequence_hash,
    mapping_signature, sha256_file as esm_sha256_file,
)
import phase0_config as repo_config

# Cross-check exact biological mapping against the pinned repository.
# phase0_config stores scientific constants on its singleton `config = Config()`.
# Keep a defensive fallback for compatibility with any snapshot exposing them
# directly at module level.
repo_cfg = getattr(repo_config, 'config', repo_config)
for _attr in ['MUTATION_SITES', 'MUTATION_SITES_KABAT', 'WILDTYPE_RESIDUES']:
    if not hasattr(repo_cfg, _attr):
        raise AttributeError(
            f"Pinned phase0_config does not expose {_attr} on config or module."
        )

assert [int(x['python_index']) for x in SITE_SPEC] == list(repo_cfg.MUTATION_SITES)
assert [int(x['kabat_site']) for x in SITE_SPEC] == list(repo_cfg.MUTATION_SITES_KABAT)
assert [str(x['expected_wt']) for x in SITE_SPEC] == list(repo_cfg.WILDTYPE_RESIDUES)
print('Repository/site mapping audit PASS')
print('  aligned indices:', list(repo_cfg.MUTATION_SITES))
print('  Kabat sites    :', list(repo_cfg.MUTATION_SITES_KABAT))
print('  WT residues    :', list(repo_cfg.WILDTYPE_RESIDUES))
print('Site mapping:', [(x['kabat_site'],x['python_index'],x['expected_wt']) for x in SITE_SPEC])

def resolve_column(frame,candidates):
    for c in candidates:
        if c in frame.columns:return c
    raise KeyError((candidates,list(frame.columns)))

EMI_CSV=REPO/'data'/'emi_binding.csv'
ISO_CSV=REPO/'data'/'iso_binding.csv'
IGG_CSV=REPO/'data'/'igg_binding.csv'
emi_binding=pd.read_csv(EMI_CSV,index_col=0)
iso_binding=pd.read_csv(ISO_CSV,index_col=0)
igg_binding=pd.read_csv(IGG_CSV,index_col=0)
emi_sequences=emi_binding.index.astype(str).to_numpy()
iso_sequences=iso_binding.index.astype(str).to_numpy()
igg_sequences=igg_binding.index.astype(str).to_numpy()

emi_aff_col=resolve_column(emi_binding,['ANT Binding','ANT','Affinity','affinity'])
emi_ova_col=resolve_column(emi_binding,['OVA Binding','OVA','PSY','Specificity','specificity'])
iso_aff_col=resolve_column(iso_binding,['ANT Binding','ANT','Affinity','affinity'])
iso_ova_col=resolve_column(iso_binding,['OVA Binding','OVA','PSY','Specificity','specificity'])
igg_aff_col=resolve_column(igg_binding,['ANT Binding','ANT','Affinity','affinity'])
igg_ova_col=resolve_column(igg_binding,['OVA Binding','OVA','PSY','Specificity','specificity'])
y_aff_emi=(emi_binding[emi_aff_col].to_numpy()>0).astype(np.float32)
y_ova_emi=(emi_binding[emi_ova_col].to_numpy()>0).astype(np.float32)
iso_aff=iso_binding[iso_aff_col].to_numpy(float); iso_ova=iso_binding[iso_ova_col].to_numpy(float)
igg_aff=igg_binding[igg_aff_col].to_numpy(float); igg_ova=igg_binding[igg_ova_col].to_numpy(float)

# ESM cache paths expected by the definitive helper; it searches direct files and ZIPs recursively.
cache_specs={
 'EMI':(emi_sequences,'emi_locked_esm2_cache.npz','emi_locked_esm2_cache.meta.json'),
 'ISO':(iso_sequences,'iso_locked_esm2_cache.npz','iso_locked_esm2_cache.meta.json'),
 'IgG':(igg_sequences,'igg_locked_esm2_cache.npz','igg_locked_esm2_cache.meta.json'),
}
cache_paths={}
for name,(seqs,fn,mn) in cache_specs.items():
    cp=CACHE_DIR/fn; mp=CACHE_DIR/mn; audit=CACHE_DIR/f'{fn}.site_mapping_audit.csv'
    meta=reuse_or_build_cache(name,seqs,cp,mp,audit,search_root=INPUT_ROOT,batch_size=64,device='cuda:0')
    if meta['mapping_signature_sha256'] != locked_config['esm']['mapping_signature_sha256']:
        raise RuntimeError(f'{name}: ESM mapping differs from base lock.')
    cache_paths[name]=cp
    print(name, cp, meta['cache_sha256'])

emi_reps=build_five_representations(emi_sequences,cache_paths['EMI'])
iso_reps=build_five_representations(iso_sequences,cache_paths['ISO'])
igg_reps=build_five_representations(igg_sequences,cache_paths['IgG'])
for f,d in FEATURE_DIMS.items():
    assert emi_reps[f].shape==(len(emi_sequences),d)
    assert iso_reps[f].shape==(len(iso_sequences),d)
    assert igg_reps[f].shape==(len(igg_sequences),d)
print('Five representations validated.')


In [ ]:
# Reconstruct exact holdouts from the locked metadata and enrich with explicit row indices
# for the component/loss-ablation workers.
base_holdout_path=base_file('mutation_holdouts_locked.json')
base_holdouts=json.loads(base_holdout_path.read_text(encoding='utf-8'))
enriched=[]
for h in base_holdouts:
    pos=int(h['python_index']); residue=str(h['heldout_residue'])
    test_idx=np.flatnonzero(np.asarray([s[pos]==residue for s in emi_sequences],bool))
    train_idx=np.flatnonzero(np.asarray([s[pos]!=residue for s in emi_sequences],bool))
    if int(h['n_eval']) != len(test_idx) or int(h['n_train']) != len(train_idx):
        raise RuntimeError(f"Holdout count mismatch: {h['holdout_id']}")
    if any(emi_sequences[i][pos]==residue for i in train_idx): raise RuntimeError('leakage')
    hh=dict(h); hh['train_idx']=train_idx.tolist(); hh['test_idx']=test_idx.tolist(); enriched.append(hh)
ENRICHED_HOLDOUT_PATH=FEATURE_DIR/'mutation_holdouts_with_indices.json'
ENRICHED_HOLDOUT_PATH.write_text(json.dumps(enriched,indent=2),encoding='utf-8')
print('Holdouts:',len(enriched),'saved',ENRICHED_HOLDOUT_PATH)

audit=pd.DataFrame([{k:v for k,v in h.items() if k not in ('train_idx','test_idx')} for h in enriched])
display(audit.sort_values(['holdout_type','kabat_site']))

# Explicit locked IgG-primary42 selection.
primary_ids=list(locked_config['igg_primary42']['sample_ids'])
if 'Sample' not in igg_binding.columns: raise RuntimeError('IgG source lacks Sample column.')
igg_sample_ids=igg_binding['Sample'].astype(str).to_numpy()
counts=pd.Series(igg_sample_ids).value_counts()
bad=[x for x in primary_ids if int(counts.get(x,0))!=1]
if bad: raise RuntimeError(f'IgG primary IDs missing/duplicated: {bad}')
igg_primary42_indices=np.asarray([int(np.flatnonzero(igg_sample_ids==x)[0]) for x in primary_ids],dtype=np.int64)
assert len(igg_primary42_indices)==42 and len(np.unique(igg_primary42_indices))==42

EMI_FEATURE_PATH=FEATURE_DIR/'emi_five_representations.npz'
np.savez_compressed(EMI_FEATURE_PATH,sequences=emi_sequences.astype('U'),y_aff=y_aff_emi,y_ova=y_ova_emi,**emi_reps)
EXTERNAL_FEATURE_PATH=FEATURE_DIR/'external_five_representations.npz'
payload={
 'iso_n':np.asarray(len(iso_sequences),dtype=np.int64),'igg_n':np.asarray(len(igg_sequences),dtype=np.int64),
 'igg_primary42_indices':igg_primary42_indices,'igg_primary42_sample_ids':np.asarray(primary_ids).astype('U'),
 'iso__sequences':iso_sequences.astype('U'),'igg__sequences':igg_sequences.astype('U'),'igg__sample_ids':igg_sample_ids.astype('U'),
 'iso__y_aff':iso_aff,'iso__y_ova':iso_ova,'igg__y_aff':igg_aff,'igg__y_ova':igg_ova,
}
for f in FEATURE_TYPES:
    payload[f'iso__{f}']=iso_reps[f]; payload[f'igg__{f}']=igg_reps[f]
np.savez_compressed(EXTERNAL_FEATURE_PATH,**payload)
print('Feature archives ready.')

## 4. Analysis configuration lock

This lock records the Standard-MOLM, ablation, Hamming, latent-representation, fixed-budget, and statistical settings used by the extended analyses. It does not imply that ISO/IgG are newly blind benchmarks.


In [ ]:
from datetime import datetime, timezone
ANALYSIS_LOCK_PATH=WORK_ROOT/'UNIFIED_ANALYSIS_LOCK.json'
EXTENSION_LOCK={
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'base_lock_sha256':BASE_LOCK_SHA,
 'repository_commit':PINNED_COMMIT,
 'interpretation':'retrospective extension; ISO/IgG were previously observed overall',
 'standard_molm':{
   'shared_dims':[256,128],'tower_dims':[64,32],'latent_dim':16,'dropout':0.2,
   'loss':'focal + task-specific ranking + gap-hinge','dominance':False,'pcgrad':False,'private_adapters':False,
 },
 'seeds':SEEDS,'features':FEATURE_DIMS,'mutation_holdouts_sha256':sha256_file(ENRICHED_HOLDOUT_PATH),
 'pareto_budgets':K_VALUES,'bootstrap_draws':BOOTSTRAP_DRAWS,
 'component_ablation':{'features':COMPONENT_FEATURES,'arm_set':COMPONENT_ARM_SET},
 'loss_ablation':{'features':LOSS_ABLATION_FEATURES,'arms':['focal','focal+ranking','focal+gap','focal+ranking+gap']},
 'hamming':{'sites':[int(s['python_index']) for s in SITE_SPEC],'bins':['d=1','d=2','d>=3'],'site_block_bootstrap_draws':BOOTSTRAP_DRAWS},
 'latent_robustness':{'spaces':['logits','pca64','pca32','pca16'],'pca_fit':'EMI training only','orientation':'positive correlation with EMI train logit'},
}
ANALYSIS_LOCK_PATH.write_text(json.dumps(EXTENSION_LOCK,indent=2,sort_keys=True)+'\n',encoding='utf-8')
ANALYSIS_LOCK_SHA=sha256_file(ANALYSIS_LOCK_PATH)
(WORK_ROOT/'UNIFIED_ANALYSIS_LOCK.sha256').write_text(ANALYSIS_LOCK_SHA+'\n')
print('Analysis lock:',ANALYSIS_LOCK_SHA)

## 5. Standard MOLM worker

The worker is intentionally separate from Routed-MOLM. It uses the same repository loss definitions but **no routing, no dominance, and no private adapters**. For external evaluation it additionally exports train-fit PCA scores from the 64D, 32D, and 16D task representations.

In [ ]:
STANDARD_WORKER_SOURCE = "from __future__ import annotations\nimport os, json, gc, random, time, hashlib\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn as nn\nfrom sklearn.decomposition import PCA\nfrom scipy.stats import spearmanr\n\nREPO=Path(os.environ['MOLM_REPO']).resolve(); CODE=Path(os.environ['MOLM_CODE_DIR']).resolve()\nimport sys\nsys.path.insert(0,str(CODE)); sys.path.insert(0,str(REPO))\nfrom molm_definitive_common import config, hard_reset_rng, task_losses, binary_metrics\n\nFEATURE=os.environ['MOLM_FEATURE_TYPE']; SEED=int(os.environ['MOLM_SEED']); STAGE=os.environ['MOLM_STAGE']\nEMI=Path(os.environ['MOLM_EMI_FEATURE_PATH']); HOLDOUT=Path(os.environ['MOLM_HOLDOUT_PATH']); EXT=Path(os.environ.get('MOLM_EXTERNAL_FEATURE_PATH',''))\nOUT=Path(os.environ['MOLM_OUTPUT_DIR']); LOCK_HASH=os.environ['MOLM_LOCK_HASH']; OUT.mkdir(parents=True,exist_ok=True)\nconfig.PARETO_LOSS=False; config.EPOCHS=int(os.environ.get('MOLM_EPOCHS','25')); config.BATCH_SIZE=int(os.environ.get('MOLM_BATCH_SIZE','64')); config.LEARNING_RATE=float(os.environ.get('MOLM_LEARNING_RATE','5e-5'))\n\nclass Block(nn.Module):\n def __init__(self,inp,out,drop): super().__init__(); self.lin=nn.Linear(inp,out); self.norm=nn.LayerNorm(out); self.drop=nn.Dropout(drop)\n def forward(self,x): return self.drop(torch.nn.functional.gelu(self.norm(self.lin(x))))\nclass StandardMOLM(nn.Module):\n def __init__(self,input_dim):\n  super().__init__(); sd=list(config.SHARED_DIMS); td=list(config.TOWER_DIMS); assert sd==[256,128] and td==[64,32]\n  self.s1=Block(input_dim,sd[0],config.DROPOUT_RATE); self.s2=Block(sd[0],sd[1],config.DROPOUT_RATE)\n  self.a64=Block(sd[1],td[0],config.DROPOUT_RATE); self.a32=Block(td[0],td[1],config.DROPOUT_RATE)\n  self.o64=Block(sd[1],td[0],config.DROPOUT_RATE); self.o32=Block(td[0],td[1],config.DROPOUT_RATE)\n  self.ap=nn.Linear(td[1],config.LATENT_DIM); self.an=nn.LayerNorm(config.LATENT_DIM); self.op=nn.Linear(td[1],config.LATENT_DIM); self.on=nn.LayerNorm(config.LATENT_DIM)\n  self.ah=nn.Linear(config.LATENT_DIM,1); self.oh=nn.Linear(config.LATENT_DIM,1)\n def forward(self,x,training=None):\n  h=self.s2(self.s1(x)); a64=self.a64(h); a32=self.a32(a64); o64=self.o64(h); o32=self.o32(o64)\n  a16=self.an(self.ap(a32)); o16=self.on(self.op(o32)); al=self.ah(a16).squeeze(-1); ol=self.oh(o16).squeeze(-1)\n  return {'aff_score':al,'spec_score':ol,'aff64':a64,'aff32':a32,'aff16':a16,'ova64':o64,'ova32':o32,'ova16':o16}\n\ndef pw(y):\n y=np.asarray(y); return float((y==0).sum()/max(int((y==1).sum()),1))\ndef fit_model(X,ya,yo):\n hard_reset_rng(SEED,f'Standard-MOLM {FEATURE} {SEED}'); random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)\n if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)\n m=StandardMOLM(X.shape[1]); dev=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); m.to(dev)\n opt=torch.optim.AdamW(m.parameters(),lr=config.LEARNING_RATE,weight_decay=1e-4); ds=torch.utils.data.TensorDataset(torch.tensor(X,dtype=torch.float32),torch.tensor(ya,dtype=torch.float32),torch.tensor(yo,dtype=torch.float32)); g=torch.Generator().manual_seed(SEED); dl=torch.utils.data.DataLoader(ds,batch_size=config.BATCH_SIZE,shuffle=True,generator=g)\n rows=[]; wa=pw(ya); wo=pw(yo)\n for ep in range(config.EPOCHS):\n  vals=[]; m.train()\n  for xb,yab,yob in dl:\n   xb,yab,yob=xb.to(dev),yab.to(dev),yob.to(dev); opt.zero_grad(set_to_none=True); out=m(xb); la,lo=task_losses(out,yab,yob,wa,wo); loss=la+lo; loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); vals.append(float(loss.detach().cpu()))\n  rows.append({'seed':SEED,'feature':FEATURE,'model':'Standard-MOLM','epoch':ep+1,'mean_total_loss':float(np.mean(vals))})\n return m,pd.DataFrame(rows)\ndef predict(m,X,full=False):\n dev=next(m.parameters()).device; m.eval(); acc=[]\n with torch.no_grad():\n  for i in range(0,len(X),1024):\n   o=m(torch.tensor(X[i:i+1024],dtype=torch.float32,device=dev)); acc.append({k:v.detach().cpu().numpy() for k,v in o.items()})\n out={k:np.concatenate([a[k] for a in acc],axis=0) for k in acc[0]}\n return out if full else (out['aff_score'].reshape(-1),out['spec_score'].reshape(-1))\ndef orient_pca(Z,logit):\n p=PCA(n_components=1,random_state=SEED).fit(Z); t=p.transform(Z).reshape(-1)\n c=np.corrcoef(t,np.asarray(logit).reshape(-1))[0,1]\n if np.isfinite(c) and c<0: p.components_ *= -1\n return p\ndef pareto_mask(points):\n points=np.asarray(points,float); mask=np.ones(len(points),bool)\n for i in range(len(points)):\n  if (np.all(points>=points[i],axis=1)&np.any(points>points[i],axis=1)).any(): mask[i]=False\n return mask\n\nemi=np.load(EMI,allow_pickle=False); Xall=np.asarray(emi[FEATURE],np.float32); yaall=np.asarray(emi['y_aff'],np.float32); yoall=np.asarray(emi['y_ova'],np.float32); seqall=emi['sequences'].astype(str)\nif STAGE=='mutation':\n hs=json.loads(HOLDOUT.read_text()); rawall=[]; metall=[]; histall=[]; pars=[]\n for h in hs:\n  tr=np.asarray(h['train_idx'],int); te=np.asarray(h['test_idx'],int); arm=OUT/h['holdout_id']; arm.mkdir(parents=True,exist_ok=True); done=arm/'DONE.json'\n  expected={'stage':'mutation','seed':SEED,'feature':FEATURE,'holdout_id':h['holdout_id'],'lock_sha256':LOCK_HASH}\n  if done.exists():\n   try:\n    old=json.loads(done.read_text())\n    if all(old.get(k)==v for k,v in expected.items()):\n     rawall.append(pd.read_csv(arm/'raw_predictions.csv.gz')); metall.append(pd.read_csv(arm/'metrics.csv')); histall.append(pd.read_csv(arm/'training_diagnostics.csv.gz')); pars.append(pd.read_csv(arm/'parameter_counts.csv')); continue\n   except Exception: pass\n  m,hist=fit_model(Xall[tr],yaall[tr],yoall[tr]); pa,po=predict(m,Xall[te]); rr=[]; mm=[]\n  for task,truth,score in [('affinity',yaall[te],pa),('ova',yoall[te],po)]:\n   mm.append({'seed':SEED,'seed_protocol':'optimization','feature':FEATURE,'model':'Standard-MOLM','task':task,**{k:v for k,v in h.items() if k not in ('train_idx','test_idx')},**binary_metrics(truth,score)})\n   pred=(np.asarray(score)>=0).astype(int)\n   for j,gid in enumerate(te): rr.append({'seed':SEED,'seed_protocol':'optimization','feature':FEATURE,'model':'Standard-MOLM','task':task,**{k:v for k,v in h.items() if k not in ('train_idx','test_idx')},'global_row_id':int(gid),'sequence_id':str(seqall[gid]),'y_true':int(truth[j]),'score':float(score[j]),'y_pred':int(pred[j])})\n  raw=pd.DataFrame(rr); met=pd.DataFrame(mm); par=pd.DataFrame([{'seed':SEED,'feature':FEATURE,'holdout_id':h['holdout_id'],'model':'Standard-MOLM','trainable_parameters':sum(p.numel() for p in m.parameters() if p.requires_grad)}]); raw.to_csv(arm/'raw_predictions.csv.gz',index=False,compression='gzip'); met.to_csv(arm/'metrics.csv',index=False); hist.to_csv(arm/'training_diagnostics.csv.gz',index=False,compression='gzip'); par.to_csv(arm/'parameter_counts.csv',index=False); done.write_text(json.dumps(expected,indent=2)); rawall.append(raw); metall.append(met); histall.append(hist); pars.append(par); del m; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None\n pd.concat(rawall,ignore_index=True).to_csv(OUT/'raw_predictions.csv.gz',index=False,compression='gzip'); pd.concat(metall,ignore_index=True).to_csv(OUT/'metrics.csv',index=False); pd.concat(histall,ignore_index=True).to_csv(OUT/'training_diagnostics.csv.gz',index=False,compression='gzip'); pd.concat(pars,ignore_index=True).to_csv(OUT/'parameter_counts.csv',index=False); (OUT/'manifest.json').write_text(json.dumps({'status':'ok','stage':'mutation','seed':SEED,'feature':FEATURE,'lock_sha256':LOCK_HASH},indent=2))\nelif STAGE=='external':\n ext=np.load(EXT,allow_pickle=False); m,hist=fit_model(Xall,yaall,yoall); train=predict(m,Xall,True)\n pcas={}\n for n in [64,32,16]:\n  pcas[('aff',n)]=orient_pca(train[f'aff{n}'],train['aff_score']); pcas[('ova',n)]=orient_pca(train[f'ova{n}'],train['spec_score'])\n idx42=np.asarray(ext['igg_primary42_indices'],int); ids=ext['igg__sample_ids'].astype(str); raw=[]; geo=[]\n for d,prefix,idx in [('ISO','iso',np.arange(int(ext['iso_n']))),('IgG-primary42','igg',idx42),('IgG-all96','igg',np.arange(int(ext['igg_n'])))]:\n  Xe=np.asarray(ext[f'{prefix}__{FEATURE}'][idx],np.float32); y1=np.asarray(ext[f'{prefix}__y_aff'][idx],float); y2=np.asarray(ext[f'{prefix}__y_ova'][idx],float); seq=ext[f'{prefix}__sequences'][idx].astype(str); sid=ids[idx] if prefix=='igg' else np.asarray(['']*len(idx)); o=predict(m,Xe,True); pa=o['aff_score'].reshape(-1); po=o['spec_score'].reshape(-1); true=pareto_mask(np.c_[y1,-y2]); pred=pareto_mask(np.c_[pa,-po])\n  pvals={}\n  for n in [64,32,16]: pvals[f'pca{n}_aff']=pcas[('aff',n)].transform(o[f'aff{n}']).reshape(-1); pvals[f'pca{n}_ova']=pcas[('ova',n)].transform(o[f'ova{n}']).reshape(-1)\n  geo.append({'seed':SEED,'seed_protocol':'optimization','feature':FEATURE,'dataset':d,'model':'Standard-MOLM','aff_spearman':float(spearmanr(pa,y1).statistic),'ova_spearman':float(spearmanr(po,y2).statistic),'predicted_cross_task_spearman':float(spearmanr(pa,po).statistic),'true_cross_task_spearman':float(spearmanr(y1,y2).statistic),'natural_front_n':int(pred.sum()),'true_front_n':int(true.sum()),'natural_front_hits':int((pred&true).sum()),'natural_front_precision':float((pred&true).sum()/max(int(pred.sum()),1)),'natural_front_recall':float((pred&true).sum()/max(int(true.sum()),1))})\n  for j in range(len(idx)):\n   row={'seed':SEED,'seed_protocol':'optimization','feature':FEATURE,'dataset':d,'model':'Standard-MOLM','row_id':j,'source_row_id':int(idx[j]),'sequence_id':str(seq[j]),'sample_id':str(sid[j]),'true_aff':float(y1[j]),'true_ova':float(y2[j]),'pred_aff':float(pa[j]),'pred_ova':float(po[j]),'pred_aff_desirability':float(pa[j]),'pred_ova_desirability':float(-po[j]),'is_true_pareto':bool(true[j]),'is_predicted_natural_pareto':bool(pred[j])}\n   for k,v in pvals.items(): row[k]=float(v[j])\n   raw.append(row)\n pd.DataFrame(raw).to_csv(OUT/'raw_external_scores.csv.gz',index=False,compression='gzip'); pd.DataFrame(geo).to_csv(OUT/'score_geometry.csv',index=False); hist.to_csv(OUT/'training_diagnostics.csv.gz',index=False,compression='gzip'); pd.DataFrame([{'seed':SEED,'feature':FEATURE,'model':'Standard-MOLM','trainable_parameters':sum(p.numel() for p in m.parameters() if p.requires_grad)}]).to_csv(OUT/'parameter_counts.csv',index=False); (OUT/'manifest.json').write_text(json.dumps({'status':'ok','stage':'external','seed':SEED,'feature':FEATURE,'lock_sha256':LOCK_HASH,'latent_pca_spaces':[64,32,16]},indent=2))\nelse: raise ValueError(STAGE)\n"
compile(STANDARD_WORKER_SOURCE,'standard_molm_worker.py','exec')
STANDARD_WORKER_PATH=CODE_DIR/'standard_molm_worker.py'
STANDARD_WORKER_PATH.write_text(STANDARD_WORKER_SOURCE,encoding='utf-8')
print('Standard worker SHA256:',hashlib.sha256(STANDARD_WORKER_SOURCE.encode()).hexdigest())

## 6. Runtime preflight + Standard MOLM mutation grid

This is the expensive new main-model computation: `5 seeds x 5 representations x 16 holdouts`. Jobs are resumable and distributed across the two T4 GPUs.

In [ ]:
# One-epoch real-data preflight.
preflight_script = """
import os, numpy as np
from pathlib import Path
os.environ['MOLM_STAGE']='mutation'; os.environ['MOLM_EPOCHS']='1'
exec(Path(os.environ['MOLM_STANDARD_WORKER']).read_text())
"""
# Instead of running a whole 16-holdout worker for smoke, verify import/architecture and one update directly.
env=os.environ.copy(); env.update({'MOLM_REPO':str(REPO),'MOLM_CODE_DIR':str(CODE_DIR),'MOLM_FEATURE_TYPE':'onehot','MOLM_SEED':'42','MOLM_EMI_FEATURE_PATH':str(EMI_FEATURE_PATH),'MOLM_HOLDOUT_PATH':str(ENRICHED_HOLDOUT_PATH),'MOLM_EXTERNAL_FEATURE_PATH':str(EXTERNAL_FEATURE_PATH),'MOLM_OUTPUT_DIR':str(WORK_ROOT/'preflight_standard'),'MOLM_LOCK_HASH':ANALYSIS_LOCK_SHA,'MOLM_STAGE':'mutation','MOLM_EPOCHS':'1','MOLM_BATCH_SIZE':str(BATCH_SIZE),'MOLM_LEARNING_RATE':str(LEARNING_RATE),'PYTHONPATH':str(CODE_DIR)+os.pathsep+str(REPO),'CUDA_VISIBLE_DEVICES':'0'})
# The worker will run all holdouts at 1 epoch, which doubles as a strong runtime preflight and is resumable in a disposable folder.
if (WORK_ROOT/'preflight_standard').exists(): shutil.rmtree(WORK_ROOT/'preflight_standard')
r=subprocess.run([sys.executable,'-u',str(STANDARD_WORKER_PATH)],env=env,capture_output=True,text=True)
print('\n'.join(r.stdout.splitlines()[-30:]))
if r.returncode!=0:
    print(r.stderr); raise RuntimeError('Standard-MOLM runtime preflight failed.')
shutil.rmtree(WORK_ROOT/'preflight_standard',ignore_errors=True)
print('STANDARD MOLM PREFLIGHT PASS')

In [ ]:
import queue, threading

def run_parallel_jobs(job_tuples, runner, gpu_ids=(0,1)):
    q=queue.Queue(); [q.put(x) for x in job_tuples]; records=[]; lk=threading.Lock()
    def loop(gpu):
        while True:
            try: job=q.get_nowait()
            except queue.Empty:return
            try: rec=runner(job,gpu)
            except Exception as e: rec={'job':repr(job),'gpu':gpu,'status':'failed','error':repr(e)}
            with lk: records.append(rec)
            q.task_done()
    ts=[threading.Thread(target=loop,args=(g,),daemon=True) for g in gpu_ids]
    [t.start() for t in ts]; [t.join() for t in ts]
    return pd.DataFrame(records)

def standard_job(job,gpu,stage):
    seed,feature=job; root=(STANDARD_MUT_DIR if stage=='mutation' else STANDARD_EXT_DIR)/f'seed_{seed}'/feature; root.mkdir(parents=True,exist_ok=True); manifest=root/'manifest.json'
    if RESUME and manifest.exists():
        try:
            m=json.loads(manifest.read_text())
            if m.get('status')=='ok' and m.get('lock_sha256')==ANALYSIS_LOCK_SHA and m.get('stage')==stage:
                return {'seed':seed,'feature':feature,'gpu':gpu,'stage':stage,'status':'skipped_verified'}
        except Exception: pass
    log=LOG_DIR/f'standard_{stage}_{seed}_{feature}.log'; env=os.environ.copy(); env.update({'CUDA_VISIBLE_DEVICES':str(gpu),'MOLM_REPO':str(REPO),'MOLM_CODE_DIR':str(CODE_DIR),'MOLM_FEATURE_TYPE':feature,'MOLM_SEED':str(seed),'MOLM_EMI_FEATURE_PATH':str(EMI_FEATURE_PATH),'MOLM_HOLDOUT_PATH':str(ENRICHED_HOLDOUT_PATH),'MOLM_EXTERNAL_FEATURE_PATH':str(EXTERNAL_FEATURE_PATH),'MOLM_OUTPUT_DIR':str(root),'MOLM_LOCK_HASH':ANALYSIS_LOCK_SHA,'MOLM_STAGE':stage,'MOLM_EPOCHS':str(EPOCHS),'MOLM_BATCH_SIZE':str(BATCH_SIZE),'MOLM_LEARNING_RATE':str(LEARNING_RATE),'PYTHONPATH':str(CODE_DIR)+os.pathsep+str(REPO)})
    with log.open('w') as f: p=subprocess.run([sys.executable,'-u',str(STANDARD_WORKER_PATH)],env=env,stdout=f,stderr=subprocess.STDOUT,text=True)
    if p.returncode!=0: raise RuntimeError('\n'.join(log.read_text(errors='replace').splitlines()[-80:]))
    return {'seed':seed,'feature':feature,'gpu':gpu,'stage':stage,'status':'ok'}

if RUN_STANDARD_MAIN:
    jobs=[(s,f) for s in SEEDS for f in FEATURE_TYPES]
    standard_mut_manifest=run_parallel_jobs(jobs,lambda job,gpu:standard_job(job,gpu,'mutation'))
    display(standard_mut_manifest)
    if not standard_mut_manifest.status.isin(['ok','skipped_verified']).all(): raise RuntimeError('Standard mutation jobs failed')

## 7. Combine frozen base predictions + Standard MOLM and re-run mutation statistics

In [ ]:
from scipy.stats import binomtest, friedmanchisquare, wilcoxon, spearmanr
from itertools import product, combinations
from sklearn.metrics import accuracy_score, balanced_accuracy_score, matthews_corrcoef, f1_score, roc_auc_score, average_precision_score, confusion_matrix

base_mut_raw=pd.read_csv(base_file('mutation_raw_predictions_all.csv.gz'))
base_mut_metrics=pd.read_csv(base_file('mutation_metrics_all.csv'))
std_raw=[]; std_met=[]
for s in SEEDS:
 for f in FEATURE_TYPES:
  root=STANDARD_MUT_DIR/f'seed_{s}'/f; std_raw.append(pd.read_csv(root/'raw_predictions.csv.gz')); std_met.append(pd.read_csv(root/'metrics.csv'))
standard_mut_raw=pd.concat(std_raw,ignore_index=True); standard_mut_metrics=pd.concat(std_met,ignore_index=True)
mutation_raw=pd.concat([base_mut_raw,standard_mut_raw],ignore_index=True,sort=False)
mutation_metrics=pd.concat([base_mut_metrics,standard_mut_metrics],ignore_index=True,sort=False)
mutation_raw.to_csv(ANALYSIS_DIR/'mutation_raw_predictions_4models.csv.gz',index=False,compression='gzip')
mutation_metrics.to_csv(ANALYSIS_DIR/'mutation_metrics_4models.csv',index=False)
print(mutation_raw.model.value_counts())

In [ ]:
def holm_adjust(values):
    p=np.asarray(values,float); out=np.full(len(p),np.nan); valid=np.flatnonzero(np.isfinite(p))
    order=valid[np.argsort(p[valid])]; running=0.; m=len(order)
    for rank,idx in enumerate(order): running=max(running,(m-rank)*p[idx]); out[idx]=min(1.,running)
    return out

def exact_sign_flip_pvalue(d):
    d=np.asarray(d,float); d=d[np.isfinite(d)]
    if not len(d):return np.nan
    observed=abs(d.mean()); signs=np.asarray(list(product([-1.,1.],repeat=len(d))))
    return float(np.mean(np.abs((signs*d[None,:]).mean(axis=1))>=observed-1e-15))

def exact_mcnemar(y,a,b):
    y=np.asarray(y,int); a=np.asarray(a,int); b=np.asarray(b,int); ca=a==y; cb=b==y; ao=int(np.sum(ca&~cb)); bo=int(np.sum(~ca&cb)); n=ao+bo
    return ao,bo,n,(1.0 if n==0 else float(binomtest(min(ao,bo),n=n,p=.5,alternative='two-sided').pvalue))

def confusion_metrics_vectorized(pos_pred,neg_pred):
    tp=pos_pred.sum(axis=1).astype(float); fn=pos_pred.shape[1]-tp; fp=neg_pred.sum(axis=1).astype(float); tn=neg_pred.shape[1]-fp; total=tp+tn+fp+fn
    acc=(tp+tn)/total; sens=tp/np.maximum(tp+fn,1); spec=tn/np.maximum(tn+fp,1); bal=(sens+spec)/2; num=tp*tn-fp*fn; den=np.sqrt((tp+fp)*(tp+fn)*(tn+fp)*(tn+fn)); mcc=np.divide(num,den,out=np.zeros_like(num),where=den>0)
    return {'accuracy':acc,'balanced_accuracy':bal,'mcc':mcc}

def paired_stratified_bootstrap(y,sa,sb,draws,seed):
    y=np.asarray(y,int); pa=(np.asarray(sa)>=0).astype(np.int8); pb=(np.asarray(sb)>=0).astype(np.int8); pos=np.flatnonzero(y==1); neg=np.flatnonzero(y==0)
    if not len(pos) or not len(neg):return None
    rng=np.random.default_rng(seed); pdx=rng.choice(pos,size=(draws,len(pos)),replace=True); ndx=rng.choice(neg,size=(draws,len(neg)),replace=True); A=confusion_metrics_vectorized(pa[pdx],pa[ndx]); B=confusion_metrics_vectorized(pb[pdx],pb[ndx]); out={}
    for metric in ['mcc','accuracy','balanced_accuracy']:
        delta=A[metric]-B[metric]; lo,hi=np.quantile(delta,[.025,.975]); p=min(1.,2*min(np.mean(delta<=0),np.mean(delta>=0))); out[metric]=(float(delta.mean()),float(np.median(delta)),float(lo),float(hi),float(p))
    return out

metric_cols=['accuracy','balanced_accuracy','mcc','f1','sensitivity','specificity','auroc','auprc']
mutation_site_summary=[]
for keys,g in mutation_metrics.groupby(['analysis_role','holdout_type','holdout_id','kabat_site','heldout_residue','feature','model','task'],sort=False):
    row=dict(zip(['analysis_role','holdout_type','holdout_id','kabat_site','heldout_residue','feature','model','task'],keys)); row['n_optimization_seeds']=int(g.loc[g.seed>=0,'seed'].nunique()); row['deterministic_model']=row['model']=='LDA'
    for m in metric_cols: row[m+'_mean']=float(g[m].mean()); row[m+'_sd']=float(g[m].std(ddof=1)) if len(g)>1 else np.nan
    mutation_site_summary.append(row)
mutation_site_summary=pd.DataFrame(mutation_site_summary)
mutation_site_summary.to_csv(ANALYSIS_DIR/'mutation_site_metrics_mean_sd_4models.csv',index=False)
mutation_aggregate=mutation_site_summary.groupby(['analysis_role','holdout_type','feature','model','task'],as_index=False).agg(n_sites=('holdout_id','nunique'),site_mean_mcc=('mcc_mean','mean'),site_sd_mcc=('mcc_mean','std'),site_mean_balanced_accuracy=('balanced_accuracy_mean','mean'),site_mean_auroc=('auroc_mean','mean'),site_mean_auprc=('auprc_mean','mean'))
mutation_aggregate.to_csv(ANALYSIS_DIR/'mutation_aggregate_summary_4models.csv',index=False)
display(mutation_aggregate.sort_values(['holdout_type','task','feature','site_mean_mcc'],ascending=[True,True,True,False]))

In [ ]:
# Seed-consensus predictions for all four models.
mutation_consensus=(mutation_raw.groupby(['feature','holdout_id','holdout_type','analysis_role','kabat_site','heldout_residue','task','global_row_id','sequence_id','model','y_true'],as_index=False).agg(mean_score=('score','mean'),score_sd=('score','std')))
mutation_consensus['y_pred']=(mutation_consensus.mean_score>=0).astype(int)
mutation_consensus.to_csv(ANALYSIS_DIR/'mutation_seed_consensus_scores_4models.csv.gz',index=False,compression='gzip')
models=['Routed-MOLM','Standard-MOLM','NN','LDA']
model_pairs=list(combinations(models,2))

mrows=[]
for (feature,hid,task),g in mutation_consensus.groupby(['feature','holdout_id','task'],sort=False):
    piv=g.pivot_table(index=['global_row_id','sequence_id','y_true'],columns='model',values='y_pred',aggfunc='first').reset_index(); meta=g.iloc[0]
    for a,b in model_pairs:
        if a not in piv or b not in piv:continue
        ao,bo,n,p=exact_mcnemar(piv.y_true,piv[a],piv[b]); mrows.append({'feature':feature,'holdout_id':hid,'holdout_type':meta.holdout_type,'analysis_role':meta.analysis_role,'kabat_site':int(meta.kabat_site),'heldout_residue':meta.heldout_residue,'task':task,'model_a':a,'model_b':b,'a_correct_b_wrong':ao,'a_wrong_b_correct':bo,'discordant':n,'p_exact':p})
mutation_mcnemar=pd.DataFrame(mrows); mutation_mcnemar['p_holm_8_within_role']=np.nan; mutation_mcnemar['p_holm_16_all']=np.nan
for _,ix in mutation_mcnemar.groupby(['feature','task','model_a','model_b','holdout_type']).groups.items(): mutation_mcnemar.loc[list(ix),'p_holm_8_within_role']=holm_adjust(mutation_mcnemar.loc[list(ix),'p_exact'])
for _,ix in mutation_mcnemar.groupby(['feature','task','model_a','model_b']).groups.items(): mutation_mcnemar.loc[list(ix),'p_holm_16_all']=holm_adjust(mutation_mcnemar.loc[list(ix),'p_exact'])
mutation_mcnemar.to_csv(ANALYSIS_DIR/'mutation_mcnemar_seed_consensus_4models.csv',index=False)

# Preserve original pre-specified bootstrap comparisons and add Standard-MOLM comparisons.
bootstrap_pairs=[('Routed-MOLM','LDA'),('Routed-MOLM','NN'),('Routed-MOLM','Standard-MOLM'),('Standard-MOLM','NN'),('Standard-MOLM','LDA')]
brows=[]
for (feature,hid,task),g in mutation_consensus.groupby(['feature','holdout_id','task'],sort=False):
    piv=g.pivot_table(index=['global_row_id','sequence_id','y_true'],columns='model',values='mean_score',aggfunc='first').reset_index(); meta=g.iloc[0]
    for pi,(a,b) in enumerate(bootstrap_pairs):
        if a not in piv or b not in piv:continue
        res=paired_stratified_bootstrap(piv.y_true,piv[a],piv[b],BOOTSTRAP_DRAWS,BOOTSTRAP_SEED+100000*pi+int(meta.kabat_site)+1000*(task=='ova')+10000*FEATURE_TYPES.index(feature))
        if res is None:continue
        for metric,v in res.items(): brows.append({'feature':feature,'holdout_id':hid,'holdout_type':meta.holdout_type,'analysis_role':meta.analysis_role,'kabat_site':int(meta.kabat_site),'heldout_residue':meta.heldout_residue,'task':task,'model_a':a,'model_b':b,'metric':metric,'draws':BOOTSTRAP_DRAWS,'mean_delta':v[0],'median_delta':v[1],'ci_low':v[2],'ci_high':v[3],'p_bootstrap':v[4]})
mutation_bootstrap=pd.DataFrame(brows); mutation_bootstrap['p_holm_8_within_role']=np.nan; mutation_bootstrap['p_holm_16_all']=np.nan
for _,ix in mutation_bootstrap.groupby(['feature','task','model_a','model_b','metric','holdout_type']).groups.items(): mutation_bootstrap.loc[list(ix),'p_holm_8_within_role']=holm_adjust(mutation_bootstrap.loc[list(ix),'p_bootstrap'])
for _,ix in mutation_bootstrap.groupby(['feature','task','model_a','model_b','metric']).groups.items(): mutation_bootstrap.loc[list(ix),'p_holm_16_all']=holm_adjust(mutation_bootstrap.loc[list(ix),'p_bootstrap'])
mutation_bootstrap.to_csv(ANALYSIS_DIR/'mutation_paired_sequence_bootstrap_10000_4models.csv',index=False)

In [ ]:
# Site blocks (not random seeds): Friedman omnibus across four models + pairwise Wilcoxon/sign-flip.
holdout_means=mutation_site_summary[['feature','holdout_type','analysis_role','holdout_id','kabat_site','heldout_residue','task','model','mcc_mean']].copy()
omni=[]; pair=[]
for feature in FEATURE_TYPES:
 for task in ['affinity','ova']:
  for hg in ['top','wildtype','combined']:
   sub=holdout_means[(holdout_means.feature==feature)&(holdout_means.task==task)]
   if hg!='combined':sub=sub[sub.holdout_type==hg]
   piv=sub.pivot_table(index='holdout_id',columns='model',values='mcc_mean',aggfunc='first').dropna()
   if len(piv)==0 or not all(m in piv for m in models):continue
   fr=friedmanchisquare(*[piv[m] for m in models]); omni.append({'feature':feature,'task':task,'holdout_group':hg,'n_holdouts':len(piv),'friedman_statistic':float(fr.statistic),'friedman_p':float(fr.pvalue)})
   loc=[]
   for a,b in model_pairs:
    d=piv[a]-piv[b]
    if np.allclose(d,0): stat,p=0.,1.
    else:
     w=wilcoxon(d,zero_method='wilcox',alternative='two-sided',method='auto'); stat,p=float(w.statistic),float(w.pvalue)
    loc.append({'feature':feature,'task':task,'holdout_group':hg,'n_holdouts':len(piv),'model_a':a,'model_b':b,'mean_delta_mcc':float(d.mean()),'median_delta_mcc':float(d.median()),'wins':int((d>0).sum()),'ties':int(np.isclose(d,0).sum()),'losses':int((d<0).sum()),'wilcoxon_statistic':stat,'wilcoxon_p':p,'exact_sign_flip_p':exact_sign_flip_pvalue(d)})
   adj=holm_adjust([x['wilcoxon_p'] for x in loc])
   for x,a in zip(loc,adj): x['wilcoxon_p_holm_6_pairs']=float(a); pair.append(x)
mutation_omnibus=pd.DataFrame(omni); mutation_pairwise=pd.DataFrame(pair); mutation_omnibus.to_csv(ANALYSIS_DIR/'mutation_friedman_omnibus_4models.csv',index=False); mutation_pairwise.to_csv(ANALYSIS_DIR/'mutation_holdout_pairwise_tests_4models.csv',index=False)
display(mutation_pairwise[(mutation_pairwise.model_a=='Routed-MOLM')|(mutation_pairwise.model_a=='Standard-MOLM')].head(80))

In [ ]:
# Direct Site-vs-Mean representation contrasts within every model.
rows=[]
for model in models:
 for task in ['affinity','ova']:
  for ht in ['top','wildtype']:
   sub=holdout_means[(holdout_means.model==model)&(holdout_means.task==task)&(holdout_means.holdout_type==ht)]
   piv=sub.pivot_table(index='holdout_id',columns='feature',values='mcc_mean',aggfunc='first').dropna(); local=[]
   for sf,mf,label in REPRESENTATION_CONTRASTS:
    if sf not in piv or mf not in piv:continue
    d=piv[sf]-piv[mf]
    if np.allclose(d,0): ws,wp=0.,1.
    else:
     w=wilcoxon(d,zero_method='wilcox',alternative='two-sided',method='auto'); ws,wp=float(w.statistic),float(w.pvalue)
    local.append({'model':model,'task':task,'holdout_type':ht,'analysis_role':'primary' if ht=='top' else 'secondary','representation_contrast':label,'site_feature':sf,'mean_feature':mf,'n_sites':len(d),'mean_delta_mcc':float(d.mean()),'median_delta_mcc':float(d.median()),'wins':int((d>0).sum()),'ties':int(np.isclose(d,0).sum()),'losses':int((d<0).sum()),'wilcoxon_statistic':ws,'wilcoxon_p':wp,'exact_sign_flip_p':exact_sign_flip_pvalue(d)})
   adj=holm_adjust([x['wilcoxon_p'] for x in local])
   for x,a in zip(local,adj):x['wilcoxon_p_holm_2_representation_contrasts']=float(a);rows.append(x)
mutation_representation_tests=pd.DataFrame(rows); mutation_representation_tests.to_csv(ANALYSIS_DIR/'mutation_representation_contrast_tests_4models.csv',index=False)
display(mutation_representation_tests[mutation_representation_tests.model=='Standard-MOLM'])

## 8. Standard MOLM external run + four-model Spearman inference

In [ ]:
if RUN_STANDARD_MAIN:
    jobs=[(s,f) for s in SEEDS for f in FEATURE_TYPES]
    standard_ext_manifest=run_parallel_jobs(jobs,lambda job,gpu:standard_job(job,gpu,'external'))
    display(standard_ext_manifest)
    if not standard_ext_manifest.status.isin(['ok','skipped_verified']).all():raise RuntimeError('Standard external jobs failed')

In [ ]:
base_ext_raw=pd.read_csv(base_file('external_raw_predictions_all.csv.gz'))
base_ext_geo=pd.read_csv(base_file('external_score_geometry_all.csv'))
stdraw=[]; stdgeo=[]
for s in SEEDS:
 for f in FEATURE_TYPES:
  root=STANDARD_EXT_DIR/f'seed_{s}'/f; stdraw.append(pd.read_csv(root/'raw_external_scores.csv.gz')); stdgeo.append(pd.read_csv(root/'score_geometry.csv'))
standard_ext_raw=pd.concat(stdraw,ignore_index=True); standard_ext_geo=pd.concat(stdgeo,ignore_index=True)
external_raw=pd.concat([base_ext_raw,standard_ext_raw],ignore_index=True,sort=False); external_geometry=pd.concat([base_ext_geo,standard_ext_geo],ignore_index=True,sort=False)
external_raw.to_csv(ANALYSIS_DIR/'external_raw_predictions_4models.csv.gz',index=False,compression='gzip'); external_geometry.to_csv(ANALYSIS_DIR/'external_score_geometry_4models.csv',index=False)
spearman_summary=external_geometry.groupby(['dataset','feature','model'],as_index=False).agg(n_runs=('seed','size'),n_optimization_seeds=('seed',lambda x:int(np.sum(np.asarray(x)>=0))),aff_spearman_mean=('aff_spearman','mean'),aff_spearman_sd=('aff_spearman','std'),ova_spearman_mean=('ova_spearman','mean'),ova_spearman_sd=('ova_spearman','std'),predicted_cross_task_mean=('predicted_cross_task_spearman','mean'),true_cross_task=('true_cross_task_spearman','first'))
spearman_summary.loc[spearman_summary.model=='LDA',['aff_spearman_sd','ova_spearman_sd']]=np.nan
spearman_summary.to_csv(ANALYSIS_DIR/'external_spearman_summary_4models.csv',index=False); display(spearman_summary[spearman_summary.dataset=='ISO'])

external_consensus=external_raw.groupby(['dataset','feature','model','row_id','sequence_id','true_aff','true_ova'],as_index=False).agg(pred_aff=('pred_aff','mean'),pred_ova=('pred_ova','mean'))
external_consensus.to_csv(ANALYSIS_DIR/'external_seed_consensus_scores_4models.csv.gz',index=False,compression='gzip')

In [ ]:
def paired_spearman_bootstrap(y,a,b,draws,seed):
    y=np.asarray(y,float); a=np.asarray(a,float); b=np.asarray(b,float); rng=np.random.default_rng(seed); n=len(y); delta=np.empty(draws,float)
    for i in range(draws):
        ix=rng.integers(0,n,size=n); delta[i]=spearmanr(a[ix],y[ix]).statistic-spearmanr(b[ix],y[ix]).statistic
    lo,hi=np.nanquantile(delta,[.025,.975]); p=min(1.,2*min(np.nanmean(delta<=0),np.nanmean(delta>=0)))
    return {'rho_a_point':float(spearmanr(a,y).statistic),'rho_b_point':float(spearmanr(b,y).statistic),'delta_point':float(spearmanr(a,y).statistic-spearmanr(b,y).statistic),'delta_boot_mean':float(np.nanmean(delta)),'ci_low':float(lo),'ci_high':float(hi),'p_bootstrap':float(p)}

external_pairs=[('Routed-MOLM','LDA'),('Routed-MOLM','NN'),('Routed-MOLM','Standard-MOLM'),('Standard-MOLM','NN'),('Standard-MOLM','LDA')]
rows=[]
for (dataset,feature),g in external_consensus.groupby(['dataset','feature'],sort=False):
    piv=g.pivot_table(index=['row_id','sequence_id','true_aff','true_ova'],columns='model',values=['pred_aff','pred_ova'],aggfunc='first').reset_index()
    for task,truth,pred in [('affinity',piv['true_aff'],'pred_aff'),('ova',piv['true_ova'],'pred_ova')]:
        for pi,(a,b) in enumerate(external_pairs):
            if (pred,a) not in piv or (pred,b) not in piv:continue
            res=paired_spearman_bootstrap(truth,piv[(pred,a)],piv[(pred,b)],BOOTSTRAP_DRAWS,BOOTSTRAP_SEED+100000*pi+10000*FEATURE_TYPES.index(feature)+1000*(task=='ova')+100*(str(dataset).startswith('IgG')))
            rows.append({'dataset':dataset,'feature':feature,'task':task,'model_a':a,'model_b':b,'draws':BOOTSTRAP_DRAWS,**res})
spearman_bootstrap=pd.DataFrame(rows); spearman_bootstrap['p_holm_5_representations']=np.nan
for _,ix in spearman_bootstrap.groupby(['dataset','task','model_a','model_b']).groups.items(): spearman_bootstrap.loc[list(ix),'p_holm_5_representations']=holm_adjust(spearman_bootstrap.loc[list(ix),'p_bootstrap'])
spearman_bootstrap.to_csv(ANALYSIS_DIR/'external_paired_spearman_bootstrap_10000_4models.csv',index=False); display(spearman_bootstrap[spearman_bootstrap.dataset=='ISO'])

In [ ]:
# Direct external Site-vs-Mean representation bootstrap within every model.
rows=[]
for (dataset,model),g in external_consensus.groupby(['dataset','model'],sort=False):
    piv=g.pivot_table(index=['row_id','sequence_id','true_aff','true_ova'],columns='feature',values=['pred_aff','pred_ova'],aggfunc='first').reset_index(); local=[]
    for ci,(sf,mf,label) in enumerate(REPRESENTATION_CONTRASTS):
        for task,truth,pred in [('affinity',piv['true_aff'],'pred_aff'),('ova',piv['true_ova'],'pred_ova')]:
            if (pred,sf) not in piv or (pred,mf) not in piv:continue
            res=paired_spearman_bootstrap(truth,piv[(pred,sf)],piv[(pred,mf)],BOOTSTRAP_DRAWS,BOOTSTRAP_SEED+700000*ci+1000*(task=='ova')+100*(str(dataset).startswith('IgG')))
            local.append({'dataset':dataset,'model':model,'task':task,'representation_contrast':label,'site_feature':sf,'mean_feature':mf,'draws':BOOTSTRAP_DRAWS,**res})
    ldf=pd.DataFrame(local)
    for task in ['affinity','ova']:
        ix=ldf.index[ldf.task==task]; ldf.loc[ix,'p_holm_2_representation_contrasts']=holm_adjust(ldf.loc[ix,'p_bootstrap'])
    rows.extend(ldf.to_dict('records'))
external_representation_bootstrap=pd.DataFrame(rows); external_representation_bootstrap.to_csv(ANALYSIS_DIR/'external_representation_spearman_bootstrap_10000_4models.csv',index=False)
display(external_representation_bootstrap[(external_representation_bootstrap.dataset=='ISO')&(external_representation_bootstrap.model=='Standard-MOLM')])

## 9. Fixed-budget Pareto evaluation for all four main models

The primary cross-model comparison uses **output logits for every model**, so every method is evaluated in the same two-objective score space. Latent-PCA robustness for Standard MOLM is a separate analysis below.

In [ ]:
def pareto_mask_max(points):
    points=np.asarray(points,float); mask=np.ones(len(points),bool)
    for i in range(len(points)):
        if (np.all(points>=points[i],axis=1)&np.any(points>points[i],axis=1)).any():mask[i]=False
    return mask

def nondominated_sort(points):
    remaining=np.arange(len(points)); fronts=[]
    while len(remaining):
        mask=pareto_mask_max(points[remaining]); fronts.append(remaining[mask]); remaining=remaining[~mask]
    return fronts

def crowding_distance(points):
    points=np.asarray(points,float); d=np.zeros(len(points),float)
    if len(points)<=2:d[:]=np.inf;return d
    for obj in range(points.shape[1]):
        order=np.argsort(points[:,obj],kind='mergesort'); d[order[0]]=d[order[-1]]=np.inf; span=points[order[-1],obj]-points[order[0],obj]
        if span<=0:continue
        for r in range(1,len(points)-1):
            cur=order[r]
            if np.isfinite(d[cur]):d[cur]+=(points[order[r+1],obj]-points[order[r-1],obj])/span
    return d

def select_fixed_budget(points,identifiers,k):
    points=np.asarray(points,float); ids=np.asarray(identifiers).astype(str); selected=[]
    for front in nondominated_sort(points):
        if len(selected)+len(front)<=k:selected.extend(front.tolist());continue
        rem=k-len(selected); dist=crowding_distance(points[front]); order=sorted(range(len(front)),key=lambda j:(-dist[j],ids[front[j]]));selected.extend(front[order[:rem]].tolist());break
    return np.asarray(selected,int)

def normalize_true_objectives(a,o):
    p=np.c_[np.asarray(a,float),-np.asarray(o,float)]; mn=p.min(axis=0); sp=p.max(axis=0)-mn; sp[sp==0]=1.;return (p-mn)/sp

def hypervolume_2d_max(points):
    points=np.asarray(points,float); points=points[np.all(points>=0,axis=1)]
    if not len(points):return 0.
    points=points[pareto_mask_max(points)];points=points[np.argsort(points[:,0])];total=0.;prev=0.
    for x,y in points:total+=max(0.,x-prev)*max(0.,y);prev=max(prev,x)
    return float(total)
def igd(true_front,selected):return float(np.sqrt(((true_front[:,None,:]-selected[None,:,:])**2).sum(axis=2)).min(axis=1).mean())
def recall_auc(g):
    g=g.sort_values('k');x=g.k.to_numpy(float);y=g.recall_at_k.to_numpy(float);return float(np.trapz(y,x)/(x[-1]-x[0])) if len(x)>1 else np.nan

budget=[]; curve=[]
for (seed,feature,dataset,model),g in external_raw.groupby(['seed','feature','dataset','model'],sort=False):
    g=g.sort_values('row_id').reset_index(drop=True); pred=np.c_[g.pred_aff.to_numpy(float),-g.pred_ova.to_numpy(float)]; true=normalize_true_objectives(g.true_aff,g.true_ova); tm=g.is_true_pareto.astype(bool).to_numpy() if 'is_true_pareto' in g else pareto_mask_max(np.c_[g.true_aff,-g.true_ova]); ntrue=int(tm.sum()); prev=ntrue/len(g)
    for k in range(1,min(FULL_CURVE_MAX_K,len(g))+1):
        sel=select_fixed_budget(pred,g.sequence_id,k);hits=int(tm[sel].sum());curve.append({'seed':int(seed),'feature':feature,'dataset':dataset,'model':model,'k':k,'hits':hits,'recall_at_k':hits/max(ntrue,1)})
        if k in K_VALUES:
            precision=hits/k;budget.append({'seed':int(seed),'seed_protocol':'deterministic' if int(seed)<0 else 'optimization','feature':feature,'dataset':dataset,'model':model,'k':k,'hits':hits,'recall_at_k':hits/max(ntrue,1),'precision_at_k':precision,'enrichment_at_k':precision/prev,'hypervolume_true_selected':hypervolume_2d_max(true[sel]),'igd_true_front_to_selected':igd(true[tm],true[sel])})
pareto_by_run=pd.DataFrame(budget);pareto_curve=pd.DataFrame(curve);pareto_by_run.to_csv(ANALYSIS_DIR/'pareto_fixed_budget_by_run_4models.csv',index=False);pareto_curve.to_csv(ANALYSIS_DIR/'pareto_recall_curve_k1_25_4models.csv',index=False)
pareto_summary=pareto_by_run.groupby(['dataset','feature','model','k'],as_index=False).agg(n_runs=('seed','size'),hits_mean=('hits','mean'),hits_sd=('hits','std'),recall_mean=('recall_at_k','mean'),recall_sd=('recall_at_k','std'),precision_mean=('precision_at_k','mean'),enrichment_mean=('enrichment_at_k','mean'),hypervolume_mean=('hypervolume_true_selected','mean'),hypervolume_sd=('hypervolume_true_selected','std'),igd_mean=('igd_true_front_to_selected','mean'),igd_sd=('igd_true_front_to_selected','std'))
auc=pd.DataFrame([{'seed':int(s),'feature':f,'dataset':d,'model':m,'recall_auc_k1_25':recall_auc(g)} for (s,f,d,m),g in pareto_curve.groupby(['seed','feature','dataset','model'])]);auc_summary=auc.groupby(['dataset','feature','model'],as_index=False).agg(auc_mean=('recall_auc_k1_25','mean'),auc_sd=('recall_auc_k1_25','std'))
pareto_summary.to_csv(ANALYSIS_DIR/'pareto_fixed_budget_summary_4models.csv',index=False);auc.to_csv(ANALYSIS_DIR/'pareto_recall_auc_by_run_4models.csv',index=False);auc_summary.to_csv(ANALYSIS_DIR/'pareto_recall_auc_summary_4models.csv',index=False)
display(pareto_summary[(pareto_summary.dataset=='ISO')&(pareto_summary.k.isin(K_VALUES))])

## 10. Standard-MOLM latent-PCA robustness

For Standard-MOLM only, compare four predeclared score spaces under the **same fixed K**:
- output logits;
- tower-64D -> PCA(1);
- tower-32D -> PCA(1);
- latent-16D -> PCA(1).

Each PCA is fit on full EMI training representations for that seed/feature, then frozen before ISO/IgG transformation. PCA sign is oriented using the corresponding EMI training logit only.


In [ ]:
if RUN_STANDARD_LATENT_ROBUSTNESS:
    sr=standard_ext_raw.copy(); spaces={'logits':('pred_aff','pred_ova'),'pca64':('pca64_aff','pca64_ova'),'pca32':('pca32_aff','pca32_ova'),'pca16':('pca16_aff','pca16_ova')}; rows=[]; curves=[]
    for (seed,feature,dataset),g in sr.groupby(['seed','feature','dataset'],sort=False):
        g=g.sort_values('row_id').reset_index(drop=True); true=normalize_true_objectives(g.true_aff,g.true_ova); tm=pareto_mask_max(np.c_[g.true_aff,-g.true_ova]); ntrue=int(tm.sum()); prev=ntrue/len(g)
        for space,(ac,oc) in spaces.items():
            pred=np.c_[g[ac].to_numpy(float),-g[oc].to_numpy(float)]
            for k in range(1,min(FULL_CURVE_MAX_K,len(g))+1):
                sel=select_fixed_budget(pred,g.sequence_id,k);hits=int(tm[sel].sum());curves.append({'seed':int(seed),'feature':feature,'dataset':dataset,'score_space':space,'k':k,'recall_at_k':hits/max(ntrue,1)})
                if k in K_VALUES:
                    precision=hits/k;rows.append({'seed':int(seed),'feature':feature,'dataset':dataset,'score_space':space,'k':k,'hits':hits,'recall_at_k':hits/max(ntrue,1),'precision_at_k':precision,'enrichment_at_k':precision/prev,'hypervolume_true_selected':hypervolume_2d_max(true[sel]),'igd_true_front_to_selected':igd(true[tm],true[sel])})
    latent_run=pd.DataFrame(rows);latent_curve=pd.DataFrame(curves);latent_auc=pd.DataFrame([{'seed':s,'feature':f,'dataset':d,'score_space':sp,'recall_auc_k1_25':recall_auc(g)} for (s,f,d,sp),g in latent_curve.groupby(['seed','feature','dataset','score_space'])]);latent_summary=latent_run.groupby(['dataset','feature','score_space','k'],as_index=False).agg(recall_mean=('recall_at_k','mean'),recall_sd=('recall_at_k','std'),precision_mean=('precision_at_k','mean'),enrichment_mean=('enrichment_at_k','mean'),hypervolume_mean=('hypervolume_true_selected','mean'),igd_mean=('igd_true_front_to_selected','mean'));latent_auc_summary=latent_auc.groupby(['dataset','feature','score_space'],as_index=False).agg(auc_mean=('recall_auc_k1_25','mean'),auc_sd=('recall_auc_k1_25','std'))
    latent_run.to_csv(ANALYSIS_DIR/'standard_latent_pca_fixed_budget_by_run.csv',index=False);latent_summary.to_csv(ANALYSIS_DIR/'standard_latent_pca_fixed_budget_summary.csv',index=False);latent_auc.to_csv(ANALYSIS_DIR/'standard_latent_pca_recall_auc_by_run.csv',index=False);latent_auc_summary.to_csv(ANALYSIS_DIR/'standard_latent_pca_recall_auc_summary.csv',index=False)
    display(latent_summary[(latent_summary.dataset=='ISO')&(latent_summary.feature=='onehot')])

## 11. Routed-MOLM A-H component ablation

Default: OneHot only, five seeds, all 16 mutation holdouts, plus ISO/IgG external evaluation. This isolates the architectural/methodological contributions without multiplying the five-representation study again.

In [ ]:
COMPONENT_WORKER_PATH=CODE_DIR/'molm_routed_component_ablation_worker_v2.py'

def component_job(job,gpu,stage):
    seed,feature=job; root=COMPONENT_DIR/stage/f'seed_{seed}'/feature; root.mkdir(parents=True,exist_ok=True); done=root/'JOB_DONE.json'
    if RESUME and done.exists():
        try:
            d=json.loads(done.read_text());
            if d.get('lock_sha256')==ANALYSIS_LOCK_SHA and d.get('stage')==stage:return {'seed':seed,'feature':feature,'stage':stage,'gpu':gpu,'status':'skipped_verified'}
        except Exception:pass
    log=LOG_DIR/f'component_{stage}_{seed}_{feature}.log';env=os.environ.copy();env.update({'CUDA_VISIBLE_DEVICES':str(gpu),'MOLM_REPO':str(REPO),'MOLM_FEATURE_TYPE':feature,'MOLM_SEED':str(seed),'MOLM_STAGE':stage,'MOLM_ARM_SET':COMPONENT_ARM_SET,'MOLM_JOB_OUTPUT':str(root),'MOLM_EMI_FEATURE_PATH':str(EMI_FEATURE_PATH),'MOLM_EXTERNAL_FEATURE_PATH':str(EXTERNAL_FEATURE_PATH),'MOLM_HOLDOUT_PATH':str(ENRICHED_HOLDOUT_PATH),'MOLM_LOCK_HASH':ANALYSIS_LOCK_SHA,'MOLM_EPOCHS':str(EPOCHS),'MOLM_BATCH_SIZE':str(BATCH_SIZE),'MOLM_LEARNING_RATE':str(LEARNING_RATE),'PYTHONPATH':str(CODE_DIR)+os.pathsep+str(REPO)})
    with log.open('w') as f:p=subprocess.run([sys.executable,'-u',str(COMPONENT_WORKER_PATH)],env=env,stdout=f,stderr=subprocess.STDOUT,text=True)
    if p.returncode!=0:raise RuntimeError('\n'.join(log.read_text(errors='replace').splitlines()[-100:]))
    done.write_text(json.dumps({'stage':stage,'seed':seed,'feature':feature,'lock_sha256':ANALYSIS_LOCK_SHA},indent=2));return {'seed':seed,'feature':feature,'stage':stage,'gpu':gpu,'status':'ok'}

if RUN_COMPONENT_ABLATION:
    jobs=[(s,f) for s in SEEDS for f in COMPONENT_FEATURES]; comp_mut_manifest=run_parallel_jobs(jobs,lambda j,g:component_job(j,g,'mutation'));display(comp_mut_manifest)
    if RUN_COMPONENT_EXTERNAL:
        comp_ext_manifest=run_parallel_jobs(jobs,lambda j,g:component_job(j,g,'external'));display(comp_ext_manifest)

In [ ]:
if RUN_COMPONENT_ABLATION:
    # Collect mutation metrics and diagnostics.
    cm=[];cd=[];cp=[]
    for s in SEEDS:
     for f in COMPONENT_FEATURES:
      root=COMPONENT_DIR/'mutation'/f'seed_{s}'/f
      for h in enriched:
       for arm in ['shared_base','shared_dom','shared_dom_pcgrad','shared_dom_private','full_routed','independent_st_dom','shared_capacity_dom','shared_capacity_dom_pcgrad']:
        d=root/h['holdout_id']/arm
        if (d/'metrics.csv').exists():cm.append(pd.read_csv(d/'metrics.csv'));cd.append(pd.read_csv(d/'training_diagnostics.csv.gz'));cp.append(pd.read_csv(d/'parameter_count.csv'))
    component_metrics=pd.concat(cm,ignore_index=True);component_diag=pd.concat(cd,ignore_index=True);component_params=pd.concat(cp,ignore_index=True)
    component_metrics.to_csv(ANALYSIS_DIR/'component_ablation_mutation_metrics.csv',index=False);component_diag.to_csv(ANALYSIS_DIR/'component_ablation_training_diagnostics.csv.gz',index=False,compression='gzip');component_params.to_csv(ANALYSIS_DIR/'component_ablation_parameter_counts.csv',index=False)
    site=component_metrics.groupby(['feature','holdout_id','kabat_site','task','arm','arm_label'],as_index=False).agg(mcc_mean=('mcc','mean'),balanced_accuracy_mean=('balanced_accuracy','mean'),accuracy_mean=('accuracy','mean'),auroc_mean=('auroc','mean'),auprc_mean=('auprc','mean'))
    summary=site.groupby(['feature','task','arm','arm_label'],as_index=False).agg(n_sites=('holdout_id','nunique'),mcc_mean=('mcc_mean','mean'),mcc_site_sd=('mcc_mean','std'),balanced_accuracy_mean=('balanced_accuracy_mean','mean'),auroc_mean=('auroc_mean','mean'),auprc_mean=('auprc_mean','mean'))
    summary.to_csv(ANALYSIS_DIR/'component_ablation_mutation_summary.csv',index=False)
    # Full Routed vs every ablated/control arm using sites as paired blocks.
    rows=[]
    for (feature,task),g in site.groupby(['feature','task']):
      piv=g.pivot_table(index='holdout_id',columns='arm',values='mcc_mean',aggfunc='first').dropna()
      local=[]
      for arm in [c for c in piv.columns if c!='full_routed']:
        d=piv.full_routed-piv[arm]; w=wilcoxon(d,zero_method='wilcox',alternative='two-sided',method='auto') if not np.allclose(d,0) else None
        local.append({'feature':feature,'task':task,'comparison':f'full_routed - {arm}','n_sites':len(d),'mean_delta_mcc':float(d.mean()),'median_delta_mcc':float(np.median(d)),'wins':int((d>0).sum()),'ties':int(np.isclose(d,0).sum()),'losses':int((d<0).sum()),'wilcoxon_p':1. if w is None else float(w.pvalue),'exact_sign_flip_p':exact_sign_flip_pvalue(d)})
      adj=holm_adjust([x['wilcoxon_p'] for x in local])
      for x,a in zip(local,adj):x['wilcoxon_p_holm']=float(a);rows.append(x)
    component_tests=pd.DataFrame(rows);component_tests.to_csv(ANALYSIS_DIR/'component_ablation_site_pairwise_tests.csv',index=False);display(summary);display(component_tests)

    if RUN_COMPONENT_EXTERNAL:
      em=[]
      for s in SEEDS:
       for f in COMPONENT_FEATURES:
        root=COMPONENT_DIR/'external'/f'seed_{s}'/f
        for arm in ['shared_base','shared_dom','shared_dom_pcgrad','shared_dom_private','full_routed','independent_st_dom','shared_capacity_dom','shared_capacity_dom_pcgrad']:
         p=root/arm/'metrics.csv'
         if p.exists():em.append(pd.read_csv(p))
      component_external=pd.concat(em,ignore_index=True);component_external.to_csv(ANALYSIS_DIR/'component_ablation_external_spearman_by_seed.csv',index=False);component_external_summary=component_external.groupby(['feature','dataset','arm','arm_label'],as_index=False).agg(aff_spearman_mean=('aff_spearman','mean'),aff_spearman_sd=('aff_spearman','std'),ova_spearman_mean=('ova_spearman','mean'),ova_spearman_sd=('ova_spearman','std'));component_external_summary.to_csv(ANALYSIS_DIR/'component_ablation_external_spearman_summary.csv',index=False);display(component_external_summary)

## 12. Ranking / gap loss ablation

Four identical Standard shared-MOLM architectures start from identical parameters and minibatch order for each seed/holdout; only ranking/gap terms change.

In [ ]:
LOSS_WORKER_PATH=CODE_DIR/'molm_ranking_gap_loss_ablation_worker.py'

def loss_job(job,gpu):
    seed,feature=job;root=LOSS_DIR/f'seed_{seed}'/feature;root.mkdir(parents=True,exist_ok=True);done=root/'JOB_DONE.json'
    if RESUME and done.exists():
      try:
       d=json.loads(done.read_text());
       if d.get('lock_sha256')==ANALYSIS_LOCK_SHA:return {'seed':seed,'feature':feature,'gpu':gpu,'status':'skipped_verified'}
      except Exception:pass
    log=LOG_DIR/f'loss_{seed}_{feature}.log';env=os.environ.copy();env.update({'CUDA_VISIBLE_DEVICES':str(gpu),'MOLM_REPO':str(REPO),'MOLM_FEATURE_TYPE':feature,'MOLM_SEED':str(seed),'MOLM_JOB_OUTPUT':str(root),'MOLM_EMI_FEATURE_PATH':str(EMI_FEATURE_PATH),'MOLM_HOLDOUT_PATH':str(ENRICHED_HOLDOUT_PATH),'MOLM_LOCK_HASH':ANALYSIS_LOCK_SHA,'MOLM_EPOCHS':str(EPOCHS),'MOLM_BATCH_SIZE':str(BATCH_SIZE),'MOLM_LEARNING_RATE':str(LEARNING_RATE),'PYTHONPATH':str(CODE_DIR)+os.pathsep+str(REPO)})
    with log.open('w') as f:p=subprocess.run([sys.executable,'-u',str(LOSS_WORKER_PATH)],env=env,stdout=f,stderr=subprocess.STDOUT,text=True)
    if p.returncode!=0:raise RuntimeError('\n'.join(log.read_text(errors='replace').splitlines()[-100:]))
    done.write_text(json.dumps({'seed':seed,'feature':feature,'lock_sha256':ANALYSIS_LOCK_SHA},indent=2));return {'seed':seed,'feature':feature,'gpu':gpu,'status':'ok'}

if RUN_LOSS_ABLATION:
    loss_manifest=run_parallel_jobs([(s,f) for s in SEEDS for f in LOSS_ABLATION_FEATURES],loss_job);display(loss_manifest)

In [ ]:
if RUN_LOSS_ABLATION:
    lm=[]
    arms=['L0_focal','LR_focal_ranking','LG_focal_gap','LRG_full']
    for s in SEEDS:
     for f in LOSS_ABLATION_FEATURES:
      root=LOSS_DIR/f'seed_{s}'/f
      for h in enriched:
       for arm in arms:
        p=root/h['holdout_id']/arm/'metrics.csv'
        if p.exists():lm.append(pd.read_csv(p))
    loss_metrics=pd.concat(lm,ignore_index=True);loss_metrics.to_csv(ANALYSIS_DIR/'ranking_gap_loss_ablation_metrics.csv',index=False)
    site=loss_metrics.groupby(['feature','holdout_id','kabat_site','task','loss_arm','loss_label'],as_index=False).agg(mcc_mean=('mcc','mean'),balanced_accuracy_mean=('balanced_accuracy','mean'),accuracy_mean=('accuracy','mean'),auroc_mean=('auroc','mean'),auprc_mean=('auprc','mean'))
    summary=site.groupby(['feature','task','loss_arm','loss_label'],as_index=False).agg(n_sites=('holdout_id','nunique'),mcc_mean=('mcc_mean','mean'),mcc_site_sd=('mcc_mean','std'),balanced_accuracy_mean=('balanced_accuracy_mean','mean'),auroc_mean=('auroc_mean','mean'),auprc_mean=('auprc_mean','mean'));summary.to_csv(ANALYSIS_DIR/'ranking_gap_loss_ablation_summary.csv',index=False)
    rows=[]
    for (feature,task),g in site.groupby(['feature','task']):
      piv=g.pivot_table(index='holdout_id',columns='loss_arm',values='mcc_mean',aggfunc='first').dropna();local=[]
      for b in ['L0_focal','LR_focal_ranking','LG_focal_gap']:
       d=piv['LRG_full']-piv[b];w=wilcoxon(d,zero_method='wilcox',alternative='two-sided',method='auto') if not np.allclose(d,0) else None;local.append({'feature':feature,'task':task,'comparison':f'LRG_full - {b}','n_sites':len(d),'mean_delta_mcc':float(d.mean()),'median_delta_mcc':float(np.median(d)),'wins':int((d>0).sum()),'ties':int(np.isclose(d,0).sum()),'losses':int((d<0).sum()),'wilcoxon_p':1. if w is None else float(w.pvalue),'exact_sign_flip_p':exact_sign_flip_pvalue(d)})
      adj=holm_adjust([x['wilcoxon_p'] for x in local])
      for x,a in zip(local,adj):x['wilcoxon_p_holm_3']=float(a);rows.append(x)
    loss_tests=pd.DataFrame(rows);loss_tests.to_csv(ANALYSIS_DIR/'ranking_gap_loss_ablation_site_tests.csv',index=False);display(summary);display(loss_tests)

## 13. Hamming-distance generalization v4 logic, now including Standard MOLM

For each frozen holdout:
- test rows are the exact `global_row_id` values in the seed-consensus prediction file;
- training support is all EMI rows minus those test IDs;
- the designated held-out residue must occur zero times in reconstructed training;
- Hamming distance is computed only over the eight fixed mutation positions;
- bins: `d=1`, `d=2`, `d>=3`;
- metrics: MCC, balanced accuracy, accuracy, AUROC, AUPRC;
- site-block bootstrap 95% CI and paired far-vs-near site tests with Holm correction.

In [ ]:
def binary_metrics_local(y,score):
    y=np.asarray(y,int);s=np.asarray(score,float);p=(s>=0).astype(int);out={'n':len(y),'accuracy':float(accuracy_score(y,p)),'balanced_accuracy':float(balanced_accuracy_score(y,p)),'mcc':float(matthews_corrcoef(y,p))}
    out['auroc']=float(roc_auc_score(y,s)) if len(np.unique(y))==2 else np.nan;out['auprc']=float(average_precision_score(y,s)) if len(np.unique(y))==2 else np.nan;return out

def min_hamming_to_support(test_seqs,train_seqs,positions):
    T=np.asarray([[s[p] for p in positions] for s in train_seqs],dtype='U1');Q=np.asarray([[s[p] for p in positions] for s in test_seqs],dtype='U1');out=np.empty(len(Q),int)
    for i,q in enumerate(Q):out[i]=int(np.sum(T!=q[None,:],axis=1).min())
    return out

def hbin(d):return 'd=1' if d==1 else ('d=2' if d==2 else 'd>=3')

if RUN_HAMMING:
    positions=[int(s['python_index']) for s in SITE_SPEC];dist_rows=[];coverage=[]
    for hid,g in mutation_consensus.groupby('holdout_id'):
        meta=g.iloc[0];test_ids=np.sort(g.global_row_id.unique().astype(int));train_ids=np.setdiff1d(np.arange(len(emi_sequences)),test_ids);pos=int(meta.kabat_site); seqpos=int([h for h in enriched if h['holdout_id']==hid][0]['python_index']);res=str(meta.heldout_residue)
        if np.any(np.asarray([emi_sequences[i][seqpos] for i in train_ids])==res):raise RuntimeError(f'{hid}: heldout residue in reconstructed train')
        d=min_hamming_to_support(emi_sequences[test_ids],emi_sequences[train_ids],positions)
        if np.any(d==0):raise RuntimeError(f'{hid}: dmin=0')
        coverage.append({'holdout_id':hid,'holdout_type':meta.holdout_type,'kabat_site':int(meta.kabat_site),'n_train':len(train_ids),'n_eval':len(test_ids),'heldout_residue':res,'heldout_residue_train_count':0,'d1':int((d==1).sum()),'d2':int((d==2).sum()),'d3plus':int((d>=3).sum())})
        dist_rows.extend([{'holdout_id':hid,'global_row_id':int(i),'dmin':int(dd),'distance_bin':hbin(int(dd))} for i,dd in zip(test_ids,d)])
    hdist=pd.DataFrame(dist_rows);coverage=pd.DataFrame(coverage);coverage.to_csv(ANALYSIS_DIR/'mutation_hamming_coverage_by_holdout.csv',index=False);hdist.to_csv(ANALYSIS_DIR/'mutation_hamming_distance_by_row.csv',index=False)
    hm=mutation_consensus.merge(hdist,on=['holdout_id','global_row_id'],how='inner',validate='many_to_one');rows=[]
    for keys,g in hm.groupby(['feature','model','task','holdout_id','holdout_type','kabat_site','distance_bin']):
        row=dict(zip(['feature','model','task','holdout_id','holdout_type','kabat_site','distance_bin'],keys));row.update(binary_metrics_local(g.y_true,g.mean_score));rows.append(row)
    hmetrics=pd.DataFrame(rows);hmetrics.to_csv(ANALYSIS_DIR/'mutation_hamming_metrics_by_holdout_bin.csv',index=False)

    # 10k site-block bootstrap of mean site metric for each distance bin.
    boots=[];rng=np.random.default_rng(BOOTSTRAP_SEED+909)
    for keys,g in hmetrics.groupby(['feature','model','task','distance_bin']):
        for metric in ['mcc','balanced_accuracy','accuracy','auroc','auprc']:
            vals=g[['holdout_id',metric]].dropna();arr=vals[metric].to_numpy(float);n=len(arr)
            if not n:continue
            draws=arr[rng.integers(0,n,size=(BOOTSTRAP_DRAWS,n))].mean(axis=1);lo,hi=np.quantile(draws,[.025,.975]);boots.append({**dict(zip(['feature','model','task','distance_bin'],keys)),'metric':metric,'n_site_blocks':n,'mean':float(arr.mean()),'ci_low':float(lo),'ci_high':float(hi),'draws':BOOTSTRAP_DRAWS})
    hboot=pd.DataFrame(boots);hboot.to_csv(ANALYSIS_DIR/'mutation_hamming_site_block_bootstrap_10000.csv',index=False)

    # Paired far-vs-near site tests where both d=1 and d>=3 are available.
    tests=[]
    for keys,g in hmetrics.groupby(['feature','model','task']):
      local=[]
      for metric in ['mcc','balanced_accuracy','accuracy']:
       piv=g.pivot_table(index='holdout_id',columns='distance_bin',values=metric,aggfunc='first').dropna(subset=['d=1','d>=3'] if all(x in g.distance_bin.unique() for x in ['d=1','d>=3']) else None)
       if 'd=1' not in piv or 'd>=3' not in piv or len(piv)<2:continue
       d=piv['d>=3']-piv['d=1'];w=wilcoxon(d,zero_method='wilcox',alternative='two-sided',method='auto') if not np.allclose(d,0) else None;local.append({**dict(zip(['feature','model','task'],keys)),'metric':metric,'n_sites':len(d),'mean_far_minus_near':float(d.mean()),'median_far_minus_near':float(np.median(d)),'wilcoxon_p':1. if w is None else float(w.pvalue),'exact_sign_flip_p':exact_sign_flip_pvalue(d)})
      adj=holm_adjust([x['wilcoxon_p'] for x in local]) if local else []
      for x,a in zip(local,adj):x['wilcoxon_p_holm']=float(a);tests.append(x)
    htests=pd.DataFrame(tests);htests.to_csv(ANALYSIS_DIR/'mutation_hamming_far_vs_near_tests.csv',index=False);display(coverage);display(hboot[(hboot.feature=='onehot')&(hboot.model.isin(['Routed-MOLM','Standard-MOLM']))])

In [ ]:
if RUN_HAMMING:
    # External sequence distance to the full 4,000-row EMI support.
    ext_distance=[]
    for dataset,seqs in [('ISO',iso_sequences),('IgG-all96',igg_sequences)]:
        d=min_hamming_to_support(seqs,emi_sequences,[int(s['python_index']) for s in SITE_SPEC])
        ext_distance.extend([{'dataset':dataset,'source_row_id':i,'sequence_id':str(seqs[i]),'dmin_to_emi':int(dd),'distance_bin':('d=0' if dd==0 else hbin(int(dd)))} for i,dd in enumerate(d)])
    ext_distance=pd.DataFrame(ext_distance);ext_distance.to_csv(ANALYSIS_DIR/'external_hamming_distance_to_full_emi.csv',index=False)
    # Spearman by external distance bin for ISO and IgG-all96.
    erows=[]
    ec=external_consensus.copy();ec['source_row_id']=ec['row_id']
    # For IgG-all96 row_id equals source row. ISO likewise. Primary42 is omitted from this distance-stratified view to avoid duplicated source semantics.
    ec=ec[ec.dataset.isin(['ISO','IgG-all96'])].merge(ext_distance,on=['dataset','source_row_id','sequence_id'],how='left')
    for keys,g in ec.groupby(['dataset','feature','model','distance_bin']):
        if len(g)<4:continue
        erows.append({**dict(zip(['dataset','feature','model','distance_bin'],keys)),'n':len(g),'aff_spearman':float(spearmanr(g.pred_aff,g.true_aff).statistic),'ova_spearman':float(spearmanr(g.pred_ova,g.true_ova).statistic)})
    external_hamming_spearman=pd.DataFrame(erows);external_hamming_spearman.to_csv(ANALYSIS_DIR/'external_spearman_by_hamming_distance.csv',index=False);display(external_hamming_spearman.head(50))

## 14. Experiment coverage summary + final bundle


In [ ]:
coverage_rows=[
 {'analysis':'Mutation inference','scope':'EMI grouped residue holdout','evidence':'4-model McNemar + 10k paired bootstrap + site-block Friedman/Wilcoxon/sign-flip'},
 {'analysis':'Shared vs independent learning','scope':'mutation/external/Pareto','evidence':'Standard-MOLM plus MOLM-ST same-loss controls'},
 {'analysis':'Fixed-budget Pareto prioritization','scope':'ISO and secondary IgG-all96','evidence':'same K=5,10,15,20,25; recall/precision/enrichment/HV/IGD'},
 {'analysis':'Sequence-distance stress test','scope':'within-scaffold mutation generalization','evidence':'Hamming d=1/d=2/d>=3 + site-block bootstrap'},
 {'analysis':'Component/capacity ablation','scope':'Routed-MOLM controls','evidence':'A-H component ablation + four-arm ranking/gap loss ablation'},
 {'analysis':'Latent representation robustness','scope':'Standard-MOLM','evidence':'logits vs PCA64/PCA32/PCA16 under identical fixed budgets and five seeds'},
 {'analysis':'Residue-focused representation','scope':'mutation extrapolation','evidence':'Mean-ESM2 vs Site-ESM2 and Mean-Fusion vs Site-Fusion paired tests'},
 {'analysis':'Statistical robustness','scope':'optimization and biological blocks','evidence':'sequence bootstrap CIs + mutation sites as paired blocks; seeds treated as optimization variability'},
]
experiment_coverage=pd.DataFrame(coverage_rows)
experiment_coverage.to_csv(ANALYSIS_DIR/'EXPERIMENT_COVERAGE_MATRIX.csv',index=False)
display(experiment_coverage)

# Master compact tables.

master=[]
for _,r in mutation_aggregate.iterrows():master.append({'evaluation':f"mutation_{r.holdout_type}",'dataset':'EMI','feature':r.feature,'model':r.model,'task':r.task,'metric':'MCC','value':r.site_mean_mcc})
for _,r in spearman_summary.iterrows():master.extend([{'evaluation':'external_continuous','dataset':r.dataset,'feature':r.feature,'model':r.model,'task':'affinity','metric':'Spearman','value':r.aff_spearman_mean},{'evaluation':'external_continuous','dataset':r.dataset,'feature':r.feature,'model':r.model,'task':'ova','metric':'Spearman','value':r.ova_spearman_mean}])
for _,r in pareto_summary.iterrows():master.append({'evaluation':f'pareto_k{int(r.k)}','dataset':r.dataset,'feature':r.feature,'model':r.model,'task':'multiobjective','metric':'Recall','value':r.recall_mean})
master=pd.DataFrame(master);master.to_csv(ANALYSIS_DIR/'MASTER_4MODEL_SUMMARY.csv',index=False)

repro={'created_utc':datetime.now(timezone.utc).isoformat(),'base_lock_sha256':BASE_LOCK_SHA,'analysis_lock_sha256':ANALYSIS_LOCK_SHA,'repository_commit':PINNED_COMMIT,'seeds':SEEDS,'features':FEATURE_DIMS,'component_features':COMPONENT_FEATURES,'loss_ablation_features':LOSS_ABLATION_FEATURES,'bootstrap_draws':BOOTSTRAP_DRAWS,'analysis_files':sorted(p.name for p in ANALYSIS_DIR.iterdir() if p.is_file())}
(ANALYSIS_DIR/'REPRODUCIBILITY_MANIFEST_UNIFIED.json').write_text(json.dumps(repro,indent=2),encoding='utf-8')

bundle=WORK_ROOT/'unified_result_bundle';shutil.rmtree(bundle,ignore_errors=True);bundle.mkdir()
for p in ANALYSIS_DIR.iterdir():
    if p.is_file():shutil.copy2(p,bundle/p.name)
for p in [ANALYSIS_LOCK_PATH,WORK_ROOT/'UNIFIED_ANALYSIS_LOCK.sha256',ENRICHED_HOLDOUT_PATH,STANDARD_WORKER_PATH,COMPONENT_WORKER_PATH,LOSS_WORKER_PATH,CODE_DIR/'molm_definitive_common.py',CODE_DIR/'molm_definitive_esm_cache.py']:
    if p.exists():shutil.copy2(p,bundle/p.name)
RESULT_ZIP=shutil.make_archive('/kaggle/working/MOLM_Unified_Experiment_Results','zip',root_dir=bundle)
print('FINAL RESULT ZIP:',RESULT_ZIP)
print('Analysis files:',len(list(ANALYSIS_DIR.iterdir())))

## Expected Kaggle settings

- **Accelerator:** T4 x2
- **Internet:** On, unless the pinned repository is also attached as an input
- **Inputs required:**
  1. the prior definitive result bundle/dataset;
  2. the ESM reuse cache dataset.

The notebook embeds the Standard-MOLM worker and both ablation workers, so those scripts do not need to be uploaded separately to Kaggle.

### Runtime note
This is intentionally a large experiment suite. The Standard main grid, A--H ablation, and loss ablation are resumable. If a Kaggle session ends, create a dataset from `/kaggle/working/molm_unified_experiments` or the final output and reattach it before continuing, or run sections separately by toggling the `RUN_*` flags.
